# H3 Video Stüdyosu (grafik arayüzlü uygulama)

Kısa bir fikir yazın → **Gemini** onu önce detaylı bir yönetmen brief'ine, sonra MiniMax-H3'ün resmi `h3-prompt-writing` yapısına çevirsin → **Pika H3 Flex** ve/veya **MiniMax-H3** ile Colab'de video üretin.
İsterseniz **karakter görseli** yükleyin (en fazla 9): uygulama referanslı moda (Ref2VA) geçer, görseldeki kişinin yüzü video boyunca korunur.

**Gerekenler**
1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → G4 GPU** (RTX PRO 6000). Başka GPU'da da çalışır ama NVFP4 yerine 27 GB'lık int8 metin kodlayıcı iner. Colab G4 seçiliyken bile L4/A100 verebiliyor; başlatma hücresi tahsis edilen GPU'yu yazar.
2. **Gemini API anahtarı:** Sol paneldeki 🔑 *Secrets* bölümüne `GEMINI` (ya da `GEMINI_API_KEY`) adıyla ekleyin ve yanındaki **Notebook erişimi** anahtarını açın; kapalıysa anahtar okunamaz. (Ya da anahtarı uygulamadaki şifre kutusuna yazın.)
3. **Çalışma zamanı → Tümünü çalıştır.** Arayüz en alttaki hücrenin altında açılır.

**İlk üretim:** ComfyUI kurulumu + modeller (~42 GB, Hugging Face'ten, SHA-256 doğrulamalı) ~5 dk sürer; sonra Pika ~6 dk, MiniMax-H3 ~18 dk (15 sn video).
Videolar `/content/h3app_videolar/` klasörüne yazılır, arayüzden indirilebilir ve her video biter bitmez Drive'a (`H3_Video_Studyosu`) kopyalanır.

**Bağlantı koparsa:** üretim arka planda sürer. Sayfaya dönünce **📂 Son Üretimleri Yükle**'ye basın: süren iş varsa günlüğü izlenir, bitince sonuçlar yüklenir. Bilgisayar uzun süre uykuda kalırsa Colab runtime'ı silebilir; o zaman bitmiş videolar Drive'dadır.

## 1 · Ayarlar

In [ ]:
# Videolar üretilir üretilmez Drive'a da kopyalansın mı? Açık bırakın: Colab bağlantısı koparsa (bilgisayar
# uykuya geçti, sekme kapandı) runtime silinebilir; Drive'a yazılmamış videolar kaybolur.
SAVE_TO_DRIVE = True
DRIVE_FOLDER = "/content/drive/MyDrive/H3_Video_Studyosu"
# Modeller önce Hugging Face'ten iner (ölçüm: ~6 dk; Drive'dan kopya 42 GB için ~12 dk, daha yavaş). HF'e
# ulaşılamazsa ya da indirme bozuksa bu Drive klasörlerindeki kopyalar kullanılır (boyut + SHA-256 kontrolüyle).
# Drive'ınızda yoksa sorun değil; istemezseniz [] yapın.
DRIVE_MODEL_DIRS = ["/content/drive/MyDrive/MiniMax-H3/models", "/content/drive/MyDrive/Pika-H3-Flex"]
# Modeller Drive'da yoksa, indirildikten sonra Drive'a da kaydedilsin mi? (DRIVE_MODEL_DIRS'in ilk klasörüne)
# Açarsanız HF'e ulaşılamayan bir oturumda yedek olarak Drive'dan alınır. Drive'ınızda ~55-65 GB boş yer
# olmalı (ücretsiz 15 GB'a sığmaz); yer yoksa kopya atlanır. Kopya arka planda yapılır, üretimi bekletmez.
SAVE_MODELS_TO_DRIVE = False  #@param {type:"boolean"}

## 2 · Uygulama dosyaları
Her hücre bir modülü `/content/h3_video_app/h3app/` altına yazar. Kaynak kod burada açıkça okunabilir.

In [ ]:
import os
os.makedirs("/content/h3_video_app/h3app", exist_ok=True)

In [ ]:
%%writefile /content/h3_video_app/h3app/__init__.py
"""H3 Video Stüdyosu: ham bir video fikrini Gemini ile MiniMax-H3 prompt'una çevirir ve
ComfyUI üzerinde Pika H3 Flex ve/veya MiniMax-H3 ile video üretir (Google Colab, G4 GPU)."""

__version__ = "1.0.0"


def launch(**kwargs):
    from .ui import launch as _launch
    return _launch(**kwargs)

In [ ]:
%%writefile /content/h3_video_app/h3app/config.py
"""Sabit sürümler ve ayarlar. Hepsi 01.11.2026'da test edilmiş sürümlere sabitlenmiştir."""

import os

WORK_DIR = os.environ.get("H3APP_WORK_DIR", "/content")
COMFY_DIR = os.path.join(WORK_DIR, "ComfyUI")
OUTPUT_DIR = os.path.join(WORK_DIR, "h3app_videolar")
CACHE_DIR = os.path.join(WORK_DIR, "h3app_cache")
# Model dosyalarının önce aranacağı klasörler (ör. Drive'daki doğrulanmış kopyalar); launch(model_dirs=...) doldurur
MODEL_SOURCE_DIRS = []
# Drive'da bulunmayan modeller indirildikten sonra buraya da kopyalanır (None: kapalı); launch(model_save_dir=...) doldurur
MODEL_SAVE_DIR = None

COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI.git"
COMFY_COMMIT = "0dbfd49"
COMFY_PORT = 8188

HF_REPO = "Comfy-Org/MiniMax-H3"
HF_REVISION = "912b6e38b2092ec4d2e86d24183e0d59ea059d48"

# rol -> (depodaki yol, ComfyUI models alt klasörü, boyut, sha256)
MODELS = {
    "unet": ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models",
             20970379616, "e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a"),
    "te_nvfp4": ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders",
                 15687142551, "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6"),
    "te_int8": ("text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors", "text_encoders",
                27141342152, "bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923"),
    "vae_video": ("vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae",
                  2811065184, "52a2c8c73583c86e4f41cdcce3a6ad0ea562987bc0bf3d60a0cef5f5c8e60c0e"),
    "vae_audio": ("vae/minimax_h3_audio_vae_fp32.safetensors", "vae",
                  605254808, "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48"),
    "pika_lora": ("loras/pika_h3_flex_fl2va_v1_bf16.safetensors", "loras",
                  1956172304, "cc5453aeecd6189468409003fe4cf1b57e52ff3a830a0a154b87decbe07358b7"),
    # Referanslı üretim (Ref2VA): yüklenen karakter/sahne görselleri
    "unet_ref": ("diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", "diffusion_models",
                 20970379616, "9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779"),
    "pika_lora_ref": ("loras/pika_h3_flex_ref2va_v1_bf16.safetensors", "loras",
                      1956172304, "de3745ceb7d8d5c20898dd8fa3a05f1c0b00089ccd55297b2a8bab95c766b0e0"),
    # H3 Fast / Turbo LoRA (LightX2V / Comfy-Org resmi şablonları: T2VA 8 adım, Ref2VA 4 adım)
    "turbo_lora": ("loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "loras",
                   1956193000, "2339acdf19bfe123f46b971ea35d367a84adb85de43627e1eceafa5a5b2b111e"),
    "turbo_lora_ref": ("loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", "loras",
                       1956193000, "5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c"),
}

# H3'ün resmi prompt yazım skill'i (MiniMax'in deposundan, sabit commit, git blob sha ile doğrulanır)
SKILL_REPO = "MiniMax-AI/MiniMax-H3"
SKILL_COMMIT = "199b7d0e1fd95e7b3eafa8c15eca692534142427"
SKILL_FILES = {
    "skills/h3-prompt-writing/SKILL.md": "b6d9b2839384a588763a9c24315225dd8ce19d56",
    "skills/h3-prompt-writing/references/base-en.txt": "40cf586a634d677d6b7107b367cf0ec9621be728",
    "skills/h3-prompt-writing/references/ref-en.txt": "7ae1b2d07d743fd2392258a96449be9e9e322d35",
}

# Pika H3 Flex'in resmi 6 adımlık programı (knot'lar 1000/750/500/250/100/40/0, video shift 12)
PIKA_SIGMAS = "1.0, 0.972973, 0.923077, 0.8, 0.571429, 0.333333, 0.0"
H3_STEPS = 20
TURBO_STEPS = 8
TURBO_REF_STEPS = 4

METHODS = {
    "turbo": {"label": "MiniMax-H3 Turbo (4/8 adım, G4'te ~2-3 dk)", "file": "minimax_h3_turbo"},
    "pika": {"label": "Pika H3 Flex (6 adım, G4'te ~6 dk)", "file": "pika_h3_flex"},
    "h3": {"label": "MiniMax-H3 (20 adım, G4'te ~18 dk)", "file": "minimax_h3"},
}

RESOLUTIONS = {
    "16:9 yatay · 1344×768": (1344, 768),
    "9:16 dikey · 768×1344": (768, 1344),
    "1:1 kare · 1024×1024": (1024, 1024),
}

# Görsel stil. Prompt'a marka adı değil tarif yazılır; anime/3d/suluboya tarifleri scenario-alter skill'inin
# (MIT) denenmiş restyle cümlelerinden. "real" bugünkü davranıştır: prompt'lar stil seçimi eklenmeden önceki gibi kalır.
STYLES = {
    "real": {"label": "Gerçek dünya", "kind": "photoreal live-action"},
    "anime": {
        "label": "Anime", "kind": "hand-drawn 2D anime",
        "look": "a hand-drawn 2D anime feature film: clean ink lines, flat cel shading with one shadow tone, "
                "painted backgrounds",
        "people": "hand-drawn anime characters",
        "background": "hand-painted anime background art with flat colour areas and painted clouds",
        "avoid": "glossy 3D render, CGI lighting, photorealism, live-action skin texture, photographic background",
        "keywords": ("anime",),
    },
    "pixar": {
        "label": "Disney Pixar", "kind": "stylised 3D animated feature film",
        "look": "a 3D animated feature film: stylised characters with slightly larger heads and hands, exaggerated "
                "rounded proportions, smooth stylised skin and sculpted hair, soft global illumination, a stylised 3D set",
        "people": "stylised 3D animated characters",
        "background": "a stylised 3D set: soft rounded shapes, sculpted clouds, clean simple surfaces",
        "avoid": "photorealism, live-action footage, real skin pores, photographic water and sky, 2D flat shading",
        "keywords": ("3d animated", "3d-animated", "3d animation"),
    },
    "watercolour": {
        "label": "Suluboya", "kind": "hand-painted watercolour animation",
        "look": "a moving watercolour painting on textured paper: visible brush strokes that follow the motion, "
                "soft edges and a little pencil underdrawing",
        "people": "loosely painted watercolour figures with visible wet-on-wet pigment bleed, soft unpainted paper "
                  "highlights on faces and clothes, simplified hand-drawn ink outlines, zero skin texture",
        "background": "loose watercolour washes on textured paper and a few confident brush strokes",
        "avoid": "photorealism, live-action footage, 3D render, CGI lighting, sharp photographic detail, "
                 "realistic human faces, smooth digital airbrushing, solid opaque skin shading",
        "keywords": ("watercolour", "watercolor"),
    },
}
DEFAULT_STYLE = "real"

# Ses seviyesi standardı (EBU R128 loudnorm hedefi)
TARGET_LUFS = -16.0

# Prompt yazımı için model zinciri (kullanıcının sırası, 16.11.2026): önce varsayılan, yanıt gelmezse yedekler sırayla.
# Arayüzde başka bir model seçilirse o önce denenir, ardından bu zincir. Gemma sistem talimatını mesajın başında alır.
GEMINI_MODEL = "gemini-4-hydrogen"
GEMINI_FALLBACK_MODELS = ("gemini-3.8-flash", "gemini-3.5-flash-lite", "gemma-4-31b-it")
# Her model ilk metin parçasını bu kadar saniyede göndermezse ya da yoğunsa (503/429) sıradakine geçilir
# 18.11.2026 ölçümü: gemini-4-hydrogen ilk parçayı ~58,6 sn'de gönderdi; 10 sn sınırı ana modeli sistematik olarak
# yedeğe (flash-lite) düşürüyordu ve brief kalitesi düşüyordu. Varsayılan 60 sn; H3APP_FIRST_CHUNK_SECONDS ile değişir.
FIRST_CHUNK_SECONDS = int(os.environ.get("H3APP_FIRST_CHUNK_SECONDS", "60"))
# Üretilen videoyu izleyen Gemini modelleri (sırayla). 16.11.2026 kör testinde fazladan eli 2.5-flash ve
# 3-flash-preview buldu, 3.5-flash-lite kaçırdı: izlemede lite model kullanılmaz. 17.11.2026: gemini-2.5-flash
# yeni kullanıcılara kapatıldı (404 NOT_FOUND), zincirden çıkarıldı; yedek olarak lite olmayan 3.8-flash.
REVIEW_MODELS = ("gemini-3-flash-preview", "gemini-3.8-flash")
# İzleme yüksek önemde kusur bulursa yeni seed ile en fazla bu kadar yeniden deneme; en iyi aday tutulur
REVIEW_RETRIES = 2

MIN_SECONDS, MAX_SECONDS = 4, 15
MAX_TOTAL_SECONDS = 60      # Çok çekimli uzun video üst sınırı (4 blok × 15 sn)
MIN_SHOT_SECONDS, MAX_SHOT_SECONDS = 2, 15
MAX_BLOCKS = 4
MAX_REF_IMAGES = 9          # H3 Ref2VA en fazla 9 referans görsel alır

AUDIO_MODES = {
    "speech": "Türkçe konuşmalı sahne (doğal ses)",
    "music": "Müzikli, konuşmasız kurgu (dış müzik)",
}
DEFAULT_AUDIO_MODE = "speech"


def frames_for(seconds):
    """Resmi şablondaki formül: 24 fps, H3'ün 17k+5 kare ızgarasına yukarı yuvarlanır."""
    n = max(5, round(float(seconds) * 24))
    return n + (5 - n % 17) % 17

In [ ]:
%%writefile /content/h3_video_app/h3app/prompt_writer.py
"""Gemini ile iki adım: (1) kısa fikir → detaylı yönetmen brief'i, (2) brief → MiniMax-H3 prompt'u.

Kural kaynağı H3'ün resmi `h3-prompt-writing` skill'idir (SKILL.md + references/base-en.txt ve
references/ref-en.txt); dosyalar MiniMax'in GitHub deposundan sabit commit ile indirilir ve git blob sha
ile doğrulanır. Referans görsel verilirse (Ref2VA) görseller Gemini'ye de gönderilir; prompt resmi altı
bölümlü yapıyla yazılır ve görseller <Picture 1>, <Picture 2>… olarak anılır.
Çıktı denetlenir: bölümler eksiksiz ve sıralı mı, kullanıcının tırnak içindeki her metni (ekran yazısı,
diyalog) harfi harfine korunmuş mu, her görsel anılmış mı. Sorun varsa Gemini'ye bir kez daha sorulur.
"""

import hashlib
import json
import mimetypes
import math
import os
import re
import time
import urllib.request

from . import config

EXAMPLE_INPUT = 'Turkish dialogue, lip sync and acting. A two-person medium shot inside a small traditional Turkish tea house in Cappadocia at dusk. An old man with a white moustache sits across from a young woman, steam rising from tulip tea glasses between them. He says in Turkish, in a warm gravelly voice, "Kızım, dünya çok hızlı dönüyor, bazen durup bakmak lazım." She smiles, nods and answers in Turkish, softly, "Haklısın dede, ama durunca da hep bir şey kaçırıyormuşum gibi hissediyorum." He chuckles and replies, "Kaçıran sensin, bakan değil." Perfect lip sync, natural micro expressions, eyes follow each speaker, hands stay stable around the glasses. Camera: 50mm, shallow depth of field, slow push-in then a gentle shift of focus between faces. Lighting: golden window light with warm lantern fill, soft rim light. Color palette: terracotta, amber, deep teal. Sound: clear synchronized Turkish speech, glass clinks, distant street ambience, soft saz-free ambient room tone. Photoreal cinematic drama.'

EXAMPLE_OUTPUT = "integrated_multimodal_description: [Shot 1] Live-action, photoreal cinematic drama shot on a 50mm lens with shallow depth of field: a two-person medium shot inside a small traditional Turkish tea house in Cappadocia at dusk. On the left an old Turkish man in his seventies with a thick white moustache, weathered tanned skin and a dark wool flat cap sits at a low wooden table; on the right, facing him, a young Turkish woman in her mid-twenties with dark hair tied back and a mustard knitted cardigan. Between them on the table stand two tulip-shaped tea glasses of deep red tea on small saucers, thin steam rising from both. Golden light falls through a deep-teal window frame behind them, a hanging copper lantern adds warm fill, and a soft rim light outlines both faces against terracotta plaster walls and amber kilim cushions. The camera pushes in with small amplitude at slow speed during the first seconds, then settles. The old man with a warm, deep, gravelly voice and a slow speaking rate (S1) looks at her, both hands resting around his tea glass, and says: <d>[Turkish] Kızım, dünya çok hızlı dönüyor, bazen durup bakmak lazım.</d> While he speaks she watches his face. At about 4.5 seconds the focus gently shifts from his face to hers. The young woman with a soft, clear, gentle voice (S2) smiles, nods once, and answers: <d>[Turkish] Haklısın dede, ama durunca da hep bir şey kaçırıyormuşum gibi hissediyorum.</d> While she speaks his eyes stay on her. At about 10.5 seconds the focus gently shifts back to the old man, who chuckles softly, his moustache lifting, and the old man (S1) replies: <d>[Turkish] Kaçıran sensin, bakan değil.</d> She lowers her eyes to her tea glass and smiles. Each speaker's lips move in exact sync with every syllable, with natural micro-expressions and blinks; their eyes follow whoever is speaking; both pairs of hands stay steady around the tea glasses throughout.\n\noverall_soundscape: Quiet room tone of a small stone tea house with a faint hiss from the lantern, the soft clink of a tea glass touching its saucer, a short warm chuckle, and distant street ambience of footsteps and a faraway car through the window.\n\nnon_diegetic_music: N/A"

# Brief örnekleri: kısa fikir → kullanıcının kendi yazdığı detaylı brief (Test 1 ve Test 2)
BRIEF_EXAMPLES = [
    ('Kapadokya\'da bir çay evinde akşamüstü yaşlı bir dede ile genç bir kadın Türkçe sohbet ediyor. Dede: "Kızım, dünya çok hızlı dönüyor, bazen durup bakmak lazım." Kadın gülümseyip: "Haklısın dede, ama durunca da hep bir şey kaçırıyormuşum gibi hissediyorum." Dede gülerek: "Kaçıran sensin, bakan değil." Dudak senkronu kusursuz olsun, sinematik.', EXAMPLE_INPUT),
    ('Gece yağmurlu bir İstanbul ara sokağı. Çay evinin üstünde kırmızı neon tabela "DUR, BİR BAK!" yazıyor, altında tebeşir menüde "ÇAY 15 TL". Trençkotlu bir adam tabelanın altından geçiyor. Yazılar 15 saniye boyunca hiç bozulmasın.', 'Typography and Turkish text rendering. A rainy Istanbul back street at night, a hand-painted neon sign above a tea house reads exactly "DUR, BİR BAK!" in glowing red letters, flickering once, reflected perfectly and mirrored in the wet cobblestones. Below it a smaller chalkboard menu reads "ÇAY 15 TL" in white chalk handwriting. A man in a trench coat walks under the sign, the rain drops catch the neon light, and the camera slowly pushes in then racks focus from the sign to his face. Text must remain perfectly legible, spelled correctly with the Turkish characters İ and Ç intact, stable and consistent across all 15 seconds with no morphing letters. Lighting: neon red and teal practicals, wet asphalt reflections, volumetric steam from the tea house. Anamorphic lens, 35mm grain, cinematic photoreal. Sound: rain, distant call to prayer, neon buzz, low synth hum.'),
    # Test 3: kullanıcının asıl istediği (referans videoyu bu yazı üretti); ses satırı geri bildirime göre yumuşatıldı
    ('Kapalıçarşı\'da öğle vakti, kırmızı paltolu ve gümüş atkılı bir kadın kalabalığın içinde yürüyor; kamera steadicam ile önünde geri geri gidiyor. Etrafta bağıran esnaflar, fotoğraf çeken turistler, çay tepsisi taşıyan bir çocuk, baharat arabası iten bir adam ve bacakların arasından koşan bir kedi var. İnsanlar önünden geçse de kadının yüzü, saçı ve kıyafeti 15 saniye boyunca hiç değişmesin; arka plandaki yüzler de bozulmasın. Fotogerçekçi, sinematik.', 'Crowd density and identity consistency. A continuous steadicam tracking shot walking backwards in front of a woman in a red coat as she moves through the packed Grand Bazaar in Istanbul at midday, framed wide and full-body from head to feet with the shops visible on both sides; she looks ahead where she is going, not into the lens. Dozens of distinct extras around her: merchants calling out, tourists with cameras, a boy carrying a tea tray, a man pushing a cart of spices, a cat darting between legs. Her face, hair, red coat and silver scarf stay perfectly consistent for the entire 15 seconds, even when people pass between her and the camera. Faces of background people stay stable and do not morph. Lantern light, hanging carpets, copper pots, drifting dust in shafts of sunlight from the vaulted ceiling. Camera: 28mm lens, f/2.8, smooth steadicam, natural motion blur at 180 degree shutter. Color palette: saturated reds, warm ochre, turquoise tiles. Sound: layered crowd murmur as the dominant layer, background bargaining in Turkish between a carpet seller and a customer ("Bu halı kaça?" "Sana özel, bin beş yüz lira abla."), a vendor calling "Buyrun, bakmak bedava!", tea glasses clinking, soft footsteps of flat boots buried under the crowd. Photoreal cinematic.'),
]

T2V_FIELDS = ("integrated_multimodal_description:", "overall_soundscape:", "non_diegetic_music:")
REF_FIELDS = ("subject_definitions:", "summary:", "retention_analysis:", "detailed_description:",
              "overall_soundscape:", "non_diegetic_music:")

# Varsayılan yönetmenlik kuralları (Test 3 geri bildirimi: çok yakın çekim, topuk sesi, pazarlık sesi yok).
# Hem brief hem H3 prompt adımında kullanılır; kullanıcı açıkça başka bir şey istediyse onun dediği geçerlidir.
_FRAMING = """- Framing: keep the user's shot size and lens. When a person walks or is tracked through a place and no
  shot size is given, frame a wide full-body shot (head to feet in frame, about a 28mm lens) with the
  surroundings visible on both sides; do not tighten to a medium shot or close-up. The person looks where
  they are going, not into the lens, and does not pose or smile at the camera."""
_REALISM = """- Realism: a named real place must look like the real place (its architecture, the shops and goods that
  really dominate it, its light and its people), not a generic or touristic version of it. Something the
  user mentions once (a spice cart, a cat) stays one element; it must not take over the whole set."""
_FOOTSTEPS = """- Footsteps: never heels, clicking or tapping steps unless asked. Footwear is flat boots or shoes and
  steps are soft and buried under the ambience."""
_ANIMALS = """- Animals: a cat, dog or other animal moves on the ground (between legs, along the floor, under stalls)
  unless the user asks otherwise; never write that it darts past lanterns or anything hanging in the air."""
_STAGING = """- Staging and hands: say where every visible person stands on screen (left / centre / right, foreground /
  background) and which hand holds what. Describe the first 2 seconds as one clear opening image in which every
  visible person is named and already in place. An object is handed only between two people who are both
  visible, hand to hand, never from outside the frame. The camera never shows a character's point of view and
  no extra hands, arms or people enter the frame unless the user asks for it."""
# 16-17.11.2026 gerçek denemelerindeki orta kusurların genel karşılığı: nesne tutan elde parmak bozukluğu, katı
# görünen sıvı, titreyen küçük tekrarlı hareketler, geç başlayan replik. Konuya özel değil, her sahneye uygulanır.
_DETAIL = """- Contact, materials and timing: whenever a hand, paw or tool touches or holds something, describe the
  contact plainly (which hand, how it grips, that it stays fully in frame) and keep it stable while it moves;
  hands keep a natural number of fingers. Fluids, smoke, steam, fire and fabric move with their real physical
  weight and translucency, never as stiff or solid shapes. Small repeated motions (tapping, stirring, writing,
  chewing) stay slow, even and clearly readable, never a jittery blur. Give every spoken line an explicit start
  time ("At about N seconds") inside the clip, start the first line early unless the user asks otherwise, and
  move the speaker's lips only while the line is spoken."""
# 17.11.2026: iki gerçek denemede de (toplantı odası, sahaf) yazı camdan/kemerden görünen karşı binada çıktı; eski
# cümle "shopfronts, arches, lanterns" diyerek dükkân cephelerini sahneye davet ediyordu.
_SIGNAGE_NONE = ("No readable text, signboards, logos, banners or lettering anywhere in the frame, including on "
                 "buildings, shops and streets seen through windows, doorways or in the distance; any distant sign "
                 "is a blank, out-of-focus shape with no letters. Places are identified by their goods and "
                 "architecture, never by written signs.")
_SIGNAGE_ONLY = ("Only the exact quoted sign text appears; no other signs, extra boards or background lettering "
                 "anywhere in the frame.")
_SIGNAGE = f"""- Signage: when the user did not ask for on-screen text in quotes, state explicitly: "{_SIGNAGE_NONE}"
  When the user asked for specific quoted sign text, state explicitly: "{_SIGNAGE_ONLY}\""""
_SPEECH = """- Background speech: ONLY when the user's scene has a crowd, market, bazaar or busy public place, write
  2-3 short, natural background lines in the local language (Turkish for Turkey) as real dialogue, e.g. a
  vendor and a customer bargaining, heard at a lower level under the ambience; the crowd voices are the
  dominant sound layer, not footsteps. In any other scene (a shop, a room, a car, a quiet street with the
  named characters) add NO background lines and no extra voices: only the characters the user named speak."""
_MUSIC_SILENT = ("ambient sound only, no voices, no speech, no singing")
_MUSIC_MODE = f"""- Music montage mode (no speech): do NOT write any spoken dialogue (<d>...</d>), crowd chatter or human
  voices. In overall_soundscape state explicitly "{_MUSIC_SILENT}" along with natural environmental foley,
  and in non_diegetic_music write "N/A" (external music will be mixed later)."""
_DIRECTION_HEAD = "Default direction (use only when the user did not ask for something else):"
DIRECTION_RULES = "\n".join([_DIRECTION_HEAD, _FRAMING, _STAGING, _DETAIL, _REALISM, _FOOTSTEPS, _ANIMALS, _SIGNAGE, _SPEECH])

# Gerçek dünya dışındaki stillerde gerçekçilik kuralı: mekân tanınır kalır ama stilde yeniden çizilir
_REALISM_STYLED = """- Real places: a named real place keeps what makes it recognisable (its architecture, the shops and
  goods that really dominate it, its light and its people), redrawn in the chosen style, not photographed.
  Something the user mentions once (a spice cart, a cat) stays one element; it must not take over the whole set.
  Never leave shops or stalls generic ("market stalls"): name what they really sell there and how it is shown
  (e.g. gold jewellery in lit glass vitrines, carpets hanging from the walls), or the model fills every stall
  with spices and sweets."""

STYLE_RULES = """Visual style (chosen by the user; it overrides every "photoreal", "live-action" or "cinematic
photography" wording in the guide and in the examples, which only show the structure):
- The whole video is {look}. The people are {people}. The background and the whole set are {background}.
- Describe the background as art in that style too, not only the people: if only the people are described,
  the model draws them and leaves the sky, water and buildings photographic.
- Nothing may look photographic: it must look like a shot made from scratch in this style, not a filter over
  live footage. Avoid: {avoid}.
- Animated styles drift toward a static chest-up portrait that faces the camera. When a person walks or is
  tracked, write the framing in the shot itself and repeat it later: a wide full-body shot, head to feet in
  frame with the floor visible, the person walking and looking along their path, never posing for the lens.
- Open the first description section with one sentence that names this style (in place of "Live-action,
  photoreal ..."), and keep lighting, palette and lens choices that suit the style."""

STYLE_REF_RULE = """- Reference pictures give the identity only: the person keeps a recognisable face shape, hair, skin
  tone and colours, but is drawn as one of the {people}, never as a photographed person. When a reference picture
  already shows the character in the chosen style with distinctive clothing, hat, eyewear/goggles, props or setting
  (e.g. a jungle) and the user does not ask for different ones, preserve that exact character design, outfit,
  accessories and environment across all shots."""


def signage_rule(request=""):
    """Kullanıcı tırnak içinde yazı istediyse yalnızca o tabelaya izin verir; istemediyse arka plan yazılarını yasaklar."""
    return _SIGNAGE_ONLY if literals(request) else _SIGNAGE_NONE


def direction_rules(style=config.DEFAULT_STYLE, n_images=0, audio_mode=config.DEFAULT_AUDIO_MODE):
    """Stil kuralları + varsayılan yönetmenlik kuralları. "real" + "speech" için bugünkü DIRECTION_RULES aynen döner."""
    s = config.STYLES.get(style) or config.STYLES[config.DEFAULT_STYLE]
    sound_rule = _MUSIC_MODE if audio_mode == "music" else _SPEECH
    if "look" not in s:
        if audio_mode == config.DEFAULT_AUDIO_MODE:
            return DIRECTION_RULES
        return "\n".join([_DIRECTION_HEAD, _FRAMING, _STAGING, _DETAIL, _REALISM, _FOOTSTEPS, _ANIMALS, _SIGNAGE, sound_rule])
    rules = STYLE_RULES.format(**s) + ("\n" + STYLE_REF_RULE.format(**s) if n_images else "")
    return rules + "\n\n" + "\n".join([_DIRECTION_HEAD, _FRAMING, _STAGING, _DETAIL, _REALISM_STYLED, _FOOTSTEPS, _ANIMALS,
                                        _SIGNAGE, sound_rule])


COMMON_RULES = """Hard rules:
- Output ONLY the final prompt sections, in the required order, each starting on its own line with its
  field name, separated by one blank line. No title, no explanation, no markdown, no code fences.
- Write the sections in English, but copy every piece of on-screen text and every spoken line from the
  user's request character for character in its original language. Keep Turkish letters exactly
  (İ ı Ş ş Ğ ğ Ç ç Ö ö Ü ü) and keep the original punctuation.
- On-screen text goes in English double quotes, e.g. a neon sign that reads "DUR, BİR BAK!".
  When the user asks for legible or stable text, say explicitly that the text stays exactly spelled,
  legible and stable for the whole video with no morphing letters, no extra words and no subtitles.
- Dialogue uses speaker IDs (S1), (S2)... and <d>[Language] exact words</d>, e.g. <d>[Turkish] ...</d>.
- The whole timeline must fit the requested duration of {seconds} seconds; use "At about N seconds"
  for timed events inside a single shot, or [Shot N] with "At 00:SS.000" for cuts.
- Prefer concrete visual and audio details. Keep every requirement of the user (camera, lens,
  lighting, palette, sound, consistency) and do not add music when the user did not ask for any
  (then write non_diegetic_music: N/A).
- Background lines you add yourself (only in crowd or market scenes) also use speaker IDs and
  <d>[Turkish] ...</d>; say they come from background speakers at a lower volume.

{direction}"""

T2V_TEMPLATE = """You rewrite a user's video request into a prompt for the MiniMax H3 video+audio model.
Follow the official H3 prompt-writing guide below exactly. The task mode is T2VA (text only, no
reference pictures), so the prompt starts directly with the three core fields
integrated_multimodal_description, overall_soundscape, non_diegetic_music.

{rules}

===== OFFICIAL GUIDE: SKILL.md =====
{skill}

===== OFFICIAL GUIDE: references/base-en.txt =====
{guide}

===== WORKED EXAMPLE (14-second request) =====
User request:
{example_in}

Final prompt:
{example_out}
"""

REF_TEMPLATE = """You rewrite a user's video request into a prompt for the MiniMax H3 video+audio model in
full-reference mode (Ref2VA). The user attached {n} reference picture(s); they are given to you in
order and are called <Picture 1> ... <Picture {n}>. Follow the official full-reference guide below
exactly and write the six sections subject_definitions, summary, retention_analysis,
detailed_description, overall_soundscape, non_diegetic_music.

Additional rules for reference pictures:
- Look at every picture carefully. In subject_definitions, define the person or object of each picture
  as a <Subject N> with "Reference role: identity anchor" (for a person) and list the visible features
  to follow exactly: face shape, eyes, eyebrows, facial marks/scars, nose, lips, skin tone, hair colour,
  length and style, approximate age, and—when the user's request does not ask for different clothing or
  setting—their exact outfit, hat, eyewear/goggles, props and scene environment from <Picture N>.
  Mention every <Picture N> at least once.
- Clothing, props and setting come from the user's request when they differ from the picture (then say that
  only the identity is taken from the picture and that its clothing/background is not carried over); when
  the user's request does not specify different clothing or setting, preserve the exact outfit, accessories
  and environment shown in <Picture N> and anchor the opening shot to <Picture N>.
- In detailed_description keep the referenced identity and outfit consistent in every moment, including when
  other people or obstacles pass in front of the subject.
- When a picture is labelled with a role, follow it exactly (the rules above about people apply to characters):
  * character N: a person; define them as a <Subject> with "Reference role: identity anchor"; when they speak,
    use speaker ID (SN) and keep the given name.
  * scene/background: define it as an environment <Subject> with "Reference role: scene anchor"; reproduce its
    architecture, layout, materials, colour palette and light; all characters act inside this place; never
    create a person from this picture and never use it as an identity anchor.
  * object/prop: define it as an object <Subject> with "Reference role: prop reference"; keep its exact shape,
    colours and markings, and show it being used as the request describes.

{rules}

===== OFFICIAL GUIDE: SKILL.md =====
{skill}

===== OFFICIAL GUIDE: references/ref-en.txt =====
{guide}
"""

BRIEF_TEMPLATE = """You are a film director's assistant. Turn a short video idea into a detailed, vivid
one-paragraph brief for an AI video model, like the examples below: setting and time of day, subjects and
their look, action beat by beat, camera (shot type, lens, movement), lighting, colour palette, sound, style,
and any consistency requirements (text that must stay legible, faces that must not morph...).
Rules: write in English; keep every quoted text and spoken line from the idea exactly as written
(same language, same letters, same punctuation) inside double quotes; keep the idea's intent and
do not add story elements that contradict it; the video lasts {seconds} seconds; output only the brief.{ref_note}
For a real, named place, describe it as it really looks (its architecture, the shops and goods that really
fill it, its light, people and sounds) from your own knowledge. Props the idea mentions (a spice cart, a tea
tray) are added as single moving elements, not as the shops around the subject.
Do not move the scene to a specific named place (a city, district, landmark, strait, coast) the idea does not name;
a generic place ("a seaside tea garden") stays generic.

{direction}

{examples}"""


_GUIDE_BYTES = {}


def _git_blob_sha(data):
    return hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()


def load_guide(cache_dir=None):
    """{dosya adı: metin}: SKILL.md, base-en.txt, ref-en.txt (ilk kullanımda indirilir ve doğrulanır)."""
    cache_dir = cache_dir or config.CACHE_DIR
    os.makedirs(cache_dir, exist_ok=True)
    texts = {}
    for path, sha in config.SKILL_FILES.items():
        local = os.path.join(cache_dir, sha)
        if os.path.isfile(local):
            data = open(local, "rb").read()
        elif sha in _GUIDE_BYTES:
            data = _GUIDE_BYTES[sha]
        else:
            url = f"https://raw.githubusercontent.com/{config.SKILL_REPO}/{config.SKILL_COMMIT}/{path}"
            data = urllib.request.urlopen(url, timeout=60).read()
        if _git_blob_sha(data) != sha:
            raise RuntimeError(f"Skill dosyası doğrulanamadı: {path}")
        _GUIDE_BYTES[sha] = data
        open(local, "wb").write(data)
        texts[os.path.basename(path)] = data.decode("utf-8")
    return texts


def system_prompt(seconds, guides, n_images=0, style=config.DEFAULT_STYLE, audio_mode=config.DEFAULT_AUDIO_MODE):
    rules = COMMON_RULES.format(seconds=seconds, direction=direction_rules(style, n_images, audio_mode))
    if n_images:
        return REF_TEMPLATE.format(n=n_images, rules=rules, skill=guides["SKILL.md"].strip(),
                                   guide=guides["ref-en.txt"].strip())
    return T2V_TEMPLATE.format(rules=rules, skill=guides["SKILL.md"].strip(), guide=guides["base-en.txt"].strip(),
                               example_in=EXAMPLE_INPUT, example_out=EXAMPLE_OUTPUT)


def brief_prompt(seconds, n_images=0, style=config.DEFAULT_STYLE, audio_mode=config.DEFAULT_AUDIO_MODE):
    examples = "\n\n".join(f"Example idea:\n{a}\n\nExample brief:\n{b}" for a, b in BRIEF_EXAMPLES)
    note = ("" if not n_images else
            f"\nThe user also attached {n_images} reference picture(s) (Picture 1 ... Picture {n_images}); refer to the "
            "person in them as \"the woman/man from reference picture N\" and do not describe a different face. "
            "Pictures may be labelled with a role: a scene/background picture is the setting (describe that place, "
            "never a person from it) and an object picture is a prop.")
    return BRIEF_TEMPLATE.format(seconds=seconds, ref_note=note, direction=direction_rules(style, n_images, audio_mode),
                                 examples=examples)


QUOTE_RE = re.compile(r'"([^"]+)"|\u201c([^\u201d]+)\u201d')
CUT_SPLIT_RE = re.compile(r"\[\s*cut\s*\]", re.I)
AUDIO_LINE_RE = re.compile(r"(?:^|\n)\s*Audio\s*:\s*(.+)$", re.I | re.S)


def _norm_quotes(text):
    """Kıvrık kesme ve tırnak işaretlerini (’ ‘ “ ”) ASCII karşılıklarına normalize eder."""
    return (text or "").replace("\u2018", "'").replace("\u2019", "'").replace("\u02bc", "'").replace("\u201c", '"').replace("\u201d", '"')


def literals(text):
    """Kullanıcı metnindeki tırnak içi her ifade (ekran yazısı, diyalog)."""
    return [a or b for a, b in QUOTE_RE.findall(text or "")]


def split_user_cuts(text):
    """Kullanıcı metninde `[cut]` varsa (çekim_listesi, genel_ses_notu) döner; yoksa ([], '')."""
    raw = (text or "").strip()
    if not CUT_SPLIT_RE.search(raw):
        return [], ""
    audio_note = ""
    m_aud = AUDIO_LINE_RE.search(raw)
    if m_aud:
        audio_note = m_aud.group(1).strip()
        raw = raw[:m_aud.start()].strip()
    parts = [p.strip() for p in CUT_SPLIT_RE.split(raw) if p and p.strip()]
    return (parts if len(parts) >= 2 else []), audio_note


def clean(text):
    text = (text or "").strip()
    return re.sub(r"^```[a-zA-Z]*\n|\n```$", "", text).strip()


CROWD_RE = re.compile(r"\b(?:crowd\w*|market\w*|bazaar\w*|souk|festival\w*|station|vendors?|merchants?|"
                      r"busy|packed|bargain\w*|passers?-?by|kalabal\w*|pazar\w*|çarşı\w*|panayır\w*|esnaf\w*|"
                      r"istasyon\w*|meydan\w*|izdiham\w*)", re.I)


def check(prompt, request, n_images=0, style=config.DEFAULT_STYLE, audio_mode=config.DEFAULT_AUDIO_MODE, roles=None):
    """Prompt'taki sorunların listesi; boş liste kullanılabilir demektir."""
    fields = REF_FIELDS if n_images else T2V_FIELDS
    problems = []
    pos = [prompt.find(f) for f in fields]
    if not prompt.startswith(fields[0]):
        problems.append(f"Prompt '{fields[0]}' ile başlamalı.")
    if any(p < 0 for p in pos):
        problems.append("Eksik bölüm: " + ", ".join(f for f, p in zip(fields, pos) if p < 0))
    elif pos != sorted(pos):
        problems.append("Bölümlerin sırası yanlış.")
    norm_prompt = _norm_quotes(prompt)
    for lit in literals(request):
        if lit.strip() and _norm_quotes(lit.strip()) not in norm_prompt:
            problems.append(f'Şu metin harfi harfine korunmamış: "{lit.strip()}"')
    for i in range(1, n_images + 1):
        if f"<Picture {i}>" not in prompt:
            problems.append(f"<Picture {i}> prompt'ta hiç anılmıyor.")
    if roles and n_images:
        defs = prompt[:prompt.find("summary:")] if "summary:" in prompt else prompt
        for i, spec in enumerate(normalize_roles(roles, n_images), 1):
            if spec["role"] == ROLE_CHARACTER:
                continue
            lines = [ln for ln in defs.splitlines() if f"<Picture {i}>" in ln]
            if any(re.search(r"identity anchor", ln, re.I) for ln in lines):
                kind = "arka plan/mekân" if spec["role"] == ROLE_SCENE else "nesne"
                problems.append(f"<Picture {i}> bir {kind} görseli ama kişi gibi 'identity anchor' olarak tanımlanmış; "
                                f"{'scene anchor' if spec['role'] == ROLE_SCENE else 'prop reference'} olmalı.")
    pov = re.compile(r"\b(point[- ]of[- ]view|POV|first[- ]person|from (?:his|her|their) (?:eyes|perspective))\b", re.I)
    if pov.search(prompt) and not re.search(pov.pattern + r"|bakış açısı|gözünden|birinci şahıs", request or "", re.I):
        problems.append("Kullanıcı istemediği halde bakış açısı (POV) çekimi yazılmış; kamera hiçbir karakterin gözünden "
                        "bakmamalı, kadraja fazladan el girmemeli.")
    if re.search(r"\bheel", prompt, re.I) and not re.search(r"\bheel|topuk", request, re.I):
        problems.append("Kullanıcı istemediği halde topuklu ayakkabı / topuk sesi yazılmış; düz bot ve yumuşak adım sesi kullan.")
    if not literals(request) and not re.search(
            r"\b(sign\w*|banner\w*|board\w*|text|letter\w*|neon|menu|chalkboard|tabela\w*|yazı\w*|afiş\w*|pano\w*|menü\w*)\b",
            request or "", re.I):
        signs = [m.group(0) for m in re.finditer(
            r"\b(?:(?:shop(?:front)?|storefront|neon|wooden|hanging)\s+signs?\b|"
            r"(?:sign(?:board)?s?|banners?|boards?|plaques?|placards?)\s+(?:that\s+)?(?:reads?|reading|with\s+(?:text|lettering|words)|bearing|displaying|saying|inscribed))",
            prompt, re.I)
            if not re.search(r"\b(no|not|non|without|avoid\w*|never|zero|free\s+of)\b[^.;:]{0,50}$",
                             prompt[max(0, m.start() - 60):m.start()], re.I)]
        if signs:
            problems.append(f"Kullanıcı yazı/tabela istemediği halde arka planda tabela veya yazı tarif edilmiş ({signs[0]}); "
                            f'kaldır ve "{_SIGNAGE_NONE}" kuralını uygula.')
    # 17.11.2026 sahaf testi: iki kişilik sessiz sahneye "background speakers" eklendi, fazladan ses son çekimi geciktirdi
    if re.search(r"background (?:speakers?|voices?|lines?|chatter)", prompt, re.I) and not CROWD_RE.search(request or ""):
        problems.append("Kullanıcının sahnesinde kalabalık/pazar yok ama arka plan konuşması eklenmiş; o replikleri ve "
                        "'background speakers' ifadesini kaldır, yalnızca kullanıcının adlandırdığı karakterler konuşsun.")
    if audio_mode == "music":
        if "<d>" in prompt or "</d>" in prompt:
            problems.append("Müzikli konuşmasız modda konuşma (<d>...</d>) yazılmamalı; kaldır.")
        if "no voices" not in prompt.lower() and "no speech" not in prompt.lower():
            problems.append(f'Müzikli konuşmasız modda overall_soundscape içinde "{_MUSIC_SILENT}" belirtilmeli.')
    s = config.STYLES.get(style) or {}
    if s.get("keywords"):
        if not any(k in prompt.lower() for k in s["keywords"]):
            problems.append(f"Seçilen stil ({s['label']}) prompt'ta yok; ilk açıklama cümlesinde stili tarif et: {s['look']}.")
        # "no photorealism", "avoid live-action" gibi olumsuz kullanımlar serbest
        photo = [m.group(0) for m in re.finditer(r"photoreal\w*|live[- ]action", prompt, re.I)
                 if not re.search(r"\b(no|not|non|without|avoid\w*|never|instead of)\b[^.;:]{0,40}$",
                                  prompt[max(0, m.start() - 60):m.start()], re.I)]
        if photo and not re.search(r"photoreal|live.action|gerçekçi", request, re.I):
            problems.append(f"{s['label']} stili seçiliyken fotoğraf gerçekçiliği yazılmış ({photo[0]}); "
                            "kaldır, gerekirse yalnızca 'no photorealism' biçiminde yaz.")
    if "```" in prompt:
        problems.append("Kod bloğu işareti var.")
    return problems


def gemini_models(api_key):
    """Anahtarın kullanabildiği Gemini metin modelleri: en yeni önce, 'pro' önce."""
    from google import genai
    client = _client(api_key)
    names = []
    for m in client.models.list():
        name = (m.name or "").removeprefix("models/")
        actions = m.supported_actions or []
        if ("generateContent" in actions and name.startswith("gemini")
                and not any(x in name for x in ("tts", "image", "banana", "embedding", "live", "audio", "robotics",
                                                "transcribe", "computer-use", "customtools"))):
            names.append(name)

    def key(n):
        # Önce config.GEMINI_MODEL, sonra "en yeni Pro" takma adı, sonra sürüme göre yeniden eskiye
        ver = re.findall(r"gemini-(\d+(?:\.\d+)?)", n)
        return (n != config.GEMINI_MODEL, n != "gemini-pro-latest", "pro" not in n,
                -(float(ver[0]) if ver else 0), "preview" in n or "exp" in n, n)
    return sorted(set(names), key=key)


ROLE_CHARACTER, ROLE_SCENE, ROLE_OBJECT = "karakter", "arka_plan", "nesne"
ROLES = (ROLE_CHARACTER, ROLE_SCENE, ROLE_OBJECT)


def normalize_roles(roles, n):
    """Her görsel için {'role', 'name', 'char'} listesi (görsel sırasıyla). Rol verilmemişse karakter sayılır;
    karakterler sırayla 1, 2, … numaralanır ve konuşmacı kimliği (S1, S2…) bu numaradır."""
    out, k = [], 0
    for i in range(n):
        r = (roles[i] if roles and i < len(roles) else None) or {}
        role = r.get("role") if r.get("role") in ROLES else ROLE_CHARACTER
        char = None
        if role == ROLE_CHARACTER:
            k += 1
            char = k
        out.append({"role": role, "name": (r.get("name") or "").strip(), "char": char})
    return out


def role_label(spec):
    """Gemini'ye görselin yanında giden kısa rol etiketi (İngilizce)."""
    name = f' "{spec["name"]}"' if spec.get("name") else ""
    if spec["role"] == ROLE_SCENE:
        return f"role: scene/background{name} — the setting, not a person"
    if spec["role"] == ROLE_OBJECT:
        return f"role: object/prop{name}"
    return f"role: character {spec['char']}{name} — speaker S{spec['char']} if they talk"


def _image_parts(images, roles=None):
    from google.genai import types
    parts = []
    specs = normalize_roles(roles, len(images or [])) if roles else None
    for i, path in enumerate(images or [], 1):
        mime = mimetypes.guess_type(path)[0] or "image/png"
        label = f"<Picture {i}> ({role_label(specs[i - 1])}):" if specs else f"<Picture {i}>:"
        parts += [label, types.Part.from_bytes(data=open(path, "rb").read(), mime_type=mime)]
    return parts


class FirstChunkTimeout(TimeoutError):
    """Model ilk metin parçasını FIRST_CHUNK_SECONDS içinde göndermedi."""


TRANSIENT_RE = re.compile(r"\b(503|429|504|500)\b|UNAVAILABLE|RESOURCE_EXHAUSTED|DEADLINE_EXCEEDED|INTERNAL|"
                          r"high demand|overloaded|timed? ?out", re.I)
# Model artık sunulmuyor (404 NOT_FOUND): oturumun geri kalanında hiç denenmez, boşa çağrı gitmez
GONE_RE = re.compile(r"\b404\b|NOT_FOUND|is not found|not supported for generateContent", re.I)
_gone_models = set()
TEXT_TIMEOUT_MS = 180_000   # ilk parça geldikten sonra tüm yanıt için üst sınır
IMAGE_TIMEOUT_MS = 300_000


def _transient(e):
    return isinstance(e, TimeoutError) or "Timeout" in type(e).__name__ or bool(TRANSIENT_RE.search(str(e)))


def gone(e):
    return bool(GONE_RE.search(str(e)))


def mark_gone(model, log=None):
    """404 veren modeli oturum boyunca atlanacaklar listesine ekler (bir kez günlüğe yazar)."""
    if model not in _gone_models:
        _gone_models.add(model)
        if log:
            log(f"🚫 {model} bu hesapta sunulmuyor (404); bu oturumda bir daha denenmeyecek.")


def available(models):
    """Sırayı koruyarak tekrarları ve 404 vermiş modelleri çıkarır."""
    return [m for m in dict.fromkeys(models) if m and m not in _gone_models]


def _client(api_key, timeout_ms=TEXT_TIMEOUT_MS):
    from google import genai
    from google.genai import types
    return genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=timeout_ms))


def _stream_text(api_key, model, system, contents, temperature, first_chunk=None, total=TEXT_TIMEOUT_MS / 1000):
    """Yanıtı akış olarak ister. İlk parça `first_chunk` sn içinde gelmezse FirstChunkTimeout; bekleyen istek
    arka planda kendiliğinden biter, sonucu yok sayılır."""
    import queue
    import threading
    from google.genai import types
    first_chunk = config.FIRST_CHUNK_SECONDS if first_chunk is None else first_chunk
    if model.startswith("gemma"):   # Gemma API'de sistem talimatı yok: talimat mesajın başına konur
        cfg = types.GenerateContentConfig(temperature=temperature)
        contents = [f"Instructions:\n{system}\n\n---\n"] + list(contents)
    else:
        cfg = types.GenerateContentConfig(system_instruction=system, temperature=temperature)
    q, done = queue.Queue(), object()

    def work():
        try:
            client = _client(api_key)   # değişkende tutulmalı: geçici istemci çöpe gidince bağlantıyı kapatıyor
            for chunk in client.models.generate_content_stream(model=model, contents=contents, config=cfg):
                q.put(getattr(chunk, "text", None) or "")
        except Exception as e:
            q.put(e)
        finally:
            q.put(done)

    threading.Thread(target=work, daemon=True, name=f"gemini-{model}").start()
    parts, deadline, got_first = [], None, False
    while True:
        try:
            if not got_first:
                item = q.get(timeout=first_chunk)
            else:
                item = q.get(timeout=max(0.1, deadline - time.time()))
        except Exception:
            if not got_first:
                raise FirstChunkTimeout(f"{model} {first_chunk:g} sn içinde yanıt vermeye başlamadı") from None
            raise TimeoutError(f"{model} yanıtı {total:g} sn içinde bitmedi") from None
        if item is done:
            break
        if isinstance(item, Exception):
            raise item
        if not got_first:
            got_first, deadline = True, time.time() + total
        parts.append(item)
    text = "".join(parts)
    if not text.strip():
        raise RuntimeError(f"{model} boş yanıt döndü (503 UNAVAILABLE gibi davranılır)")
    return text


last_model = {"name": None, "fallback": False}   # son başarılı _generate çağrısının modeli (arayüz gösterir)
used_models = {}   # adım -> metni yazan model ("brief", "H3 prompt'u", "çekim planı"); bilgi.json'a yazılır


def _generate(api_key, model, system, contents, temperature, log=None, step=None):
    """Seçili modelle dener; ilk parça FIRST_CHUNK_SECONDS içinde gelmezse ya da model yoğunsa (503/429/zaman aşımı)
    beklemeden sıradaki yedek modele geçer ve bunu `log` ile arayüze yazar. Kalıcı hatada (ör. geçersiz anahtar) durur."""
    models = available([model, config.GEMINI_MODEL, *config.GEMINI_FALLBACK_MODELS])
    errors = []
    for i, m in enumerate(models):
        try:
            text = _stream_text(api_key, m, system, contents, temperature)
        except Exception as e:
            if gone(e):
                mark_gone(m, log)
            elif not _transient(e):
                raise
            errors.append(f"{m}: {str(e)[:160]}")
            if log and i + 1 < len(models):
                log(f"⚠️ {step + ': ' if step else ''}{m} yanıt vermedi ({str(e)[:120]}); "
                    f"yedek modele geçiliyor: {models[i + 1]}")
            continue
        last_model["name"], last_model["fallback"] = m, m != models[0]
        if step:
            used_models[step] = m + (" (yedek)" if i else "")
        if i and log:
            log(f"↪️ {step or 'Bu adım'} yedek modelle yazıldı: {m} — ilk tercih {models[0]} yanıt vermedi, "
                "çıktı daha zayıf olabilir")
        return clean(text)
    raise RuntimeError("Hiçbir Gemini modeli yanıt vermedi:\n- " + "\n- ".join(errors))


def expand(idea, seconds, api_key, model, images=(), log=print, style=config.DEFAULT_STYLE,
           audio_mode=config.DEFAULT_AUDIO_MODE, roles=None):
    """Kısa fikir → detaylı brief (İngilizce, tırnak içi metinler aynen), seçilen stilde."""
    head = f"Duration: {seconds} seconds.\n\nIdea:\n{idea.strip()}"
    contents = [head] + _image_parts(images, roles)
    return _generate(api_key, model, brief_prompt(seconds, len(images or ()), style, audio_mode), contents, 0.7,
                     log=log, step="brief")


def rewrite(request, seconds, api_key, model, images=(), retries=1, log=print, style=config.DEFAULT_STYLE,
            audio_mode=config.DEFAULT_AUDIO_MODE, roles=None):
    """Brief → H3 prompt'u. (prompt, sorunlar) döner; sorun listesi boşsa tüm denetimlerden geçmiştir."""
    images = list(images or ())
    system = system_prompt(seconds, load_guide(), len(images), style, audio_mode)
    head = f"Duration: {seconds} seconds.\n\nUser request:\n{request.strip()}"
    prompt, problems = "", ["Gemini yanıt vermedi."]
    contents = [head] + _image_parts(images, roles)
    for attempt in range(retries + 1):
        prompt = _generate(api_key, model, system, contents, 0.3, log=log, step="H3 prompt'u")
        problems = check(prompt, request, len(images), style, audio_mode, roles=roles)
        if not problems:
            break
        log(f"Gemini çıktısında sorun ({attempt + 1}. deneme): " + " | ".join(problems))
        contents = [head + f"\n\nYour previous answer:\n{prompt}\n\nFix these problems and output the full "
                    "corrected prompt only:\n- " + "\n- ".join(problems)] + _image_parts(images, roles)
    return prompt, problems


SHOT_KEYS = ("block", "shot", "start_sec", "duration", "framing", "action", "camera", "sound")

PLAN_SHOTS_PROMPT = """You are a film director and editor planning a {total_seconds}-second multi-shot video for the
MiniMax H3 video+audio model. Because H3 generates at most 15 seconds per block, divide the {total_seconds}
seconds into {n_blocks} block(s) (each block between 4 and 15 seconds, summing to {total_seconds} seconds).
Within each block, plan 2 to 4 shots that cut cleanly at exact timestamps (`[Shot N] At 00:SS.000`).

Output ONLY valid JSON (no markdown fences, no commentary) with this exact structure:
{{
  "world_anchor": "2-4 sentences in English describing the unchanging character identity, garments, colours, footwear (flat boots/shoes), setting architecture, time of day, lighting, colour palette, and signage rule that must stay identical in every block.",
  "shots": [
    {{
      "block": 1,
      "shot": 1,
      "start_sec": 0.0,
      "duration": 5.0,
      "framing": "Wide establishing full-body shot, 28mm lens",
      "action": "Concrete visual action in English...",
      "camera": "Camera movement and focus...",
      "sound": "Spoken dialogue in quotes or ambient sound..."
    }}
  ]
}}

Hard rules for planning:
- `block` is 1..{n_blocks}. Within each block, `shot` starts at 1 and increments (1, 2, 3...), and `start_sec`
  is the start time (in seconds) relative to the start of THAT block (so shot 1 of every block has `start_sec: 0.0`).
- Do NOT write `At 00:SS.000` inside `action` or `sound`; the cut time comes from `start_sec`. If you time an
  event or a spoken line inside a shot, write "At about N seconds" counted from the start of THAT SHOT
  (e.g. a line 2 seconds into a shot that starts at 5.0 is "At about 2 seconds").
- Every shot `duration` must be at least 2.0 seconds. Each block's shot durations must sum to ≤ 15.0 seconds
  (and ≥ 4.0 seconds), and the total across all blocks must equal {total_seconds} seconds.
- Shot variety & 180-degree rule: vary framing across shots (wide establishing full-body shot -> medium tracking
  or two-shot -> close-up reaction or detail shot -> wide/medium resolution) while keeping consistent screen
  direction (180-degree axis).
- Tail-drift protection (CRITICAL): the LAST shot of EVERY block MUST end with a calm ~0.5-second hold or
  settling beat (steady gaze, settled posture, or still camera), never in the middle of a fast turn or sudden
  motion. Block boundaries must always fall on a natural scene cut, never splitting one continuous fast motion.
- Keep every quoted on-screen text and spoken line from the user's idea character for character (with Turkish
  letters İ ı Ş ş Ğ ğ Ç ç Ö ö Ü ü intact) inside the appropriate shot's `action` or `sound`.{ref_note}

{direction}"""


def _parse_json(text):
    text = clean(text)
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group(0) if m else text)


def check_plan(plan, total_seconds, idea="", audio_mode=config.DEFAULT_AUDIO_MODE):
    """Blok/çekim planını denetler ve start_sec değerlerini blok içinde normalize eder; sorun listesi döner."""
    problems = []
    if not isinstance(plan, dict):
        return ["Plan bir JSON nesnesi olmalı."]
    anchor = (plan.get("world_anchor") or "").strip()
    shots = plan.get("shots")
    if not anchor:
        problems.append("'world_anchor' (ortak karakter/dünya çapası) boş olamaz.")
    if not isinstance(shots, list) or not shots:
        return problems + ["'shots' boş olmayan bir çekim listesi olmalı."]
    by_block = {}
    for idx, s in enumerate(shots, 1):
        if not isinstance(s, dict):
            problems.append(f"{idx}. çekim geçerli bir nesne değil.")
            continue
        missing = [k for k in SHOT_KEYS if k not in s]
        if missing:
            problems.append(f"{idx}. çekimde eksik alan: {', '.join(missing)}")
            continue
        try:
            b = int(s["block"])
            dur = round(float(s["duration"]), 2)
        except Exception:
            problems.append(f"{idx}. çekimde block/duration sayısal değil.")
            continue
        if dur < config.MIN_SHOT_SECONDS:
            problems.append(f"Blok {b} çekim {s.get('shot')}: süre en az {config.MIN_SHOT_SECONDS} sn olmalı ({dur} sn).")
        by_block.setdefault(b, []).append(s)
    if problems:
        return problems
    total = 0.0
    for b in sorted(by_block):
        cur = 0.0
        for j, s in enumerate(by_block[b], 1):
            s["block"] = b
            s["shot"] = j
            s["start_sec"] = round(cur, 2)
            dur = round(float(s["duration"]), 2)
            s["duration"] = dur
            cur += dur
        if cur > config.MAX_SECONDS + 0.05:
            problems.append(f"Blok {b} toplam süresi 15 saniyeyi aşıyor ({cur:.1f} sn); en fazla 15 sn olmalı.")
        elif cur < config.MIN_SECONDS - 0.05:
            problems.append(f"Blok {b} toplam süresi çok kısa ({cur:.1f} sn); en az {config.MIN_SECONDS} sn olmalı.")
        total += cur
    if abs(total - float(total_seconds)) > 0.6:
        problems.append(f"Toplam süre {total_seconds} sn olmalı ama çekimlerin toplamı {total:.1f} sn.")
    user_cuts, _ = split_user_cuts(idea)
    if user_cuts and len(shots) != len(user_cuts):
        problems.append(f"Kullanıcı [cut] ile {len(user_cuts)} çekim verdi ama planda {len(shots)} çekim var; "
                        f"tam {len(user_cuts)} çekim olmalı.")
    all_text = _norm_quotes(json.dumps(plan, ensure_ascii=False))
    for lit in literals(idea):
        if lit.strip() and _norm_quotes(lit.strip()) not in all_text:
            problems.append(f'Şu tırnak içi metin planda korunmamış: "{lit.strip()}"')
    if audio_mode == "music":
        for s in shots:
            if "<d>" in (s.get("sound") or ""):
                problems.append("Müzikli konuşmasız modda çekim sesinde konuşma (<d>...</d>) olmamalı.")
                break
    return problems


def plan_shots(idea, total_seconds, api_key, model, images=(), style=config.DEFAULT_STYLE,
               audio_mode=config.DEFAULT_AUDIO_MODE, retries=1, log=print, roles=None):
    """Kısa fikir + toplam süre → {'world_anchor': ..., 'shots': [...]}. Bozuk JSON veya süre hatasında düzeltir."""
    total_seconds = int(round(float(total_seconds)))
    n_blocks = max(1, math.ceil(total_seconds / float(config.MAX_SECONDS)))
    ref_note = ("" if not images else
                f"\nThe user attached {len(images)} reference picture(s) (<Picture 1> ... <Picture {len(images)}>); "
                "anchor the main character's identity, exact outfit, accessories and scene environment to the reference "
                "picture(s) in `world_anchor`. Pictures labelled scene/background define the place (never a person); "
                "pictures labelled object are props.")
    user_cuts, audio_note = split_user_cuts(idea)
    if user_cuts:
        per_b = math.ceil(len(user_cuts) / float(n_blocks))
        cut_lines = []
        for idx, ctext in enumerate(user_cuts, 1):
            b_idx = min(n_blocks, (idx - 1) // per_b + 1)
            s_idx = (idx - 1) % per_b + 1
            cut_lines.append(f"  - Global Shot {idx} -> Block {b_idx}, Shot {s_idx}: {ctext}")
        ref_note += (
            f"\n\nCRITICAL — Explicit `[cut]` script provided by the user ({len(user_cuts)} total shots across "
            f"{n_blocks} block(s)):\nYou MUST output EXACTLY {len(user_cuts)} shots in the `shots` array in this exact "
            "order and block assignment, preserving every requested camera framing and camera move, every action and every spoken line "
            "verbatim:\n"
            + "\n".join(cut_lines)
        )
        if audio_note:
            ref_note += f"\nGlobal audio note from user (incorporate across shots' `sound` fields): {audio_note}"
    system = PLAN_SHOTS_PROMPT.format(
        total_seconds=total_seconds, n_blocks=n_blocks, ref_note=ref_note,
        direction=direction_rules(style, len(images or ()), audio_mode),
    )
    head = f"Total duration: {total_seconds} seconds ({n_blocks} blocks).\nAudio mode: {audio_mode}.\n\nIdea:\n{idea.strip()}"
    contents = [head] + _image_parts(images, roles)
    plan, problems = {"world_anchor": "", "shots": []}, ["Gemini yanıt vermedi."]
    for attempt in range(retries + 1):
        raw = _generate(api_key, model, system, contents, 0.4, log=log, step="çekim planı")
        try:
            plan = _parse_json(raw)
            problems = check_plan(plan, total_seconds, idea=idea, audio_mode=audio_mode)
        except Exception as e:
            problems = [f"Geçerli JSON üretilmedi: {e}"]
        if not problems:
            break
        log(f"Plan çıktısında sorun ({attempt + 1}. deneme): " + " | ".join(problems))
        contents = [head + f"\n\nYour previous answer:\n{raw}\n\nFix these problems and output ONLY valid JSON:\n- "
                    + "\n- ".join(problems)] + _image_parts(images, roles)
    return plan, problems


def blocks_from_plan(plan):
    """Plan içindeki çekimleri blok numarasına göre gruplar: [{'block', 'seconds', 'world_anchor', 'shots'}]."""
    anchor = (plan or {}).get("world_anchor", "").strip()
    by_block = {}
    for s in (plan or {}).get("shots") or []:
        b = int(s.get("block", 1))
        by_block.setdefault(b, []).append(dict(s))
    blocks = []
    for b in sorted(by_block):
        shots = by_block[b]
        cur = 0.0
        for j, s in enumerate(shots, 1):
            s["shot"] = j
            s["start_sec"] = round(cur, 2)
            cur += float(s.get("duration", 4.0))
        sec = max(config.MIN_SECONDS, min(config.MAX_SECONDS, int(round(cur))))
        blocks.append({"block": b, "seconds": sec, "world_anchor": anchor, "shots": shots})
    return blocks


_AT_ABOUT_RE = re.compile(r"\b([Aa]t about )(\d+(?:\.\d+)?)( seconds?)")
_SHOT_TS_RE = re.compile(r"^\s*At \d{1,2}:\d{2}(?:\.\d+)?,?\s*", re.I)


def absolute_times(text, start, dur):
    """Çekim içi "At about N seconds" zamanlarını blok başından mutlak zamana çevirir (17.11.2026 balıkçı testi:
    plan çekime göre göreli yazdı, H3 klibin başından okur; replikler önceki çekime kayacaktı). Değer zaten
    çekimin mutlak aralığındaysa dokunulmaz; çekim süresinden kısaysa göreli sayılıp çekim başlangıcı eklenir."""
    text = _SHOT_TS_RE.sub("", text or "")
    if start <= 0:
        return text

    def fix(m):
        n = float(m.group(2))
        if start <= n < start + dur or n >= dur:
            return m.group(0)
        return f"{m.group(1)}{start + n:g}{m.group(3)}"
    return _AT_ABOUT_RE.sub(fix, text)


def block_request(block_info, style=config.DEFAULT_STYLE, audio_mode=config.DEFAULT_AUDIO_MODE, idea=""):
    """Tek bir bloğun dünya çapası ve çekimlerini `rewrite()` için detaylı yönetmen brief'ine dönüştürür."""
    b = block_info["block"]
    sec = block_info["seconds"]
    anchor = block_info.get("world_anchor", "").strip()
    shots = block_info.get("shots") or []
    lines = [f"Block {b} ({sec} seconds total, {len(shots)} shot(s))."]
    if anchor:
        lines.append(f"World and character anchor (must remain identical across all blocks): {anchor}")
    lines.append(f"Signage rule: {signage_rule(idea or anchor)}")
    for j, s in enumerate(shots, 1):
        start = float(s.get("start_sec", 0.0))
        dur = float(s.get("duration", sec))
        mm, ss = divmod(start, 60.0)
        ts = f"At {int(mm):02d}:{ss:06.3f}" if j > 1 else "At 00:00.000 (opening shot)"
        hold = " Ends with a calm ~0.5-second hold/settling beat before the block ends." if j == len(shots) else ""
        action = absolute_times(str(s.get("action", "")), start, dur)
        sound = absolute_times(str(s.get("sound", "")), start, dur)
        lines.append(
            f"[Shot {j}] {ts} ({dur:.1f}s) — Framing: {s.get('framing', '')}. "
            f"Action: {action}.{hold} Camera: {s.get('camera', '')}. Sound: {sound}."
        )
    if audio_mode == "music":
        lines.append(f"Audio mode: music montage with no voices. In overall_soundscape write '{_MUSIC_SILENT}' and in non_diegetic_music write N/A.")
    return "\n".join(lines)


def rewrite_blocks(plan, api_key, model, images=(), retries=1, log=print, style=config.DEFAULT_STYLE,
                   audio_mode=config.DEFAULT_AUDIO_MODE, idea="", roles=None):
    """Plandaki her blok için H3 prompt'u üretir: [{'block', 'seconds', 'request', 'prompt', 'problems'}]."""
    out = []
    for binfo in blocks_from_plan(plan):
        req = block_request(binfo, style=style, audio_mode=audio_mode, idea=idea)
        log(f"Blok {binfo['block']} ({binfo['seconds']} sn, {len(binfo['shots'])} çekim) H3 prompt'u yazılıyor…")
        prompt, probs = rewrite(req, binfo["seconds"], api_key, model, images=images, retries=retries, log=log,
                                style=style, audio_mode=audio_mode, roles=roles)
        out.append({"block": binfo["block"], "seconds": binfo["seconds"], "request": req,
                    "prompt": prompt, "problems": probs})
    return out


def shots_to_rows(shots):
    """Çekim listesini Gradio Dataframe satırlarına çevirir."""
    return [[int(s.get("block", 1)), int(s.get("shot", i + 1)), round(float(s.get("start_sec", 0.0)), 2),
             round(float(s.get("duration", 5.0)), 2), str(s.get("framing", "")), str(s.get("action", "")),
             str(s.get("camera", "")), str(s.get("sound", ""))]
            for i, s in enumerate(shots or [])]


def rows_to_shots(rows):
    """Gradio Dataframe satırlarını (list veya pandas DataFrame) çekim sözlüklerine çevirir."""
    if hasattr(rows, "values"):
        rows = rows.values.tolist()
    shots = []
    for r in rows or []:
        if not r or len(r) < 8 or (str(r[4]).strip() == "" and str(r[5]).strip() == ""):
            continue
        shots.append({
            "block": int(float(r[0] or 1)),
            "shot": int(float(r[1] or 1)),
            "start_sec": round(float(r[2] or 0.0), 2),
            "duration": round(float(r[3] or 5.0), 2),
            "framing": str(r[4] or "").strip(),
            "action": str(r[5] or "").strip(),
            "camera": str(r[6] or "").strip(),
            "sound": str(r[7] or "").strip(),
        })
    return shots


# Test 4: yalnız yüzü dolduran portre referansıyla anime videosu göğüs planda kaldı; karakter baştan ayağa ve
# videodaki kıyafetiyle çizilince referans, istenen kadraja ve kıyafete de örnek olur.
STYLIZE_PROMPT = """Redraw the person in this photo as a full-body character in this style: {look}.
Keep the same identity: face shape, eyes, eyebrows, nose, lips, skin tone, hair colour, length and
style, approximate age. The person is drawn as one of the {people}; stylise the face, skin and hair
completely{face_texture} so the face is never semi-realistic or photographic. Avoid: {avoid}.
Show the whole figure from head to feet, standing in a relaxed neutral pose, facing three-quarters toward the
viewer, in front of a simple plain background, with even, soft light.
Clothing: {clothing}
No text, no letters, no logos, no watermark."""
CLOTHING_FROM_IDEA = ("dress the person exactly as the video idea below describes (garments, colours, accessories, "
                      "flat shoes or boots unless it says otherwise); keep the photo's clothing only for what the idea "
                      "does not mention.\nVideo idea: {idea}")
CLOTHING_FROM_PHOTO = "the same clothing and colours as in the photo, with flat shoes."

STYLIZE_SCENE_PROMPT = """Redraw this place as a background plate in this style: {look}. The background is {background}.
Keep the same place: its architecture, layout, perspective, materials, objects, colour palette and light.
Remove every person; show the empty set, ready for characters to act in it. Nothing may look photographic.
Avoid: {avoid}. No text, no letters, no logos, no watermark."""
STYLIZE_OBJECT_PROMPT = """Redraw this object in this style: {look}. Keep its exact shape, proportions, colours and
markings, on a simple plain background with even, soft light. No people. Avoid: {avoid}.
No text, no letters, no logos, no watermark."""


def image_models(api_key):
    """Görsel üretebilen Gemini modelleri (adında 'image' geçen), en yeni önce."""
    from google import genai
    client = _client(api_key)
    names = [(m.name or "").removeprefix("models/") for m in client.models.list()]
    names = [n for n in names if n.startswith("gemini") and "image" in n]
    ver = lambda n: -(float(re.findall(r"gemini-(\d+(?:\.\d+)?)", n)[0]) if re.findall(r"gemini-(\d+(?:\.\d+)?)", n) else 0)
    return sorted(set(names), key=lambda n: (ver(n), "preview" in n, n))


def stylize_reference(path, style, api_key, model, dest_dir, variants=2, idea="", role=None):
    """Referans fotoğrafı seçilen stilde yeniden çizdirir (scenario-alter'in "target frame" yöntemi: model,
    stilin bitmiş bir örneğini tarif edilmiş halinden çok daha iyi taklit eder). Kaydedilen dosya yolları döner."""
    from google import genai
    from google.genai import types
    s = config.STYLES[style]
    client = _client(api_key, IMAGE_TIMEOUT_MS)
    mime = mimetypes.guess_type(path)[0] or "image/png"
    clothing = CLOTHING_FROM_IDEA.format(idea=idea.strip()) if (idea or "").strip() else CLOTHING_FROM_PHOTO
    face_texture = " with visible brushwork and paper texture" if style == "watercolour" else " in this style"
    if role == ROLE_SCENE:
        text = STYLIZE_SCENE_PROMPT.format(**s)
    elif role == ROLE_OBJECT:
        text = STYLIZE_OBJECT_PROMPT.format(**s)
    else:
        text = STYLIZE_PROMPT.format(clothing=clothing, face_texture=face_texture, **s)
    contents = [text, types.Part.from_bytes(data=open(path, "rb").read(), mime_type=mime)]
    cfg = types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"], temperature=1.0)
    os.makedirs(dest_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(path))[0]
    out = []
    for v in range(variants):
        resp = client.models.generate_content(model=model, contents=contents, config=cfg)
        for cand in resp.candidates or []:
            for part in (cand.content.parts if cand.content else None) or []:
                if getattr(part, "inline_data", None) and part.inline_data.data:
                    ext = mimetypes.guess_extension(part.inline_data.mime_type or "image/png") or ".png"
                    dest = os.path.join(dest_dir, f"{stem}_{style}_{chr(65 + v)}{ext}")
                    open(dest, "wb").write(part.inline_data.data)
                    out.append(dest)
                    break
            else:
                continue
            break
    if not out:
        raise RuntimeError("Gemini görsel döndürmedi (model görsel üretmeyi desteklemiyor olabilir).")
    return out

In [ ]:
%%writefile /content/h3_video_app/h3app/graphs.py
"""ComfyUI API grafikleri: resmi `video_pika_h3_flex_t2v` ve `video_minimax_h3_t2v`
şablonlarının birebir karşılığı (yerel ComfyUI'ın /prompt doğrulamasından geçti)."""

import os

from . import config


def _name(role):
    return os.path.basename(config.MODELS[role][0])


def build(method, prompt, seed, width, height, frames, text_encoder="te_nvfp4", prefix="h3app/video", ref_images=()):
    """ref_images boşsa metinden video (T2VA); doluysa ComfyUI/input altındaki görsellerle referanslı video
    (Ref2VA, resmi `video_pika_h3_flex_ref2va` / `video_minimax_h3_r2v` şablonları)."""
    if method not in config.METHODS:
        raise ValueError(f"Bilinmeyen yöntem: {method}")
    ref = bool(ref_images)
    g = {
        "unet": {"class_type": "UNETLoader", "inputs": {"unet_name": _name("unet_ref" if ref else "unet"), "weight_dtype": "default"}},
        "clip": {"class_type": "CLIPLoader", "inputs": {"clip_name": _name(text_encoder), "type": "minimax", "device": "default"}},
        "vae_v": {"class_type": "VAELoader", "inputs": {"vae_name": _name("vae_video")}},
        "vae_a": {"class_type": "VAELoader", "inputs": {"vae_name": _name("vae_audio")}},
        "cond": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {"clip": ["clip", 0], "vae": ["vae_v", 0], "prompt": prompt,
                                                                    "width": width, "height": height, "length": frames}},
        "noise": {"class_type": "RandomNoise", "inputs": {"noise_seed": int(seed)}},
        "decode_v": {"class_type": "VAEDecode", "inputs": {"samples": ["sample", 0], "vae": ["vae_v", 0]}},
        "decode_a": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["sample", 0], "vae": ["vae_a", 0]}},
        "video": {"class_type": "CreateVideo", "inputs": {"images": ["decode_v", 0], "audio": ["decode_a", 0], "fps": 24}},
        "save": {"class_type": "SaveVideo", "inputs": {"video": ["video", 0], "filename_prefix": prefix,
                                                        "format": "mp4", "format.codec": "h264", "format.codec.encoding": "auto"}},
    }
    if ref:
        if len(ref_images) > config.MAX_REF_IMAGES:
            raise ValueError(f"En fazla {config.MAX_REF_IMAGES} referans görsel kullanılabilir.")
        cond = {"clip": ["clip", 0], "vae": ["vae_v", 0], "audio_vae": ["vae_a", 0], "prompt": prompt,
                "width": width, "height": height, "length": frames, "ref_image_size": "match"}
        for i, name in enumerate(ref_images):
            g[f"img{i}"] = {"class_type": "LoadImage", "inputs": {"image": name}}
            cond[f"ref_images.ref_image_{i}"] = [f"img{i}", 0]
        g["cond"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": cond}
    if method == "pika":
        g["lora"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["unet", 0],
                     "lora_name": _name("pika_lora_ref" if ref else "pika_lora"), "strength_model": 1.0}}
        g["sampler"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "lcm"}}
        g["sigmas"] = {"class_type": "ManualSigmas", "inputs": {"sigmas": config.PIKA_SIGMAS}}
        model = ["lora", 0]
    elif method == "turbo":
        g["lora"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["unet", 0],
                     "lora_name": _name("turbo_lora_ref" if ref else "turbo_lora"), "strength_model": 1.0}}
        g["sampler"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "res_multistep"}}
        steps = config.TURBO_REF_STEPS if ref else config.TURBO_STEPS
        g["sigmas"] = {"class_type": "BasicScheduler", "inputs": {"model": ["lora", 0], "scheduler": "simple",
                                                                  "steps": steps, "denoise": 1.0}}
        model = ["lora", 0]
    else:
        g["sampler"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "res_multistep"}}
        g["sigmas"] = {"class_type": "BasicScheduler", "inputs": {"model": ["unet", 0], "scheduler": "simple",
                                                                  "steps": config.H3_STEPS, "denoise": 1.0}}
        model = ["unet", 0]
    g["guider"] = {"class_type": "BasicGuider", "inputs": {"model": model, "conditioning": ["cond", 0]}}
    g["sample"] = {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["noise", 0], "guider": ["guider", 0],
                                                                      "sampler": ["sampler", 0], "sigmas": ["sigmas", 0],
                                                                      "latent_image": ["cond", 1]}}
    return g

In [ ]:
%%writefile /content/h3_video_app/h3app/setup_env.py
"""Colab runtime hazırlığı: ComfyUI kurulumu ve model dosyalarının indirilip doğrulanması."""

import hashlib
import os
import queue
import shutil
import subprocess
import sys
import threading
import time

from . import config


def gpu():
    """(ad, bellek GB, blackwell_mi). GPU yoksa None."""
    try:
        import torch
    except ImportError:
        return None
    if not torch.cuda.is_available():
        return None
    p = torch.cuda.get_device_properties(0)
    return p.name, round(p.total_memory / 1e9), torch.cuda.get_device_capability(0)[0] >= 10


def text_encoder_role():
    """NVFP4 metin kodlayıcı Blackwell (ör. G4 / RTX PRO 6000) ister; diğer GPU'larda int8 sürümü."""
    g = gpu()
    return "te_nvfp4" if g and g[2] else "te_int8"


def ensure_comfy(log=print):
    if not os.path.isdir(os.path.join(config.COMFY_DIR, ".git")):
        log("ComfyUI indiriliyor…")
        subprocess.run(["git", "clone", "-q", config.COMFY_REPO, config.COMFY_DIR], check=True)
    subprocess.run(["git", "-C", config.COMFY_DIR, "fetch", "-q", "--all"], check=False)
    subprocess.run(["git", "-C", config.COMFY_DIR, "checkout", "-q", config.COMFY_COMMIT], check=True)
    stamp = os.path.join(config.COMFY_DIR, ".h3app_requirements")
    if not os.path.isfile(stamp):
        log("ComfyUI bağımlılıkları kuruluyor (~1 dk)…")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                            os.path.join(config.COMFY_DIR, "requirements.txt")], capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError("ComfyUI kurulamadı:\n" + r.stderr[-2000:])
        open(stamp, "w").write(config.COMFY_COMMIT)
    log(f"ComfyUI hazır ({config.COMFY_COMMIT}).")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(64 << 20), b""):
            h.update(b)
    return h.hexdigest()


def _index_dirs(dirs):
    """Kaynak klasörlerdeki dosyalar: ad -> yollar (Drive'da 01/03 notebook'larının indirdiği kopyalar)."""
    idx = {}
    for root in dirs or ():
        if not os.path.isdir(root):
            continue
        for dp, _, names in os.walk(root):
            for n in names:
                idx.setdefault(n, []).append(os.path.join(dp, n))
    return idx


def _copy_verified(src, dst, size, sha, log):
    """Kaynaktan yerel diske kopyalar ve boyut + SHA-256 doğrular; tutmazsa siler ve False döner."""
    part = dst + ".part"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        shutil.copyfile(src, part)
        if os.path.getsize(part) == size and _sha256(part) == sha:
            os.replace(part, dst)
            return True
        log(f"Drive kopyası doğrulanamadı, Hugging Face'ten indirilecek: {os.path.basename(dst)}")
    except OSError as e:
        log(f"Drive'dan kopyalanamadı ({e}); Hugging Face'ten indirilecek.")
    if os.path.exists(part):
        os.remove(part)
    return False


_save_queue = queue.Queue()
_save_worker = None
_save_pending = []


def _save_loop():
    while True:
        job = _save_queue.get()
        try:
            _save_one(*job)
        finally:
            _save_queue.task_done()


def _save_one(src, dest, size, log):
    """Doğrulanmış yerel model dosyasını Drive'a kopyalar (.part → ad). Yer yoksa ya da hata olursa atlar."""
    name = os.path.basename(dest)
    try:
        if os.path.isfile(dest) and os.path.getsize(dest) == size:
            return
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        free = shutil.disk_usage(os.path.dirname(dest)).free
        if free < size * 1.02:
            log(f"💾 Drive'da yer yok ({free / 1e9:.1f} GB boş, {size / 1e9:.1f} GB gerekli); "
                f"{name} Drive'a kaydedilmedi.")
            return
        part = dest + ".part"
        shutil.copyfile(src, part)
        if os.path.getsize(part) != size:
            os.remove(part)
            log(f"💾 Drive kopyası eksik çıktı, silindi: {name}")
            return
        os.replace(part, dest)
        log(f"💾 Drive'a kaydedildi: {name} (sonraki oturumda buradan alınır)")
    except OSError as e:
        if os.path.exists(dest + ".part"):
            try:
                os.remove(dest + ".part")
            except OSError:
                pass
        log(f"💾 {name} Drive'a kaydedilemedi ({e}); üretim etkilenmez.")


def queue_model_save(src, dest, size, log=print):
    """Kopyayı arka plandaki tek işçiye sıraya koyar; üretim beklemez."""
    global _save_worker
    if _save_worker is None or not _save_worker.is_alive():
        _save_worker = threading.Thread(target=_save_loop, daemon=True, name="h3app-model-save")
        _save_worker.start()
    _save_pending.append(dest)
    _save_queue.put((src, dest, size, log))


def wait_model_saves(timeout=None):
    """Sıradaki Drive kopyalarının bitmesini bekler (testler ve kapanış için). Bittiyse True."""
    t0 = time.time()
    while _save_queue.unfinished_tasks:
        if timeout is not None and time.time() - t0 > timeout:
            return False
        time.sleep(0.05)
    return True


def _download_hf(rel, dst, size, sha, log):
    """Sabit Hugging Face revizyonundan indirir, boyut + SHA-256 doğrular, yerine koyar. Bozuksa siler ve hata verir."""
    from huggingface_hub import hf_hub_download
    name = os.path.basename(rel)
    log(f"İndiriliyor: {name} ({size / 1e9:.1f} GB)…")
    p = hf_hub_download(config.HF_REPO, rel, revision=config.HF_REVISION,
                        local_dir=os.path.join(config.WORK_DIR, "h3app_hf"))
    if os.path.getsize(p) != size or _sha256(p) != sha:
        os.remove(p)
        raise RuntimeError(f"Bozuk indirme: {rel}")
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.replace(p, dst)
    log(f"Doğrulandı: {name}")


def ensure_models(roles, log=print, source_dirs=None, save_dir=None):
    """Eksik modelleri önce sabit Hugging Face revizyonundan indirir; HF'e ulaşılamazsa ya da indirme bozuksa
    `source_dirs` (varsayılan config.MODEL_SOURCE_DIRS, ör. Drive) içindeki kopyalardan alır. Her dosya boyut +
    SHA-256 ile doğrulanır.
    `save_dir` (varsayılan config.MODEL_SAVE_DIR) verilmişse Drive'da bulunmayan her model, doğrulandıktan sonra
    arka planda oraya da kopyalanır (`save_dir/<alt klasör>/<ad>`), böylece sonraki oturum Drive'dan alır."""
    source_dirs = config.MODEL_SOURCE_DIRS if source_dirs is None else source_dirs
    save_dir = config.MODEL_SAVE_DIR if save_dir is None else save_dir
    idx = None
    for role in roles:
        rel, sub, size, sha = config.MODELS[role]
        name = os.path.basename(rel)
        dst = os.path.join(config.COMFY_DIR, "models", sub, name)
        if idx is None and save_dir:   # Drive'da zaten var mı? (varsa tekrar kaydedilmez)
            idx = _index_dirs(list(source_dirs or []) + [save_dir])
        on_drive = idx is not None and any(os.path.getsize(p) == size for p in idx.get(name, []))
        if not (os.path.isfile(dst) and os.path.getsize(dst) == size):
            # Önce Hugging Face: 17.11.2026 ölçümü HF kurulum + indirme ~6 dk, 42 GB'ın Drive kopyası ~12 dk.
            # Drive yalnızca HF'e ulaşılamazsa ya da indirme bozuk çıkarsa yedek.
            try:
                _download_hf(rel, dst, size, sha, log)
            except Exception as e:
                src = next((p for p in (idx or {}).get(name, []) if os.path.getsize(p) == size), None)
                if src is None and idx is None and source_dirs:
                    idx = _index_dirs(list(source_dirs) + ([save_dir] if save_dir else []))
                    src = next((p for p in idx.get(name, []) if os.path.getsize(p) == size), None)
                if not src:
                    raise
                log(f"Hugging Face'ten alınamadı ({str(e)[:120]}); Drive'dan kopyalanıyor: {name} "
                    f"({size / 1e9:.1f} GB)…")
                if not _copy_verified(src, dst, size, sha, log):
                    raise
                log(f"Doğrulandı: {name}")
                continue   # Drive'dan geldi: zaten orada, yeniden kaydedilmez
        if save_dir and not on_drive:
            log(f"💾 {name} arka planda Drive'a da kaydedilecek ({size / 1e9:.1f} GB).")
            queue_model_save(dst, os.path.join(save_dir, sub, name), size, log)


def roles_for(methods, te_role, ref=False):
    roles = ["unet_ref" if ref else "unet", te_role, "vae_video", "vae_audio"]
    if "turbo" in methods:
        roles.append("turbo_lora_ref" if ref else "turbo_lora")
    if "pika" in methods:
        roles.append("pika_lora_ref" if ref else "pika_lora")
    return roles

In [ ]:
%%writefile /content/h3_video_app/h3app/comfy_client.py
"""Arka planda çalışan ComfyUI sunucusu ve iş kuyruğu."""

import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from . import config

API = f"http://127.0.0.1:{config.COMFY_PORT}"
_server = None


def _get(path, timeout=10):
    return json.load(urllib.request.urlopen(API + path, timeout=timeout))


def _post(path, body):
    req = urllib.request.Request(API + path, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        return json.load(urllib.request.urlopen(req, timeout=60))
    except urllib.error.HTTPError as e:
        raise RuntimeError("ComfyUI isteği reddetti:\n" + e.read().decode()[:3000]) from None


def running():
    try:
        _get("/system_stats", timeout=2)
        return True
    except Exception:
        return False


def start(log=print):
    global _server
    if running():
        return
    log("ComfyUI sunucusu başlatılıyor…")
    out = open(os.path.join(config.WORK_DIR, "h3app_comfyui.log"), "w")
    _server = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(config.COMFY_PORT)],
                               cwd=config.COMFY_DIR, stdout=out, stderr=subprocess.STDOUT)
    for _ in range(300):
        if running():
            log("ComfyUI sunucusu hazır.")
            return
        if _server.poll() is not None:
            break
        time.sleep(2)
    tail = open(os.path.join(config.WORK_DIR, "h3app_comfyui.log")).read()[-3000:]
    raise RuntimeError("ComfyUI başlamadı:\n" + tail)


def run(graph, dest, poll=5):
    """Grafiği kuyruğa koyar, bitene kadar her `poll` saniyede geçen süreyi verir (generator),
    sonunda videoyu `dest`e kopyalar ve yolunu döndürür (StopIteration.value)."""
    pid = _post("/prompt", {"prompt": graph})["prompt_id"]
    t0 = time.time()
    while True:
        h = _get(f"/history/{pid}")
        if pid in h and h[pid].get("status", {}).get("completed") is not None:
            break
        yield int(time.time() - t0)
        time.sleep(poll)
    st = h[pid]["status"]
    if st.get("status_str") != "success":
        msgs = [m for m in st.get("messages", []) if m and m[0] == "execution_error"]
        raise RuntimeError("Üretim başarısız: " + json.dumps(msgs[-1][1] if msgs else st)[:2000])
    files = [v for o in h[pid]["outputs"].values() for k in ("images", "videos", "gifs") for v in o.get(k, [])]
    src = os.path.join(config.COMFY_DIR, "output", files[0]["subfolder"], files[0]["filename"])
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copyfile(src, dest)
    return dest

In [ ]:
%%writefile /content/h3_video_app/h3app/qa.py
"""Üretilen videonun GPU'suz kontrolü (QA v2): kesmeler (prompt'taki zamanlarla karşılaştırılır), donmuş aralıklar,
sonda kayma/bulanıklaşma tespiti (`tail_drift`), stil tutarlılığı uyarısı (`style_check`), ses kanalı ve konuşma bandı
analizi (`audio_check`), temas sayfası ve %100 kırpıntılar. Ölçü fikirleri scenario-alter skill'inin `alter_check.py`'sinden
(MIT) uyarlandı; orada kaynak video ile karşılaştırılıyordu, burada kaynak yok, beklenen kesmeler prompt'tan okunur."""

import os
import re
import subprocess

import cv2
import numpy as np

from . import config

SMALL = (160, 90)
CUT_HIST = 0.35       # ardışık iki karenin HS histogram uzaklığı (Bhattacharyya) bunu aşarsa kesme
CUT_JUMP = 6.0        # ya da gri fark, videonun ortanca farkının bu katını aşarsa (ışığı aynı kalan kesmeler)
CUT_MIN_DIFF = 12.0   # ... ve en az bu kadar (0-255) olursa
MIN_GAP = 6           # iki kesme arasında en az bu kadar kare
FROZEN_DIFF = 0.4     # ortalama gri fark bunun altındaysa kare "kıpırdamıyor"
FROZEN_MIN = 0.5      # en az bu kadar saniye kıpırdamazsa donmuş aralık
TOLERANCE = 0.35      # prompt'taki kesme zamanıyla eşleşme payı (sn)

TAIL_FRAMES = 12              # sonda kayma/bulanıklaşma kontrol penceresi (24 fps'te ~0,5 sn)
TAIL_MIN_TRIM = 6             # sonda bozulma varsa kırpılacak en az kare sayısı
TAIL_MAX_TRIM = 10            # sonda bozulma varsa kırpılacak en fazla kare sayısı
TAIL_SHARP_DROP = 0.60        # son karelerin keskinliği gövdenin bu oranının altına düşerse bulanıklaşma
TAIL_DRIFT_MULT = 2.6         # son karelerin ortalama hareketi gövde ortancasının bu katını aşarsa kayma
TAIL_DRIFT_MIN_DIFF = 6.5     # ... ve ortalama gri fark en az bu kadarsa

STYLE_MICRO_MAX = 0.55        # çizgi/boyalı stillerde yüksek frekanslı mikro-doku üst eşiği
STYLE_FLAT_MIN = 0.12         # çizgi/boyalı stillerde düz/boyalı bölge alt eşiği
SPEECH_BAND_LOW = 300.0       # konuşma bandı alt frekansı (Hz)
SPEECH_BAND_HIGH = 3400.0     # konuşma bandı üst frekansı (Hz)
SPEECH_RATIO_MIN = 0.35       # Türkçe konuşma beklenen videoda konuşma bandı enerjisi alt eşiği

# Kesme yalnızca "[Shot N] At 00:SS.000" biçimidir; "At about 00:05" gibi çekim içi olaylar kesme değildir
AT_RE = re.compile(r"\[Shot\s*\d+\]\s*At (\d{1,2}):(\d{2}(?:\.\d+)?)")
SPEECH_PROMPT_RE = re.compile(r"<d>|\[Turkish\]|\b(?:speaking|speaks|says|dialogue|bargaining)\b", re.I)


def prompt_cuts(prompt):
    """Prompt'taki "[Shot N] At 00:SS.000" kesme zamanları (0'dan büyük olanlar, sıralı)."""
    times = {round(int(m) * 60 + float(s), 3) for m, s in AT_RE.findall(prompt or "")}
    return sorted(t for t in times if t > 0)


def _scan(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    hists, grays = [], []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        small = cv2.resize(frame, SMALL, interpolation=cv2.INTER_AREA)
        h = cv2.calcHist([cv2.cvtColor(small, cv2.COLOR_BGR2HSV)], [0, 1], None, [32, 32], [0, 180, 0, 256])
        hists.append(cv2.normalize(h, h))
        grays.append(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32))
    cap.release()
    if not grays:
        raise ValueError(f"Video okunamadı: {path}")
    return fps, size, hists, grays


def find_cuts(hists, grays):
    """Kesme olan karelerin indeksleri (ilk kare hariç)."""
    hist_d = [cv2.compareHist(a, b, cv2.HISTCMP_BHATTACHARYYA) for a, b in zip(hists, hists[1:])]
    gray_d = [float(np.mean(np.abs(a - b))) for a, b in zip(grays, grays[1:])]
    base = max(float(np.median(gray_d)) if gray_d else 0.0, 1.0)
    cuts = []
    for i, (h, g) in enumerate(zip(hist_d, gray_d), start=1):
        if (h > CUT_HIST or (g > CUT_JUMP * base and g > CUT_MIN_DIFF)) and i - (cuts[-1] if cuts else 0) >= MIN_GAP:
            cuts.append(i)
    return cuts


def frozen_spans(grays, fps):
    """[(başlangıç sn, bitiş sn)]: görüntünün hiç kıpırdamadığı aralıklar."""
    spans, start = [], None
    for i in range(1, len(grays) + 1):
        still = i < len(grays) and float(np.mean(np.abs(grays[i] - grays[i - 1]))) < FROZEN_DIFF
        if still and start is None:
            start = i - 1
        elif not still and start is not None:
            if (i - start) / fps >= FROZEN_MIN:
                spans.append((round(start / fps, 2), round((i - 1) / fps, 2)))
            start = None
    return spans


def match(found, expected, tol=TOLERANCE):
    """(eşleşen [(beklenen, bulunan)], bulunamayan beklenenler, beklenmeyen bulunanlar)."""
    left, pairs, missing = list(found), [], []
    for e in expected:
        near = min(left, key=lambda f: abs(f - e), default=None)
        if near is not None and abs(near - e) <= tol:
            pairs.append((e, near))
            left.remove(near)
        else:
            missing.append(e)
    return pairs, missing, left


def _frames_at(path, indices):
    cap = cv2.VideoCapture(path)
    out = []
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok:
            out.append((i, frame))
    cap.release()
    return out


def contact_sheet(path, fps, n_frames, dest, moments=8, width=480):
    """Videonun eşit aralıklı anlarından, zaman etiketli 2 satırlık temas sayfası."""
    idx = [int(n_frames * (k + 0.5) / moments) for k in range(moments)]
    tiles = []
    for i, frame in _frames_at(path, idx):
        tile = cv2.resize(frame, (width, int(width * frame.shape[0] / frame.shape[1])), interpolation=cv2.INTER_AREA)
        cv2.putText(tile, f"{i / fps:.1f} sn", (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
        cv2.putText(tile, f"{i / fps:.1f} sn", (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        tiles.append(tile)
    while len(tiles) % 4:
        tiles.append(np.zeros_like(tiles[0]))
    cv2.imwrite(dest, np.vstack([np.hstack(tiles[k:k + 4]) for k in range(0, len(tiles), 4)]),
                [cv2.IMWRITE_JPEG_QUALITY, 88])
    return dest


def detail_crops(path, n_frames, dest, moments=4, box=320):
    """Her an için en çok hareket eden ayrıntılı bölgenin %100 kırpıntısı (genellikle eller ve yüz): parmak,
    yazı, logo ve yüz kontrolü için. Hareket, 3 kare sonrasıyla farktan; ayrıntı, kenar yoğunluğundan."""
    idx = [min(int(n_frames * (k + 0.5) / moments), max(n_frames - 4, 0)) for k in range(moments)]
    shots = dict(_frames_at(path, idx + [i + 3 for i in idx]))
    crops = []
    for i in idx:
        frame, later = shots.get(i), shots.get(i + 3, shots.get(i))
        if frame is None:
            continue
        h, w = frame.shape[:2]
        b = min(box, h, w)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
        moving = cv2.GaussianBlur(np.abs(gray - cv2.cvtColor(later, cv2.COLOR_BGR2GRAY).astype(np.float32)), (0, 0), 8)
        energy = (moving + 1.0) * np.abs(cv2.Laplacian(gray, cv2.CV_32F))
        summed = cv2.boxFilter(energy, -1, (b, b), normalize=False)[b // 2:h - b // 2 + 1, b // 2:w - b // 2 + 1]
        y, x = np.unravel_index(int(np.argmax(summed)), summed.shape)
        crops.append(frame[y:y + b, x:x + b])
    cv2.imwrite(dest, np.hstack(crops), [cv2.IMWRITE_JPEG_QUALITY, 92])
    return dest


LUFS_RE = re.compile(r"I:\s*(-?\d+(?:\.\d+)?)\s*LUFS")


def measure_lufs(path):
    """Videodaki ses kanalının EBU R128 entegre ses yüksekliği (LUFS); ses kanalı yoksa None döner."""
    try:
        p = subprocess.run(["ffmpeg", "-hide_banner", "-nostats", "-i", path, "-af", "ebur128", "-f", "null", "-"],
                           capture_output=True, text=True, timeout=60)
    except Exception:
        return None
    matches = LUFS_RE.findall(p.stderr or "")
    return round(float(matches[-1]), 1) if matches else None


def _apply_gain(path, gain_db, tp):
    """Sese doğrusal kazanç uygular (görüntü kopyalanır); yükseltiyorsa tepeleri `tp` dBTP'de sınırlar."""
    af = f"volume={gain_db:.2f}dB"
    if gain_db > 0:
        af += f",alimiter=limit={10 ** (tp / 20):.4f}:level=disabled"
    tmp = f"{path}.loudnorm.mp4"
    try:
        subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", path, "-c:v", "copy",
                        "-af", af, "-ar", "48000", "-c:a", "aac", "-b:a", "192k", tmp], check=True, timeout=120)
        os.replace(tmp, path)
        return True
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        return False


def normalize_audio(path, target_lufs=config.TARGET_LUFS, tp=-1.5, lra=11.0, tolerance=0.5):
    """Görüntü akışına dokunmadan (-c:v copy) sesi hedef entegre ses yüksekliğine (-16 LUFS) getirir.
    Ölçülen değere göre doğrusal kazanç uygulanır ve sonuç yeniden ölçülür; gerekirse bir kez daha ince ayar yapılır.
    (Tek geçişli dinamik `loudnorm` kısa, araları sessiz H3 konuşmalarında hedefi aşıyordu: -16,7 → -13,7.)
    Ses kanalı varsa ve sessiz değilse (önceki_lufs, yeni_lufs) döner; yoksa None. `lra` geriye uyumluluk için."""
    before = measure_lufs(path)
    if before is None or before <= -65.0:
        return None
    now = before
    for _ in range(2):
        gain = float(target_lufs) - now
        if abs(gain) <= tolerance or not _apply_gain(path, gain, tp):
            break
        measured = measure_lufs(path)
        if measured is None:
            break
        now = measured
    return (before, now)


def _ensure_grays(source):
    """`source` bir dosya yoluysa `_scan` ile gri kareleri ve fps'i döner; gri kare listesiyse doğrudan kullanır."""
    if isinstance(source, str):
        fps, _, _, grays = _scan(source)
        return grays, fps
    return list(source or []), 24.0


def tail_drift(source, fps=24.0, expected_cuts=None):
    """Videonun son 12 karesindeki (~0,5 sn) Laplacian keskinlik düşüşünü ve beklenmeyen global kaymayı ölçer.

    Bozulma varsa `assemble.py` içinde son 6–10 karenin otomatik kırpılabilmesi için `kirp_kare` (6..10) döner.
    Dönüş: `{bozuk, keskinlik_orani, kayma_orani, kirp_kare, neden}`.
    """
    grays, scanned_fps = _ensure_grays(source)
    fps = float(fps or scanned_fps or 24.0)
    n = len(grays)
    tail_len = min(TAIL_FRAMES, max(TAIL_MIN_TRIM, n // 4)) if n >= 20 else 0
    if tail_len < TAIL_MIN_TRIM or n < tail_len + 8:
        return {"bozuk": False, "keskinlik_orani": 1.0, "kayma_orani": 1.0, "kirp_kare": 0, "neden": None}

    # 1. Laplacian keskinlik düşüşü
    sharp = [float(np.mean(np.abs(cv2.Laplacian(g, cv2.CV_32F)))) for g in grays]
    body_start = max(0, n - tail_len - 36)
    body_sharp = sharp[body_start:n - tail_len]
    base_sharp = float(np.median(body_sharp)) if body_sharp else 0.0
    tail_sharp = sharp[n - tail_len:]
    if base_sharp > 1.0:
        sharp_ratio = round(float(np.mean(tail_sharp[-TAIL_MIN_TRIM:])) / base_sharp, 3)
        blur_bad = sharp_ratio < TAIL_SHARP_DROP
    else:
        sharp_ratio = 1.0
        blur_bad = False

    # 2. Beklenmeyen global kayma / sıçrama (prompt'ta son 0,5 sn içinde kasıtlı kesme yoksa)
    tail_start_sec = (n - tail_len) / fps
    has_expected_tail_cut = any(c >= tail_start_sec - TOLERANCE for c in (expected_cuts or []))
    diffs = [float(np.mean(np.abs(a - b))) for a, b in zip(grays, grays[1:])]
    body_diffs = [d for d in diffs[max(0, len(diffs) - tail_len - 36):len(diffs) - tail_len] if d < 25.0]
    base_diff = max(float(np.median(body_diffs)) if body_diffs else 0.0, 0.5)
    tail_diffs = diffs[len(diffs) - tail_len:]
    tail_mean = float(np.mean(tail_diffs[-TAIL_MIN_TRIM:])) if tail_diffs else 0.0
    tail_max = float(np.max(tail_diffs)) if tail_diffs else 0.0

    if not has_expected_tail_cut and tail_mean > 0:
        drift_ratio = round(tail_mean / base_diff, 3)
        drift_bad = tail_mean >= TAIL_DRIFT_MIN_DIFF and tail_mean >= TAIL_DRIFT_MULT * base_diff
    else:
        drift_ratio = round(tail_mean / base_diff, 3) if base_diff > 0 else 1.0
        drift_bad = False

    if not (blur_bad or drift_bad):
        return {
            "bozuk": False,
            "keskinlik_orani": sharp_ratio,
            "kayma_orani": drift_ratio,
            "kirp_kare": 0,
            "neden": None,
        }

    # Bozulmanın başladığı ilk kareyi bul ve kırpılacak kare sayısını [6, 10] aralığına sabitle
    bad_idx = tail_len - TAIL_MIN_TRIM
    for idx in range(tail_len):
        is_blur = blur_bad and tail_sharp[idx] < base_sharp * (TAIL_SHARP_DROP + 0.08)
        is_drift = drift_bad and idx < len(tail_diffs) and tail_diffs[idx] >= max(TAIL_DRIFT_MIN_DIFF * 0.85, base_diff * 2.2)
        if is_blur or is_drift:
            bad_idx = idx
            break
    trim = max(TAIL_MIN_TRIM, min(TAIL_MAX_TRIM, tail_len - bad_idx))
    if blur_bad and drift_bad:
        reason = "bulanıklaşma ve kayma"
    elif blur_bad:
        reason = "bulanıklaşma"
    else:
        reason = "kayma"
    return {
        "bozuk": True,
        "keskinlik_orani": sharp_ratio,
        "kayma_orani": drift_ratio,
        "kirp_kare": int(trim),
        "neden": reason,
    }


def _infer_style(style, prompt=""):
    """Stil açıkça verilmediyse prompt içindeki stil anahtar kelimelerinden tahmin eder."""
    if style and style in config.STYLES:
        return style
    low = (prompt or "").lower()
    for k, meta in config.STYLES.items():
        if k != config.DEFAULT_STYLE and any(kw in low for kw in meta.get("keywords", ())):
            return k
    return config.DEFAULT_STYLE


def _style_metrics(source):
    """Örneklenen karelerde düz/boyalı bölge oranını (`duz_oran`) ve yüksek frekanslı mikro-doku oranını (`mikro_doku`) ölçer."""
    grays, _ = _ensure_grays(source)
    if not grays:
        return 0.0, 0.0
    step = max(1, len(grays) // 8)
    sampled = grays[::step][:8]
    flats, micros = [], []
    for g in sampled:
        gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
        grad = np.hypot(gx, gy)
        blur = cv2.GaussianBlur(g, (5, 5), 1.2)
        micro = np.abs(g - blur)
        flats.append(float(np.mean(grad < 12.0)))
        micros.append(float(np.mean(micro > 3.5)))
    return round(float(np.mean(flats)), 3), round(float(np.mean(micros)), 3)


def style_check(source, style=None, prompt="", ref_source=None):
    """Anime, Pixar veya Suluboya seçildiğinde arka planın/dokunun fotoğrafa kayıp kaymadığını kontrol eder.

    `ref_source` (örn. Pika taslak videosu veya onun `style_check` sözlüğü) verilirse taslaktan H3 finaline
    geçerken oluşan stil kaymasını da denetler.
    Dönüş: `{stil, duz_oran, mikro_doku, kayma, taslaktan_kayma, uyari}`.
    """
    resolved = _infer_style(style, prompt)
    flat_ratio, micro_ratio = _style_metrics(source)
    if resolved == config.DEFAULT_STYLE:
        return {
            "stil": resolved,
            "duz_oran": flat_ratio,
            "mikro_doku": micro_ratio,
            "kayma": False,
            "taslaktan_kayma": False,
            "uyari": None,
        }

    abs_drift = (micro_ratio > STYLE_MICRO_MAX and flat_ratio < STYLE_FLAT_MIN) or (
        micro_ratio > 0.45 and flat_ratio > 0 and (micro_ratio / max(flat_ratio, 0.01)) > 5.0
    )
    rel_drift = False
    if ref_source is not None:
        if isinstance(ref_source, dict) and "duz_oran" in ref_source and "mikro_doku" in ref_source:
            ref_flat, ref_micro = float(ref_source["duz_oran"]), float(ref_source["mikro_doku"])
        else:
            ref_flat, ref_micro = _style_metrics(ref_source)
        if (micro_ratio - ref_micro >= 0.06 and flat_ratio <= ref_flat * 0.85) or (flat_ratio - ref_flat <= -0.08):
            rel_drift = True

    kayma = bool(abs_drift or rel_drift)
    label = config.STYLES.get(resolved, {}).get("label", resolved)
    uyari = None
    if abs_drift and rel_drift:
        uyari = f"{label} stilinde fotoğrafa ve taslağa göre mikro-doku kayması (düz: {flat_ratio:.2f}, doku: {micro_ratio:.2f})"
    elif abs_drift:
        uyari = f"{label} stilinde fotoğrafa kayma / aşırı mikro-doku (düz: {flat_ratio:.2f}, doku: {micro_ratio:.2f})"
    elif rel_drift:
        uyari = f"taslaktan finale geçerken {label} stilinde kayma (düz: {flat_ratio:.2f}, doku: {micro_ratio:.2f})"
    return {
        "stil": resolved,
        "duz_oran": flat_ratio,
        "mikro_doku": micro_ratio,
        "kayma": kayma,
        "taslaktan_kayma": rel_drift,
        "uyari": uyari,
    }


def audio_check(path, prompt="", lufs=None):
    """Ses kanalının dolu olup olmadığını, LUFS değerini ve konuşma bandı (300–3400 Hz) enerjisini ölçer.

    Dönüş: `{var, dolu, lufs, rms, konusma_orani, konusma_bekleniyor, uyari}`.
    """
    low_prompt = (prompt or "").lower()
    konusma_bekleniyor = bool(SPEECH_PROMPT_RE.search(prompt or "")) and "no speech" not in low_prompt
    if lufs is None:
        lufs = measure_lufs(path)
    try:
        p = subprocess.run(
            ["ffmpeg", "-v", "error", "-i", path, "-vn", "-ac", "1", "-ar", "16000", "-f", "f32le", "-"],
            capture_output=True, timeout=30,
        )
        pcm = np.frombuffer(p.stdout or b"", dtype=np.float32)
    except Exception:
        pcm = np.array([], dtype=np.float32)

    if pcm.size == 0:
        return {
            "var": False,
            "dolu": False,
            "lufs": lufs,
            "rms": 0.0,
            "konusma_orani": 0.0,
            "konusma_bekleniyor": konusma_bekleniyor,
            "uyari": "ses kanalı yok veya boş" if konusma_bekleniyor else None,
        }

    rms = round(float(np.sqrt(np.mean(pcm ** 2))), 4)
    dolu = bool(rms > 1e-4 and (lufs is None or lufs > -55.0))
    if dolu and pcm.size >= 64:
        spec = np.abs(np.fft.rfft(pcm)) ** 2
        freqs = np.fft.rfftfreq(pcm.size, 1.0 / 16000.0)
        tot = float(np.sum(spec[(freqs >= 50.0) & (freqs <= 8000.0)]))
        sp = float(np.sum(spec[(freqs >= SPEECH_BAND_LOW) & (freqs <= SPEECH_BAND_HIGH)]))
        konusma_orani = round(sp / tot, 2) if tot > 1e-12 else 0.0
    else:
        konusma_orani = 0.0

    uyari = None
    if not dolu:
        uyari = "ses kanalı sessiz/boş"
    elif konusma_bekleniyor and konusma_orani < SPEECH_RATIO_MIN:
        uyari = f"konuşma bandı enerjisi düşük ({konusma_orani:.2f})"

    return {
        "var": True,
        "dolu": dolu,
        "lufs": lufs,
        "rms": rms,
        "konusma_orani": konusma_orani,
        "konusma_bekleniyor": konusma_bekleniyor,
        "uyari": uyari,
    }


def check(path, prompt="", sheet=None, crops=None, style=None, ref_video=None):
    """Videoyu ölçer (QA v2). `{saniye, kare, fps, boyut, kesmeler, beklenen, eslesen, bulunamayan, beklenmeyen, donmus,
    lufs, ses, son_kayma, stil_kontrol, temas, kirpinti}` döner; özet için `summary()`."""
    fps, size, hists, grays = _scan(path)
    n = len(grays)
    cuts = [round(i / fps, 2) for i in find_cuts(hists, grays)]
    expected = prompt_cuts(prompt)
    pairs, missing, extra = match(cuts, expected)
    lufs = measure_lufs(path)
    ses = audio_check(path, prompt=prompt, lufs=lufs)
    son_kayma = tail_drift(grays, fps=fps, expected_cuts=expected)
    stil_kontrol = style_check(grays, style=style, prompt=prompt, ref_source=ref_video)
    report = {"saniye": round(n / fps, 2), "kare": n, "fps": round(fps, 2), "boyut": list(size),
              "kesmeler": cuts, "beklenen": expected, "eslesen": pairs, "bulunamayan": missing,
              "beklenmeyen": extra, "donmus": frozen_spans(grays, fps),
              "lufs": lufs, "ses": ses, "son_kayma": son_kayma, "stil_kontrol": stil_kontrol,
              "prompt_var": bool((prompt or "").strip())}
    if sheet:
        report["temas"] = contact_sheet(path, fps, n, sheet)
    if crops:
        report["kirpinti"] = detail_crops(path, n, crops)
    return report


def summary(r):
    """Durum kutusu için tek satırlık Türkçe özet."""
    sec = lambda xs: " · ".join(f"{x:.1f}".replace(".", ",") for x in xs) or "yok"
    parts = [f"{r['saniye']:.2f} sn, {r['kare']} kare".replace(".", ","), f"kesmeler: {sec(r['kesmeler'])}"]
    ses = r.get("ses") or {}
    if r.get("lufs") is not None:
        s_txt = f"ses: {r['lufs']:.1f} LUFS".replace(".", ",")
        if ses.get("dolu") and (ses.get("konusma_bekleniyor") or ses.get("konusma_orani", 0) > 0):
            s_txt += f" (konuşma bandı: {ses['konusma_orani']:.2f})".replace(".", ",")
        parts.append(s_txt)
    if ses.get("uyari"):
        parts.append(f"⚠️ ses: {ses['uyari']}".replace(".", ","))
    if r["beklenen"]:
        parts.append(f"prompt'ta: {sec(r['beklenen'])}")
        if r["bulunamayan"]:
            parts.append(f"bulunamayan: {sec(r['bulunamayan'])} (ışık aynıysa ölçülemeyebilir, gözle bakın)")
    if r["beklenmeyen"] and r.get("prompt_var"):
        parts.append(f"⚠️ beklenmeyen kesme/sıçrama: {sec(r['beklenmeyen'])}")
    if r["donmus"]:
        parts.append("⚠️ donmuş: " + ", ".join(f"{a:.1f}–{b:.1f} sn".replace(".", ",") for a, b in r["donmus"]))
    td = r.get("son_kayma") or {}
    if td.get("bozuk"):
        parts.append(f"⚠️ sonda {td.get('neden', 'bozulma')} (son {td.get('kirp_kare', 6)} kare kırpılabilir)")
    sc = r.get("stil_kontrol") or {}
    if sc.get("kayma") and sc.get("uyari"):
        parts.append(f"⚠️ stil kayması: {sc['uyari']}".replace(".", ","))
    return "Kontrol: " + "; ".join(parts)

In [ ]:
%%writefile /content/h3_video_app/h3app/review.py
"""Üretilen videonun Gemini ile otomatik izlenmesi: video sesiyle birlikte bir kontrol listesine göre izlenir ve
zaman damgalı kusur listesi döner (fazladan el/kişi, nesne ışınlanması, bakış açısı sıçraması, kimlik, yazı, stil,
replikler ve dudak uyumu, ilk/son 2 saniye).

16.11.2026 kör testi (ipucu verilmeden, gerçek bir Pika videosu): gemini-2.5-flash 0–2 sn'deki fazladan eli doğru
zamanla buldu, gemini-3-flash-preview de buldu, gemini-3.5-flash-lite "kusursuz" dedi. Bu yüzden izleme kendi
model sırasını kullanır ve lite modeller burada kullanılmaz. Gemini'ye ulaşılamazsa None döner; üretim engellenmez.
"""

import os
import subprocess
import tempfile

from . import config, prompt_writer

CHECKLIST = """You are a strict video QA reviewer for AI-generated video clips. Watch the whole clip frame by frame and
LISTEN to the audio. Compare it with the generation prompt and the reference pictures. Report every defect you can see
or hear, with timestamps. Check at least:
1. Anatomy and physical logic: extra or missing hands/arms/fingers, disembodied body parts, objects floating or
   teleporting between hands, objects changing size, people or objects appearing/disappearing without a cause.
2. Camera logic: unexplained point-of-view shots, sudden jumps or cuts the prompt did not ask for.
3. Identity: does each character match the reference picture / description for the whole clip?
4. Scene and style consistency over time; text or lettering that should not be there.
5. Speech: are the quoted lines spoken, in the right language, by the right character, with lips in sync?
6. The first 2 seconds and the last 2 seconds specifically (models often glitch there).
7. Wardrobe and props against the prompt: extra or different garments (e.g. a vest the prompt does not mention),
   wrong colours, objects that are already in place before someone brings them, props that change or vanish.
8. Camera: does the movement match the prompt (e.g. a slow steady push-in, not a zoom that tightens and widens)?
Severity: "high" = a viewer would clearly notice and the clip should be regenerated; "medium" = noticeable but
usable; "low" = minor. Do not invent defects: report only what you actually see or hear.
Answer ONLY with JSON: {"verdict": "pass" | "fail", "defects": [{"start_s": number, "end_s": number,
"severity": "high" | "medium" | "low", "category": string, "description": string}], "summary": string}"""

SEVERITIES = ("high", "medium", "low")
SEVERITY_TR = {"high": "yüksek", "medium": "orta", "low": "düşük"}
TIMEOUT_MS = 240_000


def _proxy(path, dest_dir):
    """İzleme için küçük kopya (540p, düşük bit hızı): istek hızlı ve satır içi gönderilebilir boyutta kalır."""
    out = os.path.join(dest_dir, "inceleme.mp4")
    try:
        subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", path, "-vf", "scale=-2:540",
                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "30", "-c:a", "aac", "-b:a", "96k", out],
                       check=True, timeout=180)
        return out
    except Exception:
        return path


def normalize(raw):
    """Model çıktısını tutarlı bir sözlüğe çevirir; karar, yüksek önemde kusur olup olmamasından türetilir."""
    defects = []
    for d in (raw or {}).get("defects") or []:
        if not isinstance(d, dict):
            continue
        sev = str(d.get("severity", "medium")).lower()
        try:
            start = float(d.get("start_s", 0) or 0)
            end = float(d.get("end_s", start) or start)
        except (TypeError, ValueError):
            start = end = 0.0
        defects.append({"start_s": round(start, 2), "end_s": round(max(start, end), 2),
                        "severity": sev if sev in SEVERITIES else "medium",
                        "category": str(d.get("category") or "").strip(),
                        "description": str(d.get("description") or "").strip()})
    defects.sort(key=lambda d: (SEVERITIES.index(d["severity"]), d["start_s"]))
    return {"verdict": "fail" if any(d["severity"] == "high" for d in defects) else "pass",
            "defects": defects, "summary": str((raw or {}).get("summary") or "").strip()}


def review_video(path, prompt, api_key, images=(), roles=None, models=None, log=None):
    """Videoyu izletir. {'verdict', 'defects', 'summary', 'model'} ya da (hiçbir model yanıt vermezse) None döner."""
    from google.genai import types
    models = prompt_writer.available(models or config.REVIEW_MODELS)
    with tempfile.TemporaryDirectory(prefix="h3app_inceleme_") as tmp:
        data = open(_proxy(path, tmp), "rb").read()
    parts = [types.Part.from_bytes(data=data, mime_type="video/mp4")]
    if images:
        parts += ["Reference pictures:"] + prompt_writer._image_parts(images, roles)
    parts.append("Generation prompt:\n" + (prompt or "").strip())
    cfg = types.GenerateContentConfig(system_instruction=CHECKLIST, temperature=0.1,
                                      response_mime_type="application/json")
    for m in models:
        try:
            client = prompt_writer._client(api_key, TIMEOUT_MS)   # değişkende tutulmalı (bkz. prompt_writer)
            resp = client.models.generate_content(model=m, contents=parts, config=cfg)
            result = normalize(prompt_writer._parse_json(resp.text or ""))
            result["model"] = m
            return result
        except Exception as e:
            if prompt_writer.gone(e):
                prompt_writer.mark_gone(m, log)
                continue
            if log:
                log(f"   🔍 {m} videoyu izleyemedi ({str(e)[:120]}); sıradaki izleyici deneniyor.")
    return None


def blocking(result):
    """Yeniden üretim gerektiren (yüksek önemde) kusur var mı?"""
    return bool(result) and any(d["severity"] == "high" for d in result.get("defects") or [])


def score(result):
    """Adayları karşılaştırmak için: daha az yüksek, sonra orta, sonra düşük kusur daha iyidir. İzlenemeyen en sona."""
    if not result:
        return (99, 99, 99)
    counts = [sum(1 for d in result["defects"] if d["severity"] == s) for s in SEVERITIES]
    return tuple(counts)


def short(result):
    """Çok kısa özet: 'kusur yok', '1 yüksek, 2 düşük kusur' ya da 'izlenemedi'."""
    if result is None:
        return "izlenemedi"
    parts = [f"{n} {SEVERITY_TR[s]}" for s, n in zip(SEVERITIES, score(result)) if n]
    return (", ".join(parts) + " kusur") if parts else "kusur yok"


def summary(result):
    """Günlük için tek satır Türkçe özet."""
    if result is None:
        return "🔍 Otomatik izleme yapılamadı (Gemini yanıt vermedi); video yine de kaydedildi."
    model = result.get("model", "")
    if not result["defects"]:
        return f"🔍 İzleme ({model}): kusur bulunmadı."
    items = [f"[{d['start_s']:.1f}–{d['end_s']:.1f} sn, {SEVERITY_TR[d['severity']]}] {d['description']}"
             for d in result["defects"][:4]]
    more = f" (+{len(result['defects']) - 4} kusur daha)" if len(result["defects"]) > 4 else ""
    return f"🔍 İzleme ({model}): {len(result['defects'])} kusur — " + " · ".join(items) + more


# ---------------- uzun video: süreklilik izlemesi ----------------

SURECLILIK = """You are a film continuity supervisor. The clip was generated in {n} separate blocks and joined at
these times (seconds): {cuts}. Watch the whole clip and LISTEN to it. Judge ONLY whether things that should stay the
same really stay the same across the block boundaries and through the clip, compared with the story bible below.
Score each from 1 (badly broken) to 10 (perfectly consistent):
- kimlik: each character is clearly the same person (face, age, hair, build)
- kiyafet: outfits, colours and accessories stay the same
- mekan: the place, its layout, props and light stay the same
- ses: each speaker keeps the same voice (timbre, pitch, accent); lines are spoken by the right person
- hikaye: the events and lines follow the story bible in order, and the joins feel like intended cuts
List concrete problems with timestamps. Do not invent problems. Answer ONLY with JSON:
{{"kimlik": int, "kiyafet": int, "mekan": int, "ses": int, "hikaye": int,
  "sorunlar": [{{"zaman_s": number, "alan": string, "aciklama": string}}], "ozet": string}}

Story bible:
{bible}"""
SURECLILIK_PUANLARI = ("kimlik", "kiyafet", "mekan", "ses", "hikaye")


def continuity_review(path, cuts, bible, api_key, models=None, log=None):
    """Birleşik uzun videoyu blok sınırlarındaki tutarlılık açısından 1–10 puanlar (+ 'ortalama', 'model').
    Görsel referans verilmez: metin ve görsel kolları adil karşılaştırılsın diye yalnızca iç tutarlılık ölçülür.
    Hiçbir model yanıt vermezse None."""
    from google.genai import types
    import json
    system = SURECLILIK.format(n=len(cuts) + 1, cuts=", ".join(f"{c:.2f}" for c in cuts) or "none",
                               bible=json.dumps(bible, ensure_ascii=False, indent=1))
    with tempfile.TemporaryDirectory(prefix="h3app_sureklilik_") as tmp:
        data = open(_proxy(path, tmp), "rb").read()
    parts = [types.Part.from_bytes(data=data, mime_type="video/mp4"), "Judge the continuity of this clip."]
    cfg = types.GenerateContentConfig(system_instruction=system, temperature=0.1,
                                      response_mime_type="application/json")
    for m in prompt_writer.available(models or config.REVIEW_MODELS):
        try:
            client = prompt_writer._client(api_key, TIMEOUT_MS)
            res = prompt_writer._parse_json(client.models.generate_content(model=m, contents=parts, config=cfg).text or "")
            for k in SURECLILIK_PUANLARI:
                res[k] = max(1, min(10, int(res[k]))) if res.get(k) is not None else None
            scores = [res[k] for k in SURECLILIK_PUANLARI if res.get(k)]
            res["ortalama"] = round(sum(scores) / len(scores), 2) if scores else None
            res.setdefault("sorunlar", [])
            res["model"] = m
            return res
        except Exception as e:  # noqa: BLE001
            if prompt_writer.gone(e):
                prompt_writer.mark_gone(m, log)
                continue
            if log:
                log(f"   🎞️ {m} sürekliliği izleyemedi ({str(e)[:120]})")
    return None

In [ ]:
%%writefile /content/h3_video_app/h3app/assemble.py
"""Çok çekimli uzun video bloklarını (15–60 sn) ffmpeg ile kare-tam birleştirme ve ses miksajı.

İki ses modu desteklenir:
1. "speech" (Türkçe konuşmalı sahne, varsayılan): Her blok kendi doğal sesiyle korunur, blok geçişlerinde
   pop/click sesini önlemek için kısa fade-out/fade-in uygulanır (senkron korunur) ve çıkış -16 LUFS seviyesine dengelenir.
2. "music" (Müzikli, konuşmasız kurgu): Kullanıcının yüklediği telifsiz müzik dosyası video süresine
   kırpılarak döşenir (isteğe bağlı olarak blokların ortam sesi -12 dB altta mikslenir) ve -16 LUFS'a dengelenir.
"""

import os
import shutil
import subprocess
import tempfile

import cv2

from . import config, qa


def has_audio(path):
    """Video dosyasında ses kanalı olup olmadığını döner."""
    try:
        p = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index", "-of", "csv=p=0", path],
            capture_output=True, text=True, timeout=15,
        )
        return bool((p.stdout or "").strip())
    except Exception:
        return False


def probe_video(path):
    """Videonun kare sayısı, fps, boyut, süre ve ses kanalı bilgisini döner."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise ValueError(f"Video açılamadı: {path}")
    fps = round(cap.get(cv2.CAP_PROP_FPS) or 24.0, 4)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = 0
    while True:
        ok, _ = cap.read()
        if not ok:
            break
        n += 1
    cap.release()
    if n == 0:
        raise ValueError(f"Videoda kare bulunamadı: {path}")
    return {
        "frames": n,
        "fps": fps,
        "width": w,
        "height": h,
        "duration": round(n / fps, 4),
        "has_audio": has_audio(path),
    }


def _prepare_clip(src, dest, target_w, target_h, target_fps, keep_frames, add_silent_audio=False):
    """Tek bir bloğu hedef çözünürlük/fps ve kesin kare sayısına (`keep_frames`) göre hazırlar."""
    info = probe_video(src)
    keep_frames = max(1, min(int(keep_frames), info["frames"]))
    dur = keep_frames / float(target_fps)
    vf = (f"fps={target_fps},scale={target_w}:{target_h}:force_original_aspect_ratio=decrease,"
          f"pad={target_w}:{target_h}:(ow-iw)/2:(oh-ih)/2,format=yuv420p,trim=end_frame={keep_frames},setpts=PTS-STARTPTS")
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", src]
    if info["has_audio"]:
        af = f"aresample=48000,aformat=sample_rates=48000:channel_layouts=stereo,atrim=0:{dur:.4f},asetpts=PTS-STARTPTS"
        cmd += ["-vf", vf, "-af", af, "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-c:a", "aac", "-b:a", "192k", dest]
    elif add_silent_audio:
        cmd += [
            "-f", "lavfi", "-t", f"{dur:.4f}", "-i", "anullsrc=channel_layout=stereo:sample_rate=48000",
            "-vf", vf, "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-c:a", "aac", "-b:a", "192k",
            "-frames:v", str(keep_frames), dest,
        ]
    else:
        cmd += ["-vf", vf, "-an", "-c:v", "libx264", "-preset", "fast", "-crf", "18", dest]
    subprocess.run(cmd, check=True, timeout=180)
    return keep_frames


def assemble(clips, dest, audio_mode=config.DEFAULT_AUDIO_MODE, music_path=None, keep_ambient=False,
             ambient_db=-12.0, crossfade_sec=0.12, overlap_frames=0, trim_tails=None,
             auto_trim_tail=True, target_lufs=config.TARGET_LUFS):
    """Blok videolarını kare-tam olarak birleştirir ve seçilen ses moduna göre miksleyip -16 LUFS'a dengeler.

    Parametreler:
      clips: Blok video dosya yolları listesi.
      dest: Çıkış video dosya yolu (.mp4).
      audio_mode: "speech" (doğal konuşmalı sahne) veya "music" (telifsiz dış müzikli kurgu).
      music_path: `audio_mode="music"` iken altına döşenecek müzik dosyasının yolu.
      keep_ambient: `audio_mode="music"` iken blokların kendi ortam sesini `ambient_db` seviyesinde koru.
      ambient_db: Ortam sesinin müzik altındaki seviyesi (dB, varsayılan -12.0).
      crossfade_sec: Blok geçişlerinde ses tıklamasını önleyen crossfade süresi (sn, varsayılan 0.12).
      overlap_frames: Kesintisiz uzatma modunda her sonraki bloktan (veya önceki bloğun sonundan) kırpılacak kare sayısı.
      trim_tails: Her bloğun sonundan kırpılacak kare sayıları listesi (None ise `auto_trim_tail` devreye girer).
      auto_trim_tail: True ise `qa.tail_drift()` ile son 12 karede bulanıklaşma/kayma bulunan blokların son 6–10 karesini otomatik kırpar.
      target_lufs: EBU R128 ses normalizasyonu hedefi (varsayılan -16.0 LUFS).
    """
    clips = [c for c in (clips or []) if c]
    if not clips:
        raise ValueError("Birleştirilecek en az bir video bloğu gerekli.")
    os.makedirs(os.path.dirname(os.path.abspath(dest)) or ".", exist_ok=True)

    infos = [probe_video(c) for c in clips]
    w, h, fps = infos[0]["width"], infos[0]["height"], round(infos[0]["fps"] or 24.0, 2)
    any_audio = any(inf["has_audio"] for inf in infos)
    use_music = audio_mode == "music" and bool(music_path and os.path.isfile(music_path))

    # Sonda bozulma varsa otomatik kırpılacak kare sayılarını belirle
    if trim_tails is None and auto_trim_tail:
        applied_trims = []
        for c in clips:
            try:
                td = qa.tail_drift(c, fps=fps)
                applied_trims.append(int(td["kirp_kare"]) if td.get("bozuk") else 0)
            except Exception:
                applied_trims.append(0)
    else:
        applied_trims = [int(trim_tails[i]) if trim_tails and i < len(trim_tails) and trim_tails[i] else 0
                         for i in range(len(clips))]

    # Her bloktan kaç kare korunacağını hesapla
    keep_counts = []
    for idx, inf in enumerate(infos):
        n = inf["frames"] - applied_trims[idx]
        if overlap_frames and idx < len(infos) - 1:
            n -= int(overlap_frames)
        keep_counts.append(max(1, n))

    total_frames = sum(keep_counts)
    total_dur = total_frames / float(fps)

    with tempfile.TemporaryDirectory(prefix="h3app_asm_") as tmpdir:
        need_clip_audio = any_audio and (audio_mode != "music" or keep_ambient or not use_music)
        prep_paths = []
        for idx, (src, kframes) in enumerate(zip(clips, keep_counts)):
            p = os.path.join(tmpdir, f"blk_{idx:02d}.mp4")
            _prepare_clip(src, p, w, h, fps, kframes, add_silent_audio=need_clip_audio)
            prep_paths.append(p)

        # Tek blok varsa ve müzik eklenmeyecekse doğrudan kopyala + normalize et
        if len(prep_paths) == 1 and not use_music:
            shutil.copyfile(prep_paths[0], dest)
            if any_audio and target_lufs is not None:
                qa.normalize_audio(dest, target_lufs=target_lufs)
            out_info = probe_video(dest)
            return {
                "path": dest, "kare": out_info["frames"], "saniye": round(out_info["frames"] / fps, 2),
                "fps": fps, "boyut": [w, h], "lufs": qa.measure_lufs(dest), "blok_sayisi": 1,
                "kirpilan_son_kareler": applied_trims,
            }

        # Görüntü akışlarını concat filtresiyle kare-tam birleştir
        n = len(prep_paths)
        inputs = []
        for p in prep_paths:
            inputs += ["-i", p]
        v_in = "".join(f"[{i}:v:0]" for i in range(n))
        filter_parts = [f"{v_in}concat=n={n}:v=1:a=0[vout]"]

        # Ses akışı kurgusu
        map_audio = False
        if need_clip_audio:
            if n == 1:
                filter_parts.append("[0:a:0]anull[amb]")
            elif crossfade_sec and crossfade_sec > 0:
                # Her blok sınırında kısa fade-out/fade-in (tıklama önleme). acrossfade sesi her geçişte kısaltıp
                # sonraki blokların sesini görüntünün önüne kaydırırdı (dudak senkronu); concat süreyi korur.
                labels = []
                for i in range(n):
                    dur_i = keep_counts[i] / fps
                    f = min(float(crossfade_sec) / 2.0, dur_i * 0.2)
                    chain = []
                    if i > 0:
                        chain.append(f"afade=t=in:st=0:d={f:.3f}")
                    if i < n - 1:
                        chain.append(f"afade=t=out:st={max(0.0, dur_i - f):.4f}:d={f:.3f}")
                    filter_parts.append(f"[{i}:a:0]{','.join(chain)}[af{i}]")
                    labels.append(f"[af{i}]")
                filter_parts.append(f"{''.join(labels)}concat=n={n}:v=0:a=1,apad,atrim=0:{total_dur:.4f},"
                                    "asetpts=PTS-STARTPTS[amb]")
            else:
                a_in = "".join(f"[{i}:a:0]" for i in range(n))
                filter_parts.append(f"{a_in}concat=n={n}:v=0:a=1[amb]")

        if use_music:
            mus_idx = n
            inputs += ["-stream_loop", "-1", "-i", music_path]
            fade_st = max(0.0, total_dur - 0.6)
            filter_parts.append(
                f"[{mus_idx}:a:0]aresample=48000,aformat=sample_rates=48000:channel_layouts=stereo,"
                f"atrim=0:{total_dur:.4f},asetpts=PTS-STARTPTS,afade=t=out:st={fade_st:.3f}:d=0.6[mus]"
            )
            if need_clip_audio and keep_ambient:
                filter_parts.append(
                    f"[amb]volume={float(ambient_db):.1f}dB[amb_low];"
                    f"[amb_low][mus]amix=inputs=2:duration=first:dropout_transition=0:normalize=0[aout]"
                )
            else:
                filter_parts.append("[mus]anull[aout]")
            map_audio = True
        elif need_clip_audio:
            filter_parts.append("[amb]anull[aout]")
            map_audio = True

        cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"] + inputs + [
            "-filter_complex", ";".join(filter_parts),
            "-map", "[vout]",
        ]
        if map_audio:
            cmd += ["-map", "[aout]", "-c:a", "aac", "-b:a", "192k"]
        else:
            cmd += ["-an"]
        cmd += ["-c:v", "libx264", "-preset", "fast", "-crf", "18", "-frames:v", str(total_frames), dest]
        subprocess.run(cmd, check=True, timeout=300)

    if map_audio and target_lufs is not None:
        qa.normalize_audio(dest, target_lufs=target_lufs)

    out_info = probe_video(dest)
    return {
        "path": dest,
        "kare": out_info["frames"],
        "saniye": round(out_info["frames"] / fps, 2),
        "fps": fps,
        "boyut": [w, h],
        "lufs": qa.measure_lufs(dest) if map_audio else None,
        "blok_sayisi": len(clips),
        "kirpilan_son_kareler": applied_trims,
    }

In [ ]:
%%writefile /content/h3_video_app/h3app/hikaye.py
"""Uzun video (16–60 sn) için hikâye kartı, karakter/mekân referans görselleri ve blok planı.

Sorun: H3 tek seferde en fazla 15 sn üretir; uzun video ayrı üretilen bloklardan oluşur. Bloklar arasında kimliği
yalnızca birkaç cümlelik bir metin taşıyınca model yüzü, kıyafeti ve mekânı bloktan bloğa farklı çizer.

Çözüm:
1. Hikâye kartı: karakterlerin görünüşü, kıyafeti ve SESİ; mekân; ışık; bloklar ve her bloğun replikleri bir kez
   yazılır, denetlenir ve her bloğa harfi harfine aynı "dünya çapası" olarak verilir.
2. Referans görseller: her karakter için boydan bir görsel, mekân için boş bir sahne görseli (Gemini görsel modeli
   ya da kullanıcının kendi görselleri). Her blok bu görsellerle Ref2VA olarak üretilir; kimliği görsel taşır.
3. Blok planı: `prompt_writer.plan_shots` karta göre çekimleri planlar, `block_request` + `rewrite` her bloğun H3
   prompt'unu yazar. "metin" kolu (görselsiz, eski yöntem) ve "ref" kolu (Ref2VA) aynı plandan üretilebilir.
"""

import json
import os
import re
import struct
import zlib

from . import config, prompt_writer as pw

MIN_TOPLAM, MAX_TOPLAM = 16, config.MAX_TOTAL_SECONDS
MAX_KARAKTER = 3
KOLLAR = ("metin", "ref")

KART_PROMPT = """You are a film writer and continuity supervisor. Turn a short video idea into a STORY BIBLE for a
{total}-second video made of {n} block(s). An AI video model renders each block separately (4–15 seconds each), so
everything that must stay identical between blocks has to be written down here once, precisely.

Output ONLY valid JSON with exactly this structure (no markdown):
{{
  "baslik": "short Turkish title",
  "karakterler": [
    {{"id": "K1", "ad": "Turkish name or role (e.g. Çaycı Ahmet)", "konusmaci": "S1",
      "gorunum": "English: sex, age, face shape, skin tone, eyes, eyebrows, nose, moustache/beard, hair colour, length and style, build, height",
      "kiyafet": "English: every garment with colour and material, accessories, flat shoes or boots",
      "ses": "English: voice timbre, pitch, pace and accent (e.g. warm deep gravelly baritone, slow pace, Istanbul Turkish)"}}
  ],
  "mekan": {{"ad": "Turkish place name", "tarif": "English: architecture, layout, materials, furniture, props that stay in place, background"}},
  "zaman_isik": "English: time of day, light direction and quality",
  "renk_paleti": "English: 3-5 colours",
  "bloklar": [
    {{"blok": 1, "sure": 15, "ozet": "English: what happens in this block, beat by beat",
      "replikler": [{{"konusmaci": "S1", "metin": "exact Turkish line"}}]}}
  ]
}}

Rules:
- At most {max_char} characters. Only speaking characters get a "konusmaci" (S1, S2... in order of first line);
  silent characters use "konusmaci": "".
- Block durations are whole seconds between 4 and 15 and sum to exactly {total}. Use {n} block(s).
- Every quoted line and quoted on-screen text of the idea appears EXACTLY (same letters and punctuation) in some
  block's "replikler" "metin". Do not add lines the idea does not imply; you may add at most one short line per block
  only if the idea asks for a conversation without giving all lines.
- About 2.5 words per second of speech at most; leave the first and last second of every block without speech.
- Each block ends on a calm moment (no fast motion across the block boundary).
- The place, outfits and props never change between blocks unless the idea says so.
- Turkish fields ("baslik", "ad", "mekan.ad", "metin") keep Turkish letters exactly (ı İ ş Ş ğ Ğ ç Ç ö Ö ü Ü):
  "Rıza", not "Riza"; "Sahaf Dükkânı", not "Sahaf Dukkani".
- Do not move the story to a specific named place (city, district, landmark) the idea does not name.
- "mekan.tarif" describes the set as it is BEFORE the first block starts: an object that a character brings, serves
  or places later in the story (a tea glass, a tray, a letter) must NOT be in it. Props that are already there at
  the start (used glasses waiting to be cleared, furniture) may be.
- Visual style of the video: {kind}."""

PORTRE = """A {kind} full-body character reference image of one person: {gorunum}. Wearing: {kiyafet}.
Standing in a relaxed neutral pose, facing three-quarters toward the viewer, the whole figure from head to feet,
in front of a plain light-grey background, even soft studio light. Exactly one person. {look}
No text, no letters, no logos, no watermark."""

MEKAN_GORSELI = """A {kind} wide establishing image of an empty place, ready for actors: {tarif}. {isik}.
Colour palette: {palet}. Eye-level camera, natural perspective. No people, no animals. {look}
No readable text, no signs, no letters, no logos, no watermark."""


def blok_sayisi(toplam):
    return -(-int(toplam) // config.MAX_SECONDS)


# ---------------- hikâye kartı ----------------

def kart_denetle(kart, toplam, fikir=""):
    """Kartın biçimini ve fikirdeki tırnak içi metinlerin korunmasını denetler; sorun listesi döner."""
    sorunlar = []
    if not isinstance(kart, dict):
        return ["Kart bir JSON nesnesi olmalı."]
    karakterler = kart.get("karakterler") or []
    if not karakterler:
        sorunlar.append("En az bir karakter olmalı.")
    if len(karakterler) > MAX_KARAKTER:
        sorunlar.append(f"En fazla {MAX_KARAKTER} karakter olabilir ({len(karakterler)} var).")
    idler, konusmacilar = set(), []
    for k in karakterler:
        for alan in ("id", "ad", "gorunum", "kiyafet"):
            if not (k.get(alan) or "").strip():
                sorunlar.append(f"Karakter {k.get('id', '?')}: '{alan}' boş.")
        if k.get("id") in idler:
            sorunlar.append(f"Karakter kimliği tekrar ediyor: {k.get('id')}")
        idler.add(k.get("id"))
        if k.get("konusmaci"):
            konusmacilar.append(k["konusmaci"])
            if not (k.get("ses") or "").strip():
                sorunlar.append(f"Konuşan karakter {k.get('id')}: 'ses' tarifi boş.")
    if konusmacilar != [f"S{i}" for i in range(1, len(konusmacilar) + 1)]:
        sorunlar.append(f"Konuşmacı kimlikleri S1, S2… sırasıyla verilmeli (şu an: {konusmacilar}).")
    if not ((kart.get("mekan") or {}).get("tarif") or "").strip():
        sorunlar.append("Mekân tarifi boş.")
    bloklar = kart.get("bloklar") or []
    if not bloklar:
        sorunlar.append("Blok listesi boş.")
    toplam_blok = 0
    for i, b in enumerate(bloklar, 1):
        try:
            s = int(b.get("sure"))
        except (TypeError, ValueError):
            sorunlar.append(f"Blok {i}: süre sayı değil.")
            continue
        if not config.MIN_SECONDS <= s <= config.MAX_SECONDS:
            sorunlar.append(f"Blok {i}: süre {config.MIN_SECONDS}–{config.MAX_SECONDS} sn olmalı ({s}).")
        toplam_blok += s
        if int(b.get("blok", i)) != i:
            sorunlar.append(f"Blok numaraları 1'den başlayıp sırayla artmalı ({b.get('blok')}).")
        for r in b.get("replikler") or []:
            if r.get("konusmaci") not in konusmacilar:
                sorunlar.append(f"Blok {i}: replikteki konuşmacı ({r.get('konusmaci')}) karakterlerde yok.")
            kelime = len((r.get("metin") or "").split())
            if kelime > s * 2.5:
                sorunlar.append(f"Blok {i}: replik bloğa sığmayacak kadar uzun ({kelime} kelime).")
    if toplam_blok != int(toplam):
        sorunlar.append(f"Blok süreleri toplamı {toplam_blok} sn; {toplam} sn olmalı.")
    tum = pw._norm_quotes(json.dumps(kart, ensure_ascii=False))
    for lit in pw.literals(fikir or ""):
        if lit.strip() and pw._norm_quotes(lit.strip()) not in tum:
            sorunlar.append(f'Fikirdeki şu metin kartta yok: "{lit.strip()}"')
    return sorunlar


def kart_yaz(fikir, toplam, api_key, model=None, style=config.DEFAULT_STYLE, retries=1, log=print):
    """Fikir → hikâye kartı (Gemini). (kart, sorunlar) döner; sorun varsa Gemini'ye bir kez düzelttirilir."""
    n = blok_sayisi(toplam)
    sistem = KART_PROMPT.format(total=toplam, n=n, max_char=MAX_KARAKTER, kind=config.STYLES[style]["kind"])
    bas = f"Total duration: {toplam} seconds in {n} block(s).\n\nIdea:\n{fikir.strip()}"
    icerik, kart, sorunlar = [bas], None, ["Gemini yanıt vermedi."]
    for deneme in range(retries + 1):
        ham = pw._generate(api_key, model or config.GEMINI_MODEL, sistem, icerik, 0.6, log=log, step="hikâye kartı")
        try:
            kart = pw._parse_json(ham)
            sorunlar = kart_denetle(kart, toplam, fikir)
        except Exception as e:  # noqa: BLE001
            sorunlar = [f"Geçerli JSON değil: {e}"]
        if not sorunlar:
            break
        log(f"Kartta sorun ({deneme + 1}. deneme): " + " | ".join(sorunlar))
        icerik = [bas + f"\n\nYour previous answer:\n{ham}\n\nFix these problems and output ONLY valid JSON:\n- "
                  + "\n- ".join(sorunlar)]
    return kart, sorunlar


def dunya_capasi(kart):
    """Her blokta ve her kolda harfi harfine aynı kalacak dünya/karakter çapası (İngilizce, ses tarifleriyle)."""
    parcalar = []
    for k in kart["karakterler"]:
        kim = f"{k['ad']}" + (f" ({k['konusmaci']})" if k.get("konusmaci") else "")
        ses = f" Voice: {k['ses']}." if k.get("konusmaci") and k.get("ses") else ""
        parcalar.append(f"{kim}: {k['gorunum']}. Outfit: {k['kiyafet']}.{ses}")
    m = kart["mekan"]
    parcalar.append(f"Place — {m.get('ad', '')}: {m['tarif']}.")
    if kart.get("zaman_isik"):
        parcalar.append(f"Time and light: {kart['zaman_isik']}.")
    if kart.get("renk_paleti"):
        parcalar.append(f"Colour palette: {kart['renk_paleti']}.")
    parcalar.append("Faces, hair, outfits, voices, the place and its props stay exactly the same in every block.")
    return " ".join(parcalar)


def plan_fikri(kart):
    """plan_shots için fikir metni: kart blok blok, replikler tırnak içinde (planda korunmaları denetlenir)."""
    satirlar = ["Follow this story bible exactly. Keep the block count and each block's duration as given.",
                "Characters: " + "; ".join(
                    f"{k['ad']}" + (f" (speaker {k['konusmaci']}, voice: {k.get('ses', '')})" if k.get("konusmaci") else "")
                    for k in kart["karakterler"]),
                f"Place: {kart['mekan'].get('ad', '')} — {kart['mekan']['tarif']}"]
    ad = {k["konusmaci"]: k["ad"] for k in kart["karakterler"] if k.get("konusmaci")}
    for b in kart["bloklar"]:
        satir = f"Block {b['blok']} — exactly {b['sure']} seconds: {b['ozet']}"
        for r in b.get("replikler") or []:
            satir += f' {ad.get(r["konusmaci"], r["konusmaci"])} ({r["konusmaci"]}) says in Turkish: "{r["metin"]}"'
        satirlar.append(satir)
    return "\n".join(satirlar)


# ---------------- referans görseller ----------------

def gorsel_listesi(kart, dizin):
    """Ref2VA görselleri, sırasıyla: konuşan karakterler (S1, S2… — normalize_roles numarasıyla eşleşir),
    sessiz karakterler, en son mekân. [{'id','dosya','rol','ad'}]"""
    karakterler = sorted(kart["karakterler"], key=lambda k: (not k.get("konusmaci"), k.get("konusmaci") or "", k["id"]))
    liste = [{"id": k["id"], "dosya": os.path.join(dizin, f"{k['id']}.png"), "rol": pw.ROLE_CHARACTER, "ad": k["ad"]}
             for k in karakterler]
    liste.append({"id": "MEKAN", "dosya": os.path.join(dizin, "MEKAN.png"), "rol": pw.ROLE_SCENE,
                  "ad": kart["mekan"].get("ad", "")})
    return liste


def gorsel_metni(kart, kimlik, style=config.DEFAULT_STYLE):
    """Bir karakterin ya da mekânın (kimlik='MEKAN') referans görseli için Gemini'ye gidecek metin."""
    s = config.STYLES[style]
    look = f"Style: {s['look']}. Avoid: {s['avoid']}." if s.get("look") else "Photorealistic, natural skin texture."
    if kimlik == "MEKAN":
        m = kart["mekan"]
        return MEKAN_GORSELI.format(kind=s["kind"], tarif=m["tarif"], isik=kart.get("zaman_isik", ""),
                                    palet=kart.get("renk_paleti", ""), look=look)
    k = next(x for x in kart["karakterler"] if x["id"] == kimlik)
    return PORTRE.format(kind=s["kind"], gorunum=k["gorunum"], kiyafet=k["kiyafet"], look=look)


def gorsel_uret(metin, api_key, hedef, models=None):
    """Gemini görsel modeliyle tek görsel üretir ve `hedef`e yazar; kullanılan modeli döner."""
    from google.genai import types
    modeller = list(models or pw.image_models(api_key))
    if not modeller:
        raise RuntimeError("Bu anahtarla görsel üretebilen Gemini modeli yok.")
    client = pw._client(api_key, pw.IMAGE_TIMEOUT_MS)
    cfg = types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"], temperature=1.0)
    hatalar = []
    for model in modeller[:2]:
        try:
            resp = client.models.generate_content(model=model, contents=[metin], config=cfg)
        except Exception as e:  # noqa: BLE001
            hatalar.append(f"{model}: {str(e)[:150]}")
            continue
        for cand in resp.candidates or []:
            for part in (cand.content.parts if cand.content else None) or []:
                if getattr(part, "inline_data", None) and part.inline_data.data:
                    os.makedirs(os.path.dirname(os.path.abspath(hedef)), exist_ok=True)
                    with open(hedef, "wb") as f:
                        f.write(part.inline_data.data)
                    return model
        hatalar.append(f"{model}: görsel dönmedi")
    raise RuntimeError("Görsel üretilemedi: " + " | ".join(hatalar))


# ---------------- plan ----------------

def planla(kart, api_key, gorseller=(), kollar=("ref",), model=None, style=config.DEFAULT_STYLE, fikir="", log=print):
    """Kart (+ görseller) → çekim planı → her kol için blok prompt'ları.
    (plan, {kol: [{'blok','sure','istek','prompt','sorunlar'}]}, sorunlar) döner."""
    model = model or config.GEMINI_MODEL
    yollar = [g["dosya"] for g in gorseller]
    roller = [{"role": g["rol"], "name": g["ad"]} for g in gorseller]
    ref_var = "ref" in kollar
    toplam = sum(int(b["sure"]) for b in kart["bloklar"])
    plan, psorun = pw.plan_shots(plan_fikri(kart), toplam, api_key, model, images=yollar if ref_var else (),
                                 style=style, log=log, roles=roller if ref_var else None)
    if psorun:
        return plan, {}, [f"çekim planı: {s}" for s in psorun]
    plan["world_anchor"] = dunya_capasi(kart)
    return plan, *_kol_promptlari(plan, kollar, yollar, roller, style, fikir,
                                  lambda istek, sn, imgs, rls: pw.rewrite(istek, sn, api_key, model, imgs, log=log,
                                                                          style=style, roles=rls))


def _kol_promptlari(plan, kollar, yollar, roller, style, fikir, yaz):
    sonuc, sorunlar = {}, []
    for kol in kollar:
        imgs, rls = (yollar, roller) if kol == "ref" else ((), None)
        bloklar = []
        for binfo in pw.blocks_from_plan(plan):
            istek = pw.block_request(binfo, style=style, idea=fikir)
            prompt, sorun = yaz(istek, binfo["seconds"], imgs, rls)
            bloklar.append({"blok": binfo["block"], "sure": binfo["seconds"], "istek": istek,
                            "prompt": prompt, "sorunlar": list(sorun)})
            sorunlar += [f"[{kol}] blok {binfo['block']}: {s}" for s in sorun]
        sonuc[kol] = bloklar
    return sonuc, sorunlar


# ---------------- test / kuru deneme yardımcıları (Gemini'siz) ----------------

def ornek_kart(fikir, toplam):
    """Gemini'siz deneme kartı: fikirdeki tırnak içi metinler sırayla bloklara dağıtılır."""
    lits = re.findall(r'"([^"]+)"', fikir or "") or ["Merhaba."]
    n = blok_sayisi(toplam)
    sureler = [toplam // n + (1 if i < toplam % n else 0) for i in range(n)]
    return {
        "baslik": "Deneme",
        "karakterler": [{"id": "K1", "ad": "Çaycı", "konusmaci": "S1", "gorunum": "a man in his fifties",
                         "kiyafet": "white apron", "ses": "warm deep voice"}],
        "mekan": {"ad": "Çay bahçesi", "tarif": "a seaside tea garden"},
        "zaman_isik": "golden hour", "renk_paleti": "amber, teal",
        "bloklar": [{"blok": i + 1, "sure": s, "ozet": "the tea seller serves tea",
                     "replikler": [{"konusmaci": "S1", "metin": lits[i]}] if i < len(lits) else []}
                    for i, s in enumerate(sureler)],
    }


def ornek_plan(kart):
    shots = []
    for b in kart["bloklar"]:
        yarim = b["sure"] / 2
        sesler = " ".join(f'"{r["metin"]}"' for r in b.get("replikler") or [])
        shots += [{"block": b["blok"], "shot": 1, "start_sec": 0.0, "duration": yarim, "framing": "wide",
                   "action": b["ozet"], "camera": "static", "sound": sesler or "ambience"},
                  {"block": b["blok"], "shot": 2, "start_sec": yarim, "duration": b["sure"] - yarim,
                   "framing": "medium", "action": "calm hold", "camera": "static", "sound": "ambience"}]
    return {"world_anchor": dunya_capasi(kart), "shots": shots}


def ornek_planla(kart, kollar=("ref",), gorseller=(), style=config.DEFAULT_STYLE, fikir=""):
    """planla()'nın Gemini'siz karşılığı (kuru deneme ve testler için)."""
    plan = ornek_plan(kart)
    return plan, *_kol_promptlari(plan, kollar, [g["dosya"] for g in gorseller],
                                  [{"role": g["rol"], "name": g["ad"]} for g in gorseller], style, fikir,
                                  lambda *a: (pw.EXAMPLE_OUTPUT, []))


def ornek_png(yol, renk=(200, 180, 150), boyut=(64, 96)):
    """Bağımlılıksız küçük bir PNG (yalnızca deneme)."""
    w, h = boyut
    ham = b"".join(b"\x00" + bytes(renk) * w for _ in range(h))

    def parca(tur, veri):
        return struct.pack(">I", len(veri)) + tur + veri + struct.pack(">I", zlib.crc32(tur + veri) & 0xFFFFFFFF)
    os.makedirs(os.path.dirname(os.path.abspath(yol)), exist_ok=True)
    with open(yol, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + parca(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + parca(b"IDAT", zlib.compress(ham)) + parca(b"IEND", b""))

In [ ]:
%%writefile /content/h3_video_app/h3app/uzun.py
"""Uzun videonun GPU tarafı: hikâye kartından yazılmış blokları üretir, izler, birleştirir ve sürekliliği puanlar.

İki kullanım:
1. Parça parça (başsız ajan akışı, önerilen): `prepare` → her blok için `render_block` → `merge`. Gemini izlemesi,
   yeniden üretim kararı, süreklilik puanı ve köprü prompt'u çağıranın makinesinde yapılır; GPU makinesi Gemini'ye
   hiç gitmez (anahtar oraya taşınmaz).
2. Tek seferde: `generate` — hepsini aynı makinede yapar (Gemini anahtarı o makinede geçerliyse).

Kollar: ref (kimliği referans görseller taşır, Ref2VA) / metin (yalnızca dünya çapası metni, T2VA). Deneyde iki kol
aynı plan ve aynı seed'lerle üretilir. Köprü (deneysel): ref kolunda her bloğa önceki bloğun son karesi de sahne
görseli olarak verilir ve bloğun prompt'u o görselle yeniden yazılır.
"""

import json
import os
import shutil
import subprocess
import time

from . import assemble, comfy_client, config, graphs, prompt_writer as pw, qa, review, setup_env

_hazir = {}


def last_frame(video, dest):
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-sseof", "-0.2", "-i", video, "-frames:v", "1", dest], check=True)
    return dest


def block_cuts(blocks):
    """Birleşik videoda blok sınırlarının zamanları (sn)."""
    cuts, t = [], 0.0
    for b in blocks[:-1]:
        t += config.frames_for(b["sure"]) / 24
        cuts.append(round(t, 2))
    return cuts


def _fake():
    return os.environ.get("H3APP_FAKE") == "1"


def prepare(method, with_refs=True, without_refs=False, log=print):
    """ComfyUI'yi ve gereken modelleri hazırlar (tekrar çağrılırsa hızlı geçer). Metin kodlayıcı rolünü döner."""
    if _fake():
        log("Test modu: kurulum atlandı.")
        return None
    key = (method, with_refs, without_refs)
    if key in _hazir:
        return _hazir[key]
    g = setup_env.gpu()
    if not g:
        raise RuntimeError("GPU yok.")
    log(f"GPU: {g[0]} ({g[1]} GB).")
    te = setup_env.text_encoder_role()
    setup_env.ensure_comfy(log=log)
    need = []
    for ref in ([True] if with_refs else []) + ([False] if without_refs else []):
        for r in setup_env.roles_for([method], te, ref):
            if r not in need:
                need.append(r)
    setup_env.ensure_models(need, log=log)
    comfy_client.start(log=log)
    _hazir[key] = te
    return te


def render_block(method, prompt, seed, seconds, resolution, style, name, refs=(), te=None, log=print):
    """Bir bloğu üretir (refs verilirse Ref2VA), sesi dengeler, otomatik kontrol yapar.
    {'yol','sure_sn','kontrol','temas','kirpinti'} döner."""
    width, height = config.RESOLUTIONS[resolution]
    out_dir = config.OUTPUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    dest = os.path.join(out_dir, name)
    t0 = time.time()
    if _fake():
        time.sleep(0.2)
        shutil.copyfile(os.environ["H3APP_FAKE_VIDEO"], dest)
    else:
        names = []
        os.makedirs(os.path.join(config.COMFY_DIR, "input"), exist_ok=True)
        for p in refs:
            n = f"h3app_{os.path.basename(p)}"
            shutil.copyfile(p, os.path.join(config.COMFY_DIR, "input", n))
            names.append(n)
        grf = graphs.build(method, prompt, seed, width, height, config.frames_for(seconds), te,
                           prefix=f"h3app/{name[:-4]}", ref_images=names)
        shown = 0
        for sec in comfy_client.run(grf, dest):
            if sec // 60 > shown:
                shown = sec // 60
                log(f"   … {shown} dk")
    qa.normalize_audio(dest)
    out = {"yol": dest, "sure_sn": int(time.time() - t0), "kontrol": None, "temas": None, "kirpinti": None}
    try:
        r = qa.check(dest, prompt, sheet=dest[:-4] + "_kontrol.jpg", crops=dest[:-4] + "_kirpinti.jpg", style=style)
        log("   " + qa.summary(r))
        out.update(kontrol={k: r[k] for k in ("kesmeler", "bulunamayan", "donmus", "lufs", "son_kayma")},
                   temas=r["temas"], kirpinti=r["kirpinti"])
    except Exception as e:  # noqa: BLE001
        log(f"   Kontrol yapılamadı: {e}")
    return out


def merge(files, dest, style, log=print):
    """Blokları kare-tam birleştirir (konuşmalı ses modu) ve birleşik videoyu kontrol eder."""
    res = assemble.assemble(list(files), dest, audio_mode="speech")
    out = {"yol": dest, "saniye": res.get("saniye"), "temas": None, "kirpinti": None}
    try:
        r = qa.check(dest, "", sheet=dest[:-4] + "_kontrol.jpg", crops=dest[:-4] + "_kirpinti.jpg", style=style)
        out.update(temas=r["temas"], kirpinti=r["kirpinti"])
    except Exception as e:  # noqa: BLE001
        log(f"   Birleşik kontrol yapılamadı: {e}")
    return out


def generate(arms, method, resolution, seed, style, refs=(), roles=None, bible=None, api_key=None, retries=0,
             bridge=False, experiment=False, log=print):
    """Hepsini tek makinede yapar. arms: {kol: [{'blok','sure','istek','prompt'}]}; refs: Ref2VA görsel yolları.
    {'secilen', 'dosyalar', 'temel', 'ozet'} döner."""
    fake = _fake()
    stamp = time.strftime("%Y%m%d_%H%M%S")
    out_dir = config.OUTPUT_DIR
    refs, roles = list(refs or []), list(roles or [])
    retries = max(0, min(3, int(retries)))
    watch = bool(api_key) and not fake
    bridge = bool(bridge) and (bool(api_key) or fake)
    te = prepare(method, with_refs="ref" in arms, without_refs=any(a != "ref" for a in arms), log=log)
    files, essential = [], []
    summary = {"deney": bool(experiment), "kollar": {}}
    mfile = config.METHODS[method]["file"]
    for arm, blocks in arms.items():
        use_refs = arm == "ref"
        log(f"━━ Kol: {arm} ({'görsel referanslı, Ref2VA' if use_refs else 'yalnızca metin çapası'}) ━━")
        a_imgs, a_roles = (refs, roles or None) if use_refs else ([], None)
        chosen, arm_summary = [], []
        for b in blocks:
            n, seconds = b["blok"], b["sure"]
            prompt, b_imgs, b_roles = b["prompt"], list(a_imgs), a_roles
            if bridge and use_refs and chosen and len(a_imgs) < config.MAX_REF_IMAGES:
                frame = last_frame(chosen[-1], os.path.join(out_dir, f"{stamp}_{arm}_blok{n}_kopru.png"))
                b_imgs = a_imgs + [frame]
                b_roles = (a_roles or []) + [{"role": pw.ROLE_SCENE, "name": "final frame of the previous block"}]
                log(f"🔗 Blok {n}: önceki bloğun son karesi sahne görseli olarak eklendi; prompt yeniden yazılıyor")
                if fake:
                    log("   Test modu: prompt yeniden yazılmadı.")
                else:
                    prompt, probs = pw.rewrite(b["istek"], seconds, api_key, config.GEMINI_MODEL, b_imgs, log=log,
                                               style=style, roles=b_roles)
                    if probs:
                        log(f"   ⚠️ Köprü prompt'unda uyarı: {' | '.join(probs)}")
                files.append(frame)
            pfile = os.path.join(out_dir, f"{stamp}_{arm}_blok{n}_prompt.txt")
            with open(pfile, "w", encoding="utf-8") as f:
                f.write(prompt.strip() + "\n")
            essential.append(pfile)
            cands = []
            for attempt in range(retries + 1):
                s = int(seed) + (n - 1) + 1000 * attempt
                name = f"{stamp}_{arm}_blok{n}_{mfile}_seed{s}.mp4"
                log(f"▶ [{arm}] Blok {n}/{len(blocks)} · {seconds} sn · seed {s}")
                r = render_block(method, prompt, s, seconds, resolution, style, name, b_imgs, te, log)
                files += [x for x in (r["yol"], r["temas"], r["kirpinti"]) if x]
                cand = {"dosya": name, "yol": r["yol"], "seed": s, "sure_sn": r["sure_sn"], "inceleme": None}
                if watch:
                    cand["inceleme"] = review.review_video(r["yol"], prompt, api_key, b_imgs, b_roles, log=log)
                    log("   " + review.summary(cand["inceleme"]))
                cands.append(cand)
                if not review.blocking(cand["inceleme"]):
                    break
                if attempt < retries:
                    log(f"🔁 Blok {n}: yüksek önemde kusur; yeni seed ile yeniden üretiliyor")
            best = min(cands, key=lambda c: review.score(c["inceleme"])) if watch else cands[-1]
            chosen.append(best["yol"])
            essential.append(best["yol"][:-4] + "_kontrol.jpg")
            arm_summary.append({"blok": n, "secilen": best["dosya"], "seed": best["seed"],
                                "inceleme": review.short(best["inceleme"]) if watch else "atlandı",
                                "adaylar": [{k: c.get(k) for k in ("dosya", "seed", "sure_sn", "inceleme")}
                                            for c in cands]})
        log(f"🎬 [{arm}] {len(chosen)} blok birleştiriliyor")
        m = merge(chosen, os.path.join(out_dir, f"{stamp}_{arm}_{mfile}_birlesik.mp4"), style, log)
        if m["temas"]:
            essential.append(m["temas"])
        if m["kirpinti"]:
            files.append(m["kirpinti"])
        cuts = block_cuts(blocks)
        cont = None
        if watch:
            log(f"🎞️ [{arm}] Süreklilik izleniyor")
            cont = review.continuity_review(m["yol"], cuts, bible or {}, api_key, log=log)
        essential.append(m["yol"])
        summary["kollar"][arm] = {"birlesik": os.path.basename(m["yol"]), "saniye": m["saniye"],
                                  "bloklar": arm_summary, "sureklilik": cont, "blok_kesmeleri": cuts}
    if experiment:
        summary["karsilastirma"] = {k: (v["sureklilik"] or {}).get("ortalama") for k, v in summary["kollar"].items()}
        log(f"⚖️ Deney sonucu (süreklilik ortalaması): {summary['karsilastirma']}")
    result_file = os.path.join(out_dir, f"{stamp}_uzun_sonuc.json")
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    essential.insert(0, result_file)
    main_arm = "ref" if "ref" in summary["kollar"] else next(iter(summary["kollar"]))
    return {"secilen": os.path.join(out_dir, summary["kollar"][main_arm]["birlesik"]), "dosyalar": files,
            "temel": essential, "ozet": summary}

In [ ]:
%%writefile /content/h3_video_app/h3app/ui.py
"""Gradio arayüzü: fikir → Gemini ile H3 prompt'u → Pika H3 Flex / MiniMax-H3 videosu."""

import copy
import datetime
import inspect
import json
import os
import queue
import re
import shutil
import tempfile
import threading
import time

import gradio as gr

from . import assemble, comfy_client, config, graphs, prompt_writer, qa, review, setup_env

FAKE = os.environ.get("H3APP_FAKE") == "1"   # yerel arayüz testi: GPU ve Gemini olmadan
# H3APP_FAKE=1 ile birlikte H3APP_REAL_GEMINI=1: video sahte, Gemini gerçek (yerel prompt denemesi)
REAL_GEMINI = os.environ.get("H3APP_REAL_GEMINI") == "1"


def _fake_gemini(key):
    return FAKE and not (REAL_GEMINI and key)
_lock = threading.Lock()
_state = {"secret_key": None, "secret_hint": None, "drive_dir": None}
STAMP_RE = re.compile(r"^(\d{8}_\d{6})_")


def _to_pil(path):
    """Galeri bileşenlerinde proxyPort(7860) iç adres sorununu önlemek için görseli PIL.Image olarak yükler."""
    if not isinstance(path, str) or not os.path.isfile(path):
        return path
    try:
        from PIL import Image
        with Image.open(path) as im:
            im.load()
            return im.copy()
    except Exception:
        return os.path.abspath(path)


def _preview_refs(files):
    """Yüklenen referans görsellerini galeri önizlemesi için PIL.Image listesine çevirir."""
    return [_to_pil(p) for p in _paths(files)]


def _pair_videos(vdict):
    """Sol oynatıcıda taslak (Pika veya Turbo), sağ oynatıcıda final (H3; yoksa ikinci taslak) gösterilir."""
    if vdict.get("turbo") and vdict.get("pika") and not vdict.get("h3"):
        return vdict["turbo"], vdict["pika"]
    return vdict.get("pika") or vdict.get("turbo"), vdict.get("h3")


def _write_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    return path


def _sync_to_drive(paths, say=None):
    """Verilen dosya veya dosya listesini Drive klasörü açıksa anında oraya kopyalar (kesinti koruması)."""
    drive_dir = _state.get("drive_dir")
    if not drive_dir:
        return
    os.makedirs(drive_dir, exist_ok=True)
    if isinstance(paths, str):
        paths = [paths]
    copied = []
    for p in paths:
        if p and isinstance(p, str) and os.path.isfile(p):
            shutil.copyfile(p, os.path.join(drive_dir, os.path.basename(p)))
            copied.append(os.path.basename(p))
    if copied and say:
        say(f"   💾 Drive'a anında kaydedildi ({drive_dir}): {', '.join(copied)}")


def _colab_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception as e:
        # Sır var ama "Notebook erişimi" kapalıysa NotebookAccessError gelir: kullanıcıya nedenini söyle
        if "Access" in type(e).__name__ or "access" in str(e).lower():
            _state["secret_hint"] = (f"Colab Secrets'ta {name} var ama bu notebook'a erişimi kapalı: 🔑 panelinde "
                                     "'Notebook erişimi' anahtarını açıp hücreyi yeniden çalıştırın")
    return (os.environ.get(name) or "").strip() or None


def _key(typed):
    return (typed or "").strip() or _state["secret_key"]


def _status_line():
    g = setup_env.gpu()
    gpu = f"{g[0]} · {g[1]} GB" if g else "GPU yok (Çalışma zamanı → türünü değiştir → G4 GPU seçin)"
    if g and g[1] < 40:
        gpu += " ⛔ H3 için yetersiz (en az 40 GB; G4 ya da A100 seçin)"
    key = ("Colab Secrets'tan alındı" if _state["secret_key"]
           else f"girilmedi ⚠️ ({_state['secret_hint']})" if _state.get("secret_hint") else
           "girilmedi (Colab Secrets'a GEMINI ya da GEMINI_API_KEY adıyla ekleyin)")
    drive = _state["drive_dir"] or "kapalı ⚠️ (bağlantı koparsa videolar kaybolur)"
    models = (f"Drive'a da kaydedilir ({config.MODEL_SAVE_DIR})" if config.MODEL_SAVE_DIR
              else ("HF'ten iner, HF yoksa Drive'dan" if config.MODEL_SOURCE_DIRS else "her oturumda indirilir"))
    return (f"**GPU:** {gpu} &nbsp;·&nbsp; **Gemini anahtarı:** {key} &nbsp;·&nbsp; **Drive'a kayıt:** {drive}"
            f" &nbsp;·&nbsp; **Modeller:** {models}")


def list_models(typed_key):
    key = _key(typed_key)
    if _fake_gemini(key):
        return gr.update(choices=["gemini-test-pro"], value="gemini-test-pro"), "Model listesi alındı (test modu)."
    if not key:
        return gr.update(), "Önce Gemini API anahtarını girin ya da Colab Secrets'a GEMINI (veya GEMINI_API_KEY) adıyla ekleyip notebook erişimini açın."
    try:
        names = prompt_writer.gemini_models(key)
    except Exception as e:
        return gr.update(), f"Gemini model listesi alınamadı: {e}"
    if not names:
        return gr.update(), "Bu anahtarla kullanılabilen Gemini metin modeli bulunamadı."
    return gr.update(choices=names, value=names[0]), f"{len(names)} model bulundu; en yenisi seçildi: {names[0]}"


def _paths(files):
    return [f if isinstance(f, str) else f.name for f in (files or [])][:config.MAX_REF_IMAGES]


def _refs(refs, style, picks, styled):
    """Kullanılacak referans görseller, yükleme sırasıyla. Çizgi/3D stilde bir görsel için işaretli stil çizimi
    varsa ("Görsel N · A/B") o kullanılır, yoksa yüklenen görsel; böylece sıra ve roller hiç kaymaz."""
    paths = _paths(refs)
    if style == config.DEFAULT_STYLE or not picks:
        return paths
    out = []
    for i, src in enumerate(paths, 1):
        chosen = next((styled[l] for l in picks if l.startswith(f"Görsel {i} ·") and l in (styled or {})), None)
        out.append(chosen or src)
    return out


ROLE_CHOICES = [("Karakter", prompt_writer.ROLE_CHARACTER), ("Arka plan / mekân", prompt_writer.ROLE_SCENE),
                ("Nesne / eşya", prompt_writer.ROLE_OBJECT)]


def _roles(ref_roles, refs):
    """Arayüzdeki rol listesini yüklenen görsel sayısına göre hizalar (eksikler karakter sayılır)."""
    n = len(_paths(refs))
    return prompt_writer.normalize_roles(list(ref_roles or [])[:n], n) if n else []


def _role_caption(i, spec):
    name = f" — {spec['name']}" if spec.get("name") else ""
    if spec["role"] == prompt_writer.ROLE_SCENE:
        return f"Görsel {i} · Arka plan{name}"
    if spec["role"] == prompt_writer.ROLE_OBJECT:
        return f"Görsel {i} · Nesne{name}"
    return f"Görsel {i} · Karakter {spec['char']} (S{spec['char']}){name}"


def roles_summary(ref_roles, refs):
    """Kullanıcıya gösterilen tek satırlık özet: hangi görsel ne olarak kullanılacak."""
    specs = _roles(ref_roles, refs)
    return " · ".join(_role_caption(i, sp) for i, sp in enumerate(specs, 1))


def _on_refs(files):
    """Görsel yüklenince: numaralı önizleme, her görsel için rol satırı (varsayılan karakter) ve rol durumu."""
    paths = _paths(files)
    roles = [{"role": prompt_writer.ROLE_CHARACTER, "name": ""} for _ in paths]
    specs = prompt_writer.normalize_roles(roles, len(paths))
    gallery = [(_to_pil(p), _role_caption(i, sp)) for i, (p, sp) in enumerate(zip(paths, specs), 1)]
    rows = [gr.update(visible=i < len(paths)) for i in range(config.MAX_REF_IMAGES)]
    dds = [gr.update(value=prompt_writer.ROLE_CHARACTER,
                     label=f"Görsel {i + 1} rolü" + (f" ({os.path.basename(paths[i])})" if i < len(paths) else ""))
           for i in range(config.MAX_REF_IMAGES)]
    names = [gr.update(value="") for _ in range(config.MAX_REF_IMAGES)]
    return (gallery or None, *rows, *dds, *names, roles)


def _collect_roles(files, *vals):
    """Rol açılır menüleri ve ad kutularından rol listesi + güncel numaralı önizleme."""
    n = len(_paths(files))
    dds, names = vals[:config.MAX_REF_IMAGES], vals[config.MAX_REF_IMAGES:]
    roles = [{"role": dds[i] or prompt_writer.ROLE_CHARACTER, "name": (names[i] or "").strip()} for i in range(n)]
    specs = prompt_writer.normalize_roles(roles, n)
    gallery = [(_to_pil(p), _role_caption(i, sp)) for i, (p, sp) in enumerate(zip(_paths(files), specs), 1)]
    return roles, (gallery or None)


def _roles_from_inputs(vals):
    """Rol açılır menüleri + ad kutularının o anki değerlerinden rol listesi (MAX_REF_IMAGES uzunluğunda)."""
    m = len(vals) // 2
    return [{"role": vals[i] or prompt_writer.ROLE_CHARACTER, "name": (vals[m + i] or "").strip()} for i in range(m)]


def _live_roles(fn, inputs, state, role_inputs):
    """İşleyicinin girdilerindeki rol State'ini, rol menüleri ve ad kutularının kendisiyle değiştirir.
    17.11.2026: ad kutusu her tuşta State'e yazıyordu; Colab proxy'si güncellemeyi geciktirince "Patron Kedi"
    yerine ilk harf "P" kaydedildi. Değerler tıklama anında doğrudan okunursa yarış olmaz."""
    i = next(j for j, c in enumerate(inputs) if c is state)
    k = len(role_inputs)
    if inspect.isgeneratorfunction(fn):
        def wrapped(*a):
            yield from fn(*a[:i], _roles_from_inputs(a[i:i + k]), *a[i + k:])
    else:
        def wrapped(*a):
            return fn(*a[:i], _roles_from_inputs(a[i:i + k]), *a[i + k:])
    wrapped.__name__ = getattr(fn, "__name__", "handler")
    return wrapped, list(inputs[:i]) + list(role_inputs) + list(inputs[i + 1:])


def do_stylize(refs, style, typed_key, idea="", ref_roles=None):
    """Yüklenen her referansı seçilen stilde 2 kez yeniden çizdirir: (galeri, seçim kutusu, eşlem, mesaj)."""
    none = (None, gr.update(choices=[], value=[]), {})
    images = _paths(refs)
    if style == config.DEFAULT_STYLE:
        return *none, "Gerçek dünya stilinde görseller olduğu gibi kullanılır."
    if not images:
        return *none, "Önce bir karakter / referans görseli yükleyin."
    key = _key(typed_key)
    use_fake = _fake_gemini(key)
    if not key and not use_fake:
        return *none, "❌ Gemini API anahtarı yok."
    dest = os.path.join(config.OUTPUT_DIR, "stil_referans")
    styled, labels, gallery = {}, [], []
    try:
        model = None if use_fake else (prompt_writer.image_models(key) or [None])[0]
        if not use_fake and not model:
            return *none, "❌ Bu anahtarla görsel üretebilen Gemini modeli bulunamadı."
        specs = _roles(ref_roles, refs)
        for i, src in enumerate(images, 1):
            if use_fake:
                os.makedirs(dest, exist_ok=True)
                outs = []
                for v in "AB":
                    outs.append(os.path.join(dest, f"test_{i}_{v}{os.path.splitext(src)[1]}"))
                    shutil.copyfile(src, outs[-1])
            else:
                outs = prompt_writer.stylize_reference(src, style, key, model, dest, idea=idea or "",
                                                       role=specs[i - 1]["role"] if specs else None)
            for j, path in enumerate(outs):
                label = f"Görsel {i} · {chr(65 + j)}"
                styled[label] = path
                labels.append(label)
                gallery.append((_to_pil(path), label))
    except Exception as e:
        return *none, f"❌ Görsel stile çevrilemedi: {e}"
    first = [l for l in labels if l.endswith("· A")]
    name = config.STYLES[style]["label"]
    return (gallery, gr.update(choices=labels, value=first), styled,
            f"✅ {len(images)} görsel {name} stilinde çizildi{'' if use_fake else f' ({model})'}. "
            "Her görsel için birini işaretleyin; Ref2VA'ya işaretlenenler verilir.")


def _ready(typed_key, model):
    """(anahtar, kullanılacak model, hata). Model seçilmemişse anahtarla görülen en yeni model seçilir."""
    key = _key(typed_key)
    if _fake_gemini(key):
        return key, model, None
    if not key:
        return None, model, "❌ Gemini API anahtarı yok."
    if not model or model == "gemini-test-pro":
        try:
            names = prompt_writer.gemini_models(key)
            if names:
                model = names[0]
        except Exception:
            pass
    if not model or model == "gemini-test-pro":
        return None, model, "❌ Önce 'Modelleri listele' ile bir Gemini modeli seçin."
    return key, model, None


def _used_model(notes, model):
    """Yedek modele geçildiyse metni gerçekten yazan model; yoksa seçili model."""
    for n in reversed(notes):
        m = re.match(r"↪️ .*?yazıldı: (\S+)", n)
        if m:
            return m.group(1)
    return model


def do_expand(idea, seconds, model, typed_key, refs, style, picks, styled, ref_roles=None):
    if not (idea or "").strip():
        return gr.update(), "Önce kısa fikri yazın."
    key, model, err = _ready(typed_key, model)
    if err:
        return gr.update(), err
    if _fake_gemini(key):
        time.sleep(1)
        return prompt_writer.BRIEF_EXAMPLES[1][1], "✅ Test modu: örnek brief yüklendi."
    try:
        notes = []
        brief = prompt_writer.expand(idea, int(seconds), key, model, _refs(refs, style, picks, styled),
                                     roles=_roles(ref_roles, refs),
                                     log=notes.append, style=style)
    except Exception as e:
        return gr.update(), f"❌ Gemini hatası: {e}"
    missing = [l for l in prompt_writer.literals(idea)
               if prompt_writer._norm_quotes(l.strip()) not in prompt_writer._norm_quotes(brief)]
    if missing:
        return brief, "⚠️ Brief'te şu metinler birebir yok, düzeltin: " + ", ".join(f'"{m}"' for m in missing)
    return brief, (f"✅ Detaylı brief hazır ({_used_model(notes, model)}). İsterseniz düzenleyin, sonra H3 prompt'una çevirin."
                   + "".join("\n\n" + n for n in notes))


def do_rewrite(idea, brief, seconds, model, typed_key, refs, style, picks, styled, ref_roles=None):
    request = (brief or "").strip() or (idea or "").strip()
    if not request:
        return gr.update(), "Önce fikri ya da brief'i yazın."
    key, model, err = _ready(typed_key, model)
    if err:
        return gr.update(), err
    images = _refs(refs, style, picks, styled)
    if _fake_gemini(key):
        time.sleep(1)
        return prompt_writer.EXAMPLE_OUTPUT, "✅ Test modu: örnek prompt yüklendi."
    notes = []
    try:
        prompt, problems = prompt_writer.rewrite(request, int(seconds), key, model, images, log=notes.append,
                                                 roles=_roles(ref_roles, refs),
                                                 style=style)
    except Exception as e:
        return gr.update(), f"❌ Gemini hatası: {e}"
    if problems:
        return prompt, "⚠️ Gemini çıktısında hâlâ sorun var, prompt'u elle düzeltin:\n- " + "\n- ".join(problems)
    lits = prompt_writer.literals(request)
    kept = f" Tırnak içindeki {len(lits)} metin harfi harfine korundu." if lits else ""
    mode = f" Referanslı mod: {roles_summary(ref_roles, refs)}." if images else ""
    mode += f" Stil: {config.STYLES[style]['label']}."
    return prompt, f"✅ H3 prompt'u hazır ({_used_model(notes, model)}).{kept}{mode}" + "".join("\n\n" + n for n in notes)


BLOCK_SEP_RE = re.compile(r"^===\s*BLOK\s*(\d+)\s*(?:\((\d+)\s*sn\))?\s*===\s*$", re.I | re.M)


def _format_block_prompts(blocks_out):
    """[{'block', 'seconds', 'prompt'}] listesini tek metin kutusunda düzenlenebilir biçime çevirir."""
    parts = []
    for b in blocks_out:
        parts.append(f"=== BLOK {b['block']} ({b['seconds']} sn) ===\n{(b.get('prompt') or '').strip()}")
    return "\n\n".join(parts)


def _parse_block_prompts(text, fallback_blocks=None):
    """Metin kutusundaki `=== BLOK N (S sn) ===` bölümlerini [{'block', 'seconds', 'prompt'}] listesine ayrıştırır."""
    text = (text or "").strip()
    if not text:
        return []
    matches = list(BLOCK_SEP_RE.finditer(text))
    if not matches:
        sec = (fallback_blocks[0]["seconds"] if fallback_blocks else 15)
        return [{"block": 1, "seconds": sec, "prompt": text}]
    sec_map = {b["block"]: b["seconds"] for b in (fallback_blocks or [])}
    out = []
    for idx, m in enumerate(matches):
        bnum = int(m.group(1))
        sec = int(m.group(2)) if m.group(2) else sec_map.get(bnum, 15)
        start = m.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            out.append({"block": bnum, "seconds": sec, "prompt": body})
    return out


def do_plan_shots(idea, total_seconds, audio_mode, model, typed_key, refs, style, picks, styled, ref_roles=None):
    """Kısa fikir + toplam süre → ortak dünya çapası + düzenlenebilir blok/çekim tablosu."""
    if not (idea or "").strip():
        return gr.update(), gr.update(), "Önce kısa fikri yazın."
    key, model, err = _ready(typed_key, model)
    if err:
        return gr.update(), gr.update(), err
    images = _refs(refs, style, picks, styled)
    total_sec = int(round(float(total_seconds)))
    if _fake_gemini(key):
        time.sleep(0.5)
        n_blocks = max(1, (total_sec + 14) // 15)
        per_block = total_sec // n_blocks
        anchor = ("A woman in a red wool coat and silver scarf with flat leather boots in the vaulted Grand Bazaar "
                  "at midday; warm lantern light and shafts of dusty sunlight; no written signs in the background.")
        shots = []
        for b in range(1, n_blocks + 1):
            d1 = round(per_block * 0.5, 1)
            d2 = round(per_block - d1, 1)
            shots += [
                {"block": b, "shot": 1, "start_sec": 0.0, "duration": d1, "framing": "Wide full-body shot, 28mm",
                 "action": f"Block {b} opening: she walks forward through the bazaar aisle.",
                 "camera": "Smooth backward steadicam tracking", "sound": "Ambient bazaar murmur"},
                {"block": b, "shot": 2, "start_sec": d1, "duration": d2, "framing": "Medium shot, 50mm",
                 "action": f"Block {b} close: she pauses by copper lanterns and holds a calm gaze for 0.5s.",
                 "camera": "Gentle push-in settling to a still hold", "sound": "Soft footsteps and copper clink"},
            ]
        return anchor, prompt_writer.shots_to_rows(shots), f"✅ Test modu: {n_blocks} blok ({total_sec} sn) planlandı."
    notes = []
    try:
        plan, problems = prompt_writer.plan_shots(idea, total_sec, key, model, images=images, style=style,
                                                  roles=_roles(ref_roles, refs),
                                                  audio_mode=audio_mode, log=notes.append)
    except Exception as e:
        return gr.update(), gr.update(), f"❌ Gemini planlama hatası: {e}"
    rows = prompt_writer.shots_to_rows(plan.get("shots") or [])
    anchor = plan.get("world_anchor") or ""
    blocks = prompt_writer.blocks_from_plan(plan)
    if problems:
        return anchor, rows, "⚠️ Planda uyarı var (tabloda düzeltebilirsiniz):\n- " + "\n- ".join(problems)
    return anchor, rows, (f"✅ {len(blocks)} blok · {len(rows)} çekim ({total_sec} sn) planlandı ({_used_model(notes, model)}). "
                          "Tabloyu inceleyin veya düzenleyin, ardından '2 · Blokların H3 Prompt'larını Hazırla' düğmesine basın."
                          + "".join("\n\n" + n for n in notes))


def do_rewrite_blocks(idea, world_anchor, shot_rows, audio_mode, model, typed_key, refs, style, picks, styled,
                      ref_roles=None):
    """Düzenlenmiş dünya çapası + çekim tablosu → her blok için H3 prompt'u."""
    shots = prompt_writer.rows_to_shots(shot_rows)
    if not shots:
        return gr.update(), gr.update(), "Önce '1 · Blok ve Çekim Tablosu Planla' ile çekim tablosunu oluşturun."
    key, model, err = _ready(typed_key, model)
    if err:
        return gr.update(), gr.update(), err
    images = _refs(refs, style, picks, styled)
    plan = {"world_anchor": (world_anchor or "").strip(), "shots": shots}
    blocks = prompt_writer.blocks_from_plan(plan)
    if _fake_gemini(key):
        time.sleep(0.5)
        fake_out = []
        for b in blocks:
            p = prompt_writer.EXAMPLE_OUTPUT
            if audio_mode == "music":
                p = re.sub(r"<d>.*?</d>", "", p)
                p = p.replace("overall_soundscape:", f"overall_soundscape: {prompt_writer._MUSIC_SILENT};")
            fake_out.append({"block": b["block"], "seconds": b["seconds"], "prompt": p, "problems": []})
        choices = [b["block"] for b in blocks]
        return (_format_block_prompts(fake_out), gr.update(choices=choices, value=choices[0]),
                f"✅ Test modu: {len(blocks)} blok için H3 prompt'u hazır.")
    notes = []
    try:
        out = prompt_writer.rewrite_blocks(plan, key, model, images=images, style=style, audio_mode=audio_mode,
                                           roles=_roles(ref_roles, refs),
                                           idea=idea or "", log=notes.append)
    except Exception as e:
        return gr.update(), gr.update(), f"❌ Gemini blok prompt hatası: {e}"
    all_probs = [f"Blok {b['block']}: {' | '.join(b['problems'])}" for b in out if b.get("problems")]
    choices = [b["block"] for b in out]
    text = _format_block_prompts(out)
    if all_probs:
        return (text, gr.update(choices=choices, value=choices[0]),
                "⚠️ Bazı blok prompt'larında uyarı var:\n- " + "\n- ".join(all_probs))
    return (text, gr.update(choices=choices, value=choices[0]),
            f"✅ {len(out)} blok için H3 prompt'ları hazır ({_used_model(notes, model)}). İsterseniz düzenleyip üretime geçin."
            + "".join("\n\n" + n for n in notes))


def _assemble_method(bstate, method, audio_mode, music_file, keep_ambient, say):
    """Belirli bir yöntem ('turbo', 'pika' veya 'h3') için tamamlanmış blok videolarını kare-tam birleştirir."""
    blocks_dict = bstate.get("blocks") or {}
    ordered = [blocks_dict[k] for k in sorted(blocks_dict) if blocks_dict[k].get(method)]
    if not ordered:
        return None
    clips = [b[method] for b in ordered]
    stamp = bstate.get("stamp") or f"{datetime.datetime.now():%Y%m%d_%H%M%S}"
    dest = os.path.join(config.OUTPUT_DIR, f"{stamp}_{config.METHODS[method]['file']}_birlesik.mp4")
    music_path = music_file if isinstance(music_file, str) else getattr(music_file, "name", None)
    res = assemble.assemble(clips, dest, audio_mode=audio_mode, music_path=music_path, keep_ambient=bool(keep_ambient))
    lufs_str = f", ses: {res['lufs']:.1f} LUFS".replace(".", ",") if res.get("lufs") is not None else ""
    trims = res.get("kirpilan_son_kareler") or []
    trim_str = f", sonda kırpılan kareler: {trims}" if any(trims) else ""
    say(f"🎬 Birleştirildi ({config.METHODS[method]['file']}): {os.path.basename(dest)} "
        f"({res['blok_sayisi']} blok, {res['saniye']:.2f} sn, {res['kare']} kare{lufs_str}{trim_str})")
    bstate.setdefault("finals", {})[method] = dest
    info_path = os.path.join(config.OUTPUT_DIR, f"{stamp}_bilgi.json")
    _write_json(info_path, bstate)
    _sync_to_drive([dest, info_path])
    return dest


def _generate_long_job(block_prompts_text, world_anchor, shot_rows, audio_mode, music_file, keep_ambient,
                  methods, resolution, seed, idea, refs, style=config.DEFAULT_STYLE, picks=None, styled=None,
                  ref_roles=None):
    """Çok çekimli uzun video üretici: her bloğu sırayla üretir, anında diske/Drive'a yazar ve birleştirir."""
    log, finals, files, sheets, bvideos = [], {m: None for m in config.METHODS}, [], [], []
    bstate = {}

    def say(msg):
        log.append(f"[{datetime.datetime.now():%H:%M:%S}] {msg}")

    def out():
        v_left, v_right = _pair_videos(finals)
        return "\n".join(log), v_left, v_right, bvideos or None, files or None, sheets or None, bstate

    shots = prompt_writer.rows_to_shots(shot_rows)
    fallback = prompt_writer.blocks_from_plan({"world_anchor": world_anchor, "shots": shots}) if shots else None
    blocks = _parse_block_prompts(block_prompts_text, fallback_blocks=fallback)
    if not blocks:
        say("❌ Önce blok H3 prompt'larını oluşturun.")
        yield out(); return
    if not methods:
        say("❌ En az bir yöntem seçin.")
        yield out(); return
    if audio_mode == "music" and not music_file:
        say("⚠️ Müzikli mod seçildi ancak müzik dosyası yüklenmedi; video ortam sesiyle/sessiz birleştirilecek.")
    if not _lock.acquire(blocking=False):
        say("❌ Başka bir üretim sürüyor; bitmesini bekleyin.")
        yield out(); return
    try:
        width, height = config.RESOLUTIONS[resolution]
        stamp = f"{datetime.datetime.now():%Y%m%d_%H%M%S}"
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        images = _refs(refs, style, picks, styled)
        ref_names = []
        if FAKE:
            say(f"Test modu: {len(blocks)} blok üretimi başlıyor.")
        else:
            g = setup_env.gpu()
            if not g:
                raise RuntimeError("GPU yok. Çalışma zamanı → Çalışma zamanı türünü değiştir → G4 GPU.")
            say(f"GPU: {g[0]} ({g[1]} GB).")
            yield out()
            te = setup_env.text_encoder_role()
            yield from _heartbeat(lambda: setup_env.ensure_comfy(log=say), out)
            yield from _heartbeat(lambda: setup_env.ensure_models(setup_env.roles_for(methods, te, bool(images)),
                                                                  log=say), out)
            yield from _heartbeat(lambda: comfy_client.start(log=say), out)
            for i, src in enumerate(images):
                name = f"h3app_{stamp}_{i + 1}{os.path.splitext(src)[1].lower() or '.png'}"
                os.makedirs(os.path.join(config.COMFY_DIR, "input"), exist_ok=True)
                shutil.copyfile(src, os.path.join(config.COMFY_DIR, "input", name))
                ref_names.append(name)

        info_file = os.path.join(config.OUTPUT_DIR, f"{stamp}_bilgi.json")
        bstate = {
            "uzun_video": True, "stamp": stamp, "resolution": resolution, "style": style, "audio_mode": audio_mode,
            "fikir": idea, "world_anchor": world_anchor or "", "shots": shots,
            "gemini_modelleri": dict(prompt_writer.used_models),
            "ref_names": ref_names, "images": images, "roller": _roles(ref_roles, refs),
            "methods": list(methods), "blocks": {}, "finals": {},
        }
        _write_json(info_file, bstate)
        files.append(info_file)
        _sync_to_drive(info_file)

        active_methods = [m for m in config.METHODS if m in methods]
        for b in blocks:
            bnum, bsec, bprompt = b["block"], b["seconds"], b["prompt"]
            frames = config.frames_for(bsec)
            bseed = int(seed) + bnum - 1
            bstate["blocks"][bnum] = {"seconds": bsec, "prompt": bprompt, "seed": bseed}
            pfile = os.path.join(config.OUTPUT_DIR, f"{stamp}_blok{bnum}_prompt.txt")
            open(pfile, "w", encoding="utf-8").write(bprompt + "\n")
            files.append(pfile)
            _write_json(info_file, bstate)
            _sync_to_drive([pfile, info_file])
            for method in active_methods:
                name = f"{stamp}_blok{bnum}_{config.METHODS[method]['file']}_seed{bseed}.mp4"
                dest = os.path.join(config.OUTPUT_DIR, name)
                say(f"▶ Blok {bnum}/{len(blocks)} · {config.METHODS[method]['label']} ({bsec} sn, {frames} kare, seed {bseed})")
                yield out()
                t0 = time.time()
                if FAKE:
                    time.sleep(0.3)
                    shutil.copyfile(os.environ["H3APP_FAKE_VIDEO"], dest)
                else:
                    te = setup_env.text_encoder_role()
                    grf = graphs.build(method, bprompt, bseed, width, height, frames, te,
                                       prefix=f"h3app/{stamp}_b{bnum}_{method}", ref_images=ref_names)
                    shown = 0
                    for sec in comfy_client.run(grf, dest):
                        if sec // 60 > shown:
                            shown = sec // 60
                            say(f"   … {shown} dk geçti")
                        yield out()
                took = int(time.time() - t0)
                qa.normalize_audio(dest)
                bstate["blocks"][bnum][method] = dest
                files.append(dest)
                say(f"✅ Blok {bnum} bitti: {name} ({took} sn)")
                to_sync = [dest]
                try:
                    stem = dest[:-4]
                    ref_v = (bstate["blocks"][bnum].get("pika") or bstate["blocks"][bnum].get("turbo")) if method == "h3" else None
                    r = qa.check(dest, bprompt, sheet=stem + "_kontrol.jpg", crops=stem + "_kirpinti.jpg",
                                 style=style, ref_video=ref_v)
                    say("   " + qa.summary(r))
                    sheets += [(_to_pil(r["temas"]), f"Blok {bnum} · {config.METHODS[method]['file']} · anlar"),
                               (_to_pil(r["kirpinti"]), f"Blok {bnum} · {config.METHODS[method]['file']} · %100 kırpıntı")]
                    files += [r["temas"], r["kirpinti"]]
                    to_sync += [r["temas"], r["kirpinti"]]
                except Exception as e:
                    say(f"   Kontrol yapılamadı: {e}")
                _write_json(info_file, bstate)
                to_sync.append(info_file)
                _sync_to_drive(to_sync, say=say)
                bvideos = [(bstate["blocks"][k].get(active_methods[0]), f"Blok {k}")
                           for k in sorted(bstate["blocks"]) if bstate["blocks"][k].get(active_methods[0])]
                yield out()

        for method in active_methods:
            merged = _assemble_method(bstate, method, audio_mode, music_file, keep_ambient, say)
            if merged:
                finals[method] = merged
                files.append(merged)
        yield out()
    except Exception as e:
        say(f"❌ Hata: {e}")
    finally:
        _lock.release()
    yield out()


def _reroll_job(block_num, new_seed, method, block_prompts_text, audio_mode, music_file, keep_ambient, bstate):
    """Yalnızca seçilen bloğu yeni seed ile yeniden üretir ve diğer bloklarla anında yeniden birleştirir."""
    log, finals, files, sheets, bvideos = [], {m: None for m in config.METHODS}, [], [], []

    def say(msg):
        log.append(f"[{datetime.datetime.now():%H:%M:%S}] {msg}")

    def out():
        v_left, v_right = _pair_videos(finals)
        return "\n".join(log), v_left, v_right, bvideos or None, files or None, sheets or None, bstate

    if not bstate or not bstate.get("blocks"):
        say("❌ Önce '3 · Tüm Blokları Üret ve Birleştir' ile blokları üretin.")
        yield out(); return
    bnum = int(block_num or 1)
    if bnum not in bstate["blocks"]:
        say(f"❌ Blok {bnum} mevcut üretimde bulunamadı.")
        yield out(); return
    if not _lock.acquire(blocking=False):
        say("❌ Başka bir üretim sürüyor; bitmesini bekleyin.")
        yield out(); return
    try:
        parsed = {b["block"]: b["prompt"] for b in _parse_block_prompts(block_prompts_text)}
        binfo = bstate["blocks"][bnum]
        if bnum in parsed and parsed[bnum].strip():
            binfo["prompt"] = parsed[bnum].strip()
        bsec = binfo["seconds"]
        bprompt = binfo["prompt"]
        bseed = int(new_seed)
        binfo["seed"] = bseed
        frames = config.frames_for(bsec)
        width, height = config.RESOLUTIONS[bstate["resolution"]]
        stamp = bstate["stamp"]
        info_file = os.path.join(config.OUTPUT_DIR, f"{stamp}_bilgi.json")
        name = f"{stamp}_blok{bnum}_{config.METHODS[method]['file']}_seed{bseed}.mp4"
        dest = os.path.join(config.OUTPUT_DIR, name)
        say(f"🔄 Blok {bnum} yeniden üretiliyor ({config.METHODS[method]['label']}, seed {bseed})…")
        yield out()
        if FAKE:
            time.sleep(0.3)
            shutil.copyfile(os.environ["H3APP_FAKE_VIDEO"], dest)
        else:
            te = setup_env.text_encoder_role()
            yield from _heartbeat(lambda: comfy_client.start(log=say), out)
            grf = graphs.build(method, bprompt, bseed, width, height, frames, te,
                               prefix=f"h3app/{stamp}_b{bnum}_{method}_s{bseed}",
                               ref_images=bstate.get("ref_names") or [])
            for _ in comfy_client.run(grf, dest):
                yield out()
        qa.normalize_audio(dest)
        binfo[method] = dest
        files.append(dest)
        say(f"✅ Blok {bnum} yenilendi: {name}")
        to_sync = [dest]
        try:
            stem = dest[:-4]
            ref_v = (binfo.get("pika") or binfo.get("turbo")) if method == "h3" else None
            r = qa.check(dest, bprompt, sheet=stem + "_kontrol.jpg", crops=stem + "_kirpinti.jpg",
                         style=bstate.get("style"), ref_video=ref_v)
            say("   " + qa.summary(r))
            sheets += [(_to_pil(r["temas"]), f"Blok {bnum} (seed {bseed}) · anlar"),
                       (_to_pil(r["kirpinti"]), f"Blok {bnum} (seed {bseed}) · %100 kırpıntı")]
            files += [r["temas"], r["kirpinti"]]
            to_sync += [r["temas"], r["kirpinti"]]
        except Exception as e:
            say(f"   Kontrol yapılamadı: {e}")
        _write_json(info_file, bstate)
        to_sync.append(info_file)
        _sync_to_drive(to_sync, say=say)

        for m in config.METHODS:
            if any(bstate["blocks"][k].get(m) for k in bstate["blocks"]):
                merged = _assemble_method(bstate, m, audio_mode, music_file, keep_ambient, say)
                if merged:
                    finals[m] = merged
                    files.append(merged)
        primary = method if any(bstate["blocks"][k].get(method) for k in bstate["blocks"]) else next(
            (m for m in config.METHODS if any(bstate["blocks"][k].get(m) for k in bstate["blocks"])), "pika"
        )
        bvideos = [(bstate["blocks"][k].get(primary), f"Blok {k}")
                   for k in sorted(bstate["blocks"]) if bstate["blocks"][k].get(primary)]
    except Exception as e:
        say(f"❌ Hata: {e}")
    finally:
        _lock.release()
    yield out()


def reassemble_blocks(audio_mode, music_file, keep_ambient, bstate):
    """Üretilmiş mevcut blokları (ör. yeni müzik veya ses moduyla) yeniden üretmeden 1-2 saniyede birleştirir."""
    log, finals, files = [], {m: None for m in config.METHODS}, []

    def say(msg):
        log.append(f"[{datetime.datetime.now():%H:%M:%S}] {msg}")

    if not bstate or not bstate.get("blocks"):
        say("❌ Birleştirilecek hazır blok bulunamadı.")
        return "\n".join(log), None, None, None
    try:
        for m in config.METHODS:
            if any(bstate["blocks"][k].get(m) for k in bstate["blocks"]):
                merged = _assemble_method(bstate, m, audio_mode, music_file, keep_ambient, say)
                if merged:
                    finals[m] = merged
                    files.append(merged)
    except Exception as e:
        say(f"❌ Birleştirme hatası: {e}")
    v_left, v_right = _pair_videos(finals)
    return "\n".join(log), v_left, v_right, files or None


def _generate_job(h3_prompt, methods, resolution, seconds, seed, idea, brief, refs, style=config.DEFAULT_STYLE,
             picks=None, styled=None, ref_roles=None, review_on=True, review_retries=config.REVIEW_RETRIES,
             typed_key=None):
    """Generator: (günlük, taslak video, final video, indirilecek dosyalar, kontrol görselleri).
    review_on: her video Gemini ile izlenir; yüksek önemde kusur varsa yeni seed ile en fazla `review_retries`
    kez yeniden üretilir ve en az kusurlu aday seçilir."""
    log, videos, files, sheets = [], {m: None for m in config.METHODS}, [], []

    def say(msg):
        log.append(f"[{datetime.datetime.now():%H:%M:%S}] {msg}")

    def out():
        v_left, v_right = _pair_videos(videos)
        return "\n".join(log), v_left, v_right, files or None, sheets or None

    if not (h3_prompt or "").strip():
        say("❌ Önce H3 prompt'unu oluşturun ya da yazın.")
        yield out(); return
    if not methods:
        say("❌ En az bir yöntem seçin.")
        yield out(); return
    request = (brief or "").strip() or (idea or "").strip()
    images = _refs(refs, style, picks, styled)
    problems = prompt_writer.check(h3_prompt, request, len(images), style, roles=_roles(ref_roles, refs))
    if problems:
        say("⚠️ Prompt uyarısı (yine de devam ediliyor): " + " | ".join(problems))
    if not _lock.acquire(blocking=False):
        say("❌ Başka bir üretim sürüyor; bitmesini bekleyin.")
        yield out(); return
    try:
        width, height = config.RESOLUTIONS[resolution]
        frames = config.frames_for(seconds)
        stamp = f"{datetime.datetime.now():%Y%m%d_%H%M%S}"
        base = os.path.join(config.OUTPUT_DIR, stamp)
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        prompt_file = base + "_prompt.txt"
        info_file = base + "_bilgi.json"
        open(prompt_file, "w", encoding="utf-8").write(h3_prompt.strip() + "\n")
        info = {"stil": config.STYLES[style]["label"], "fikir": idea, "brief": brief,
                "gemini_modelleri": dict(prompt_writer.used_models),
                "roller": _roles(ref_roles, refs),
                "referanslar": [os.path.basename(p) for p in images],
                "yontemler": methods, "seed": int(seed), "boyut": [width, height],
                "kare": frames, "saniye": round(frames / 24, 2), "videolar": {}}
        ref_names = []
        if FAKE:
            say("Test modu: kurulum atlandı.")
        else:
            g = setup_env.gpu()
            if not g:
                raise RuntimeError("GPU yok. Çalışma zamanı → Çalışma zamanı türünü değiştir → G4 GPU.")
            say(f"GPU: {g[0]} ({g[1]} GB).")
            yield out()
            te = setup_env.text_encoder_role()
            yield from _heartbeat(lambda: setup_env.ensure_comfy(log=say), out)
            yield from _heartbeat(lambda: setup_env.ensure_models(setup_env.roles_for(methods, te, bool(images)),
                                                                  log=say), out)
            yield from _heartbeat(lambda: comfy_client.start(log=say), out)
            for i, src in enumerate(images):
                name = f"h3app_{stamp}_{i + 1}{os.path.splitext(src)[1].lower() or '.png'}"
                os.makedirs(os.path.join(config.COMFY_DIR, "input"), exist_ok=True)
                shutil.copyfile(src, os.path.join(config.COMFY_DIR, "input", name))
                ref_names.append(name)
            if images:
                say(f"Referanslı mod (Ref2VA): {len(images)} görsel <Picture 1…{len(images)}> olarak verildi — "
                    f"{roles_summary(ref_roles, refs)}.")
        kept_refs = []
        for i, src in enumerate(images):   # referans görseller de sonuçla birlikte saklanır
            kept = f"{base}_referans{i + 1}{os.path.splitext(src)[1].lower() or '.png'}"
            shutil.copyfile(src, kept)
            info["referanslar"][i] = os.path.basename(kept)
            kept_refs.append(kept)
        # Kesinti koruması: üretim başlamadan önce prompt, bilgi ve referansları diske ve Drive'a yaz
        _write_json(info_file, info)
        files += [prompt_file, info_file]
        _sync_to_drive([prompt_file, info_file] + kept_refs)
        yield out()
        rv_key = _key(typed_key)
        rv_on = bool(review_on) and bool(rv_key) and not _fake_gemini(rv_key)
        if review_on and not rv_on:
            say("ℹ️ Otomatik izleme atlandı: " + ("Gemini anahtarı yok." if not rv_key else "test modu."))
        retries = max(0, min(5, int(review_retries or 0))) if rv_on else 0
        roles = _roles(ref_roles, refs)

        def render(method, s_used):
            name = f"{stamp}_{config.METHODS[method]['file']}_seed{s_used}.mp4"
            dest = os.path.join(config.OUTPUT_DIR, name)
            say(f"▶ {config.METHODS[method]['label']} başladı ({width}×{height}, {frames} kare, seed {s_used}).")
            yield out()
            t0 = time.time()
            if FAKE:
                for _ in range(3):
                    time.sleep(1)
                    yield out()
                shutil.copyfile(os.environ["H3APP_FAKE_VIDEO"], dest)
            else:
                te = setup_env.text_encoder_role()
                g = graphs.build(method, h3_prompt.strip(), s_used, width, height, frames, te,
                                 prefix=f"h3app/{stamp}_{method}", ref_images=ref_names)
                shown = 0
                for sec in comfy_client.run(g, dest):
                    if sec // 60 > shown:
                        shown = sec // 60
                        say(f"   … {shown} dk geçti")
                    yield out()
            took = int(time.time() - t0)
            norm = qa.normalize_audio(dest)
            if norm:
                say(f"   🔊 Ses dengelendi: {norm[0]:.1f} → {norm[1]:.1f} LUFS".replace(".", ","))
            say(f"✅ Bitti: {name} ({took} sn, {os.path.getsize(dest) / 1e6:.1f} MB)")
            return dest, name, took

        for method in [m for m in config.METHODS if m in methods]:
            cands = []
            for attempt in range(retries + 1):
                s_used = int(seed) + 1000 * attempt
                dest, name, took = yield from render(method, s_used)
                files.append(dest)
                cand = {"dest": dest, "dosya": name, "seed": s_used, "sure_sn": took, "kontrol": None,
                        "inceleme": None}
                to_sync = [dest]
                try:   # GPU'suz kontrol: kesmeler prompt'la karşılaştırılır, temas sayfası ve kırpıntılar çıkarılır
                    stem = dest[:-4]
                    ref_v = (videos.get("pika") or videos.get("turbo")) if method == "h3" else None
                    r = qa.check(dest, h3_prompt, sheet=stem + "_kontrol.jpg", crops=stem + "_kirpinti.jpg",
                                 style=style, ref_video=ref_v)
                    say("   " + qa.summary(r))
                    cand["kontrol"] = {k: r[k] for k in ("kesmeler", "beklenen", "bulunamayan", "beklenmeyen",
                                                         "donmus", "lufs", "ses", "son_kayma", "stil_kontrol")}
                    tag = f"{config.METHODS[method]['file']} · seed {s_used}"
                    sheets += [(_to_pil(r["temas"]), f"{tag} · anlar"), (_to_pil(r["kirpinti"]), f"{tag} · %100 kırpıntı")]
                    files += [r["temas"], r["kirpinti"]]
                    to_sync += [r["temas"], r["kirpinti"]]
                except Exception as e:
                    say(f"   Kontrol yapılamadı: {e}")
                if rv_on:
                    say("   🔍 Gemini videoyu izliyor…")
                    yield out()
                    res = yield from _heartbeat(lambda: review.review_video(dest, h3_prompt, rv_key, images, roles,
                                                                          log=say), out)
                    cand["inceleme"] = res
                    say("   " + review.summary(res))
                cands.append(cand)
                # Her aday biter bitmez bilgi.json ve dosyalar diske ve Drive'a (kesinti koruması)
                info["videolar"][method] = {"adaylar": [{k: c[k] for k in ("dosya", "seed", "sure_sn", "inceleme")}
                                                        for c in cands]}
                _write_json(info_file, info)
                to_sync += [prompt_file, info_file]
                _sync_to_drive(to_sync, say=say)
                yield out()
                if not review.blocking(cand["inceleme"]):
                    break
                if attempt < retries:
                    say(f"🔁 Yüksek önemde kusur var; yeni seed ({int(seed) + 1000 * (attempt + 1)}) ile yeniden "
                        f"üretiliyor ({attempt + 1}/{retries}).")
                    yield out()
            best = min(cands, key=lambda c: review.score(c["inceleme"])) if rv_on else cands[-1]
            videos[method] = best["dest"]
            info["videolar"][method].update({k: best[k] for k in ("dosya", "seed", "sure_sn", "kontrol", "inceleme")})
            if rv_on:
                say(f"⭐ Seçilen: {best['dosya']} (seed {best['seed']}, {len(cands)} aday) — "
                    f"izleme: {review.short(best['inceleme'])}")
            if review.blocking(best["inceleme"]):
                say("⚠️ Denemelerin hepsinde yüksek önemde kusur kaldı; en az kusurlu aday seçildi. "
                    "Prompt'u sadeleştirip tekrar deneyin.")
            _write_json(info_file, info)
            _sync_to_drive([info_file], say=say)
            yield out()
        say(f"Tamamlandı. Dosyalar: {config.OUTPUT_DIR}")
    except Exception as e:
        say(f"❌ Hata: {e}")
    finally:
        _lock.release()
    yield out()


COMFY_OUT_RE = re.compile(r"^(\d{8}_\d{6})_(?:b(\d+)_)?(turbo|pika|h3)(?:_s(\d+))?_\d+_?\.mp4$")


def _recover_comfy_outputs(stamp, info):
    """ComfyUI'nin bitirip uygulamanın kopyalayamadığı videoları (ör. Python çekirdeği yeniden başladı)
    ComfyUI/output/h3app altından OUTPUT_DIR'e uygulamanın adlandırmasıyla alır. Alınan dosya adlarını döner."""
    src_dir = os.path.join(config.COMFY_DIR, "output", "h3app")
    if not os.path.isdir(src_dir):
        return []
    found = {}
    for fn in sorted(os.listdir(src_dir)):   # aynı önekte birden fazla sayaç varsa sonuncusu kalır
        m = COMFY_OUT_RE.match(fn)
        if not m or m.group(1) != stamp:
            continue
        block, method, rseed = m.group(2), m.group(3), m.group(4)
        mfile = config.METHODS[method]["file"]
        if block:
            binfo = (info.get("blocks") or {}).get(block) or (info.get("blocks") or {}).get(int(block)) or {}
            seed = rseed or binfo.get("seed")
            if seed is None:
                continue
            name = f"{stamp}_blok{block}_{mfile}_seed{seed}.mp4"
        else:
            seed = info.get("seed")
            if seed is None:
                continue
            name = f"{stamp}_{mfile}_seed{seed}.mp4"
        found[name] = os.path.join(src_dir, fn)
    recovered = []
    for name, src in found.items():
        dest = os.path.join(config.OUTPUT_DIR, name)
        if not os.path.isfile(dest):
            shutil.copyfile(src, dest)
            recovered.append(name)
    if recovered:
        _sync_to_drive([os.path.join(config.OUTPUT_DIR, n) for n in recovered])
    return recovered


def load_last_outputs():
    """Tarayıcı bağlantısı kopup yeniden bağlanıldığında OUTPUT_DIR içindeki son tamamlanmış üretimleri arayüze geri yükler."""
    noop = (gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update())
    if not os.path.isdir(config.OUTPUT_DIR):
        return "📂 Henüz kaydedilmiş bir üretim klasörü bulunamadı.", None, None, None, None, *noop
    entries = sorted(os.listdir(config.OUTPUT_DIR))
    stamps = sorted({m.group(1) for name in entries if (m := STAMP_RE.match(name))}, reverse=True)
    if not stamps:
        return "📂 Henüz tamamlanmış bir üretim bulunamadı.", None, None, None, None, *noop

    stamp = stamps[0]
    prefix = f"{stamp}_"
    info_path = os.path.join(config.OUTPUT_DIR, f"{stamp}_bilgi.json")
    info = {}
    if os.path.isfile(info_path):
        try:
            info = json.load(open(info_path, encoding="utf-8"))
        except Exception:
            info = {}
    recovered = []
    if not job_running():
        try:
            recovered = _recover_comfy_outputs(stamp, info)
        except Exception:
            recovered = []
        if recovered:
            entries = sorted(os.listdir(config.OUTPUT_DIR))
    session_files = [os.path.join(config.OUTPUT_DIR, f) for f in entries
                     if f.startswith(prefix) and os.path.isfile(os.path.join(config.OUTPUT_DIR, f))]

    videos = {m: None for m in config.METHODS}
    sheets = []
    log = [f"📂 Son üretim diskten geri yüklendi (oturum: {stamp})."]
    if recovered:
        log.append("   ♻️ ComfyUI çıktısından kurtarıldı: " + ", ".join(recovered))

    is_long = bool(info.get("uzun_video")) or any(f"_{ 'blok' }" in os.path.basename(p) for p in session_files)
    if is_long:
        raw_blocks = info.get("blocks") or {}
        blocks = {int(k): v for k, v in raw_blocks.items()}
        for p in session_files:
            bn = os.path.basename(p)
            m_blk = re.match(rf"^{re.escape(stamp)}_blok(\d+)_(.+)_seed(\d+)\.mp4$", bn)
            if m_blk:
                bnum, mfile, bseed = int(m_blk.group(1)), m_blk.group(2), int(m_blk.group(3))
                mkey = next((k for k, v in config.METHODS.items() if v["file"] == mfile), None)
                if mkey:
                    blocks.setdefault(bnum, {"seconds": 15, "prompt": "", "seed": bseed})[mkey] = p
        for bnum in sorted(blocks):
            pfile = os.path.join(config.OUTPUT_DIR, f"{stamp}_blok{bnum}_prompt.txt")
            if not blocks[bnum].get("prompt") and os.path.isfile(pfile):
                blocks[bnum]["prompt"] = open(pfile, encoding="utf-8").read().strip()

        for mkey, mmeta in config.METHODS.items():
            merged = os.path.join(config.OUTPUT_DIR, f"{stamp}_{mmeta['file']}_birlesik.mp4")
            if os.path.isfile(merged):
                videos[mkey] = merged
                log.append(f"   🎬 Birleşik video: {os.path.basename(merged)}")
            elif blocks:
                first_blk = next((blocks[k].get(mkey) for k in sorted(blocks) if blocks[k].get(mkey)), None)
                if first_blk:
                    videos[mkey] = first_blk

        for p in session_files:
            bn = os.path.basename(p)
            if bn.endswith("_kontrol.jpg"):
                sheets.append((_to_pil(p), bn[len(prefix):-len("_kontrol.jpg")] + " · anlar"))
            elif bn.endswith("_kirpinti.jpg"):
                sheets.append((_to_pil(p), bn[len(prefix):-len("_kirpinti.jpg")] + " · %100 kırpıntı"))

        bstate = dict(info) if info else {"stamp": stamp}
        bstate["blocks"] = blocks
        bprompts_txt = _format_block_prompts([
            {"block": k, "seconds": blocks[k].get("seconds", 15), "prompt": blocks[k].get("prompt", "")}
            for k in sorted(blocks)
        ]) if blocks else gr.update()
        anchor_upd = info.get("world_anchor") or gr.update()
        shots_upd = prompt_writer.shots_to_rows(info["shots"]) if info.get("shots") else gr.update()
        primary = next((m for m in config.METHODS if any(blocks[k].get(m) for k in blocks)), "pika")
        bvideos = [(blocks[k].get(primary), f"Blok {k}") for k in sorted(blocks) if blocks[k].get(primary)]
        log.append(f"   ✅ {len(blocks)} blok ve {len(session_files)} dosya yüklendi.")
        v_left, v_right = _pair_videos(videos)
        return ("\n".join(log), v_left, v_right, session_files or None, sheets or None,
                gr.update(), bprompts_txt, anchor_upd, shots_upd, bvideos or None, bstate)

    # Tekil video oturumu
    prompt_file = os.path.join(config.OUTPUT_DIR, f"{stamp}_prompt.txt")
    prompt_upd = open(prompt_file, encoding="utf-8").read().strip() if os.path.isfile(prompt_file) else gr.update()
    for mkey, mmeta in config.METHODS.items():
        chosen = ((info.get("videolar") or {}).get(mkey) or {}).get("dosya")
        cands = [p for p in session_files if os.path.basename(p).startswith(f"{stamp}_{mmeta['file']}_seed")
                 and p.endswith(".mp4")]
        pick = next((p for p in cands if os.path.basename(p) == chosen), cands[-1] if cands else None)
        if pick:
            videos[mkey] = pick
            note = f" ({len(cands)} aday, seçilen)" if len(cands) > 1 else ""
            log.append(f"   🎬 Video ({mmeta['label']}): {os.path.basename(pick)}{note}")
    for p in session_files:
        bn = os.path.basename(p)
        if bn.endswith("_kontrol.jpg"):
            sheets.append((_to_pil(p), bn[len(prefix):-len("_kontrol.jpg")] + " · anlar"))
        elif bn.endswith("_kirpinti.jpg"):
            sheets.append((_to_pil(p), bn[len(prefix):-len("_kirpinti.jpg")] + " · %100 kırpıntı"))
    log.append(f"   ✅ Toplam {len(session_files)} dosya geri yüklendi ({config.OUTPUT_DIR}).")
    v_left, v_right = _pair_videos(videos)
    return ("\n".join(log), v_left, v_right, session_files or None, sheets or None,
            prompt_upd, gr.update(), gr.update(), gr.update(), gr.update(), gr.update())


# ---------------------------------------------------------------------------------------------------------
# Arka plan işi: üretim Gradio isteğinden bağımsız bir iş parçacığında koşar. Tarayıcı sekmesi kapanır ya da
# Colab bağlantısı koparsa Gradio kendi generator'ını durdurur; iş parçacığı ise ComfyUI'yi beklemeye, videoyu
# kopyalamaya, kontrol etmeye, birleştirmeye ve Drive'a yazmaya devam eder. Yeniden bağlanınca
# "Son Üretimleri Yükle" süren işi izler, bitince sonuçları diskten yükler.
_DONE = object()
_job = {"name": None, "thread": None, "last": None}


def job_running():
    t = _job.get("thread")
    return bool(t and t.is_alive())


def _freeze(snap):
    """İş parçacığı listeleri/sözlükleri değiştirmeye devam ederken arayüze o anın kopyası gider."""
    out = []
    for x in snap:
        if isinstance(x, list):
            x = list(x)
        elif isinstance(x, dict):
            x = copy.deepcopy(x)
        out.append(x)
    return tuple(out)


HEARTBEAT_SECONDS = 1.5   # Colab'in colab.dev proxy'si akışı tamponluyor: sık güncelleme tamponu boşaltır


def _heartbeat(fn, out, every=2.0):
    """Uzun ve sessiz bir adımı (kurulum, model indirme) ayrı iş parçacığında çalıştırır; bu sırada `every` sn'de
    bir güncel günlüğü verir (generator). Adımın hatası aynen yükseltilir."""
    box = {}

    def run():
        try:
            box["value"] = fn()
        except BaseException as e:   # noqa: BLE001 - hata çağırana aynen taşınır
            box["error"] = e

    t = threading.Thread(target=run, daemon=True, name="h3app-step")
    t.start()
    while t.is_alive():
        t.join(timeout=every)
        yield out()
    if "error" in box:
        raise box["error"]
    return box.get("value")


def _in_background(job_fn, name):
    def runner(*args, **kwargs):
        q = queue.Queue()
        attached = {"on": True}
        owner = not job_running()   # süren bir iş varken yeni tıklama yalnızca "meşgul" mesajını üretir

        def work():
            try:
                for snap in job_fn(*args, **kwargs):
                    snap = _freeze(snap)
                    if owner:
                        _job["last"] = snap
                    if attached["on"]:
                        q.put(snap)
            except Exception as e:   # iş fonksiyonları hatalarını kendileri yakalar; bu son güvence
                print(f"h3app arka plan işi hatası ({name}): {e}")
            finally:
                q.put(_DONE)

        t = threading.Thread(target=work, name=f"h3app-{name}", daemon=True)
        if owner:
            _job.update(name=name, thread=t, last=None)
        t.start()
        last = None
        try:
            while True:
                try:
                    snap = q.get(timeout=HEARTBEAT_SECONDS)
                except queue.Empty:
                    if last is not None:
                        yield last   # yeni bir şey yok: son durumu tekrarla (proxy tamponu dolsun, ekrana düşsün)
                    continue
                if snap is _DONE:
                    return
                last = snap
                yield snap
        finally:
            attached["on"] = False   # sekme kapandı: iş sürer, yalnızca arayüze akış durur

    runner.__name__ = job_fn.__name__.strip("_").replace("_job", "")
    runner.__doc__ = job_fn.__doc__
    return runner


# ---------------------------------------------------------------------------------------------------------
# Hızlı üretim: tek düğmeyle fikir → brief → H3 prompt'u → video. Hem insanlar hem tarayıcı kullanan ajanlar
# (ör. kişisel asistan botlar) için: birkaç kutu, tek düğme ve tek satırlık açık bir durum yazısı.
QUICK_METHODS = [(config.METHODS[k]["label"], k) for k in ("pika", "turbo", "h3")]


def _quick_job(idea, seconds, style, method, resolution, seed, model, typed_key, refs=None, ref_roles=None,
               picks=None, styled=None, review_on=True, review_retries=config.REVIEW_RETRIES):
    """Generator: (durum, brief, H3 prompt'u, video, indirilecek dosyalar, ayrıntılı günlük)."""
    state = {"status": "", "brief": "", "prompt": "", "video": None, "files": None, "log": ""}

    def out(status=None):
        if status is not None:
            state["status"] = status
        return (state["status"], state["brief"], state["prompt"], state["video"], state["files"], state["log"])

    idea = (idea or "").strip()
    style = style or config.DEFAULT_STYLE
    method = method if method in config.METHODS else "pika"
    resolution = resolution if resolution in config.RESOLUTIONS else list(config.RESOLUTIONS)[0]
    seconds = int(min(config.MAX_SECONDS, max(config.MIN_SECONDS, int(seconds or 10))))
    if not idea:
        yield out("❌ Hata: önce fikri yazın.")
        return
    key, model, err = _ready(typed_key, model)
    if err:
        yield out(err)
        return
    images = _refs(refs, style, picks, styled)
    roles = _roles(ref_roles, refs)
    notes, weak = [], []   # weak: yedek modelle yazılan adımlar (durum satırında da gösterilir)
    try:
        prompt_writer.last_model.update(name=None, fallback=False)
        prompt_writer.used_models.clear()
        yield out("⏳ 1/3 Gemini detaylı brief'i yazıyor…")
        if _fake_gemini(key):
            state["brief"] = prompt_writer.BRIEF_EXAMPLES[1][1]
        else:
            state["brief"] = prompt_writer.expand(idea, seconds, key, model, images, log=notes.append, style=style,
                                                  roles=roles)
            if prompt_writer.last_model["fallback"]:
                weak.append(f"brief {prompt_writer.last_model['name']}")
        yield out("⏳ 2/3 Gemini H3 prompt'unu yazıyor…" + (f" (⚠️ {weak[0]} yedek modeliyle yazıldı)" if weak else ""))
        problems = []
        if _fake_gemini(key):
            state["prompt"] = prompt_writer.EXAMPLE_OUTPUT
        else:
            state["prompt"], problems = prompt_writer.rewrite(state["brief"], seconds, key, model, images,
                                                              log=notes.append, style=style, roles=roles)
            if prompt_writer.last_model["fallback"]:
                weak.append(f"prompt {prompt_writer.last_model['name']}")
        if not state["prompt"].strip():
            yield out("❌ Hata: Gemini prompt yazamadı. Biraz sonra tekrar deneyin.")
            return
    except Exception as e:
        state["log"] = "\n".join(notes)
        yield out(f"❌ Hata (Gemini): {e}")
        return
    pre = "\n".join(notes + ([f"⚠️ Prompt uyarısı: {' | '.join(problems)}"] if problems else []))
    label = config.METHODS[method]["label"].split(" (")[0]
    yield out(f"⏳ 3/3 Video üretiliyor ({label}, {seconds} sn)… İlk üretimde model indirme ~6 dk sürer.")
    last_log = ""
    for log, v_left, v_right, files, _sheets in _generate_job(state["prompt"], [method], resolution, seconds, seed,
                                                            idea, state["brief"], refs, style, picks, styled,
                                                            ref_roles, review_on, review_retries, typed_key):
        last_log = log or ""
        state["log"] = (pre + "\n" + last_log).strip()
        state["files"] = files
        state["video"] = v_left or v_right
        tail = [ln for ln in last_log.splitlines() if ln.strip()]
        step = tail[-1].split("] ", 1)[-1].strip() if tail else ""
        if state["video"]:
            yield out(f"⏳ 3/3 Video hazır, kontrol ediliyor… {step}")
        else:
            yield out(f"⏳ 3/3 Video üretiliyor ({label})… {step}")
    errors = [ln.split("] ", 1)[-1] for ln in last_log.splitlines() if "❌" in ln]
    if state["video"] and os.path.isfile(state["video"]):
        where = f" · Drive: {_state['drive_dir']}" if _state.get("drive_dir") else ""
        chosen = [ln for ln in last_log.splitlines() if "⭐ Seçilen:" in ln]
        rv = (" · " + chosen[-1].split(" — ", 1)[1]) if chosen else ""
        fb = f" · ⚠️ yedek modelle yazıldı: {', '.join(weak)}" if weak else ""
        yield out(f"✅ Hazır: {os.path.basename(state['video'])}{rv}{fb}{where}")
    else:
        yield out(errors[-1] if errors else "❌ Hata: video üretilemedi; ayrıntılı günlüğe bakın.")


generate = _in_background(_generate_job, "generate")
quick_generate = _in_background(_quick_job, "quick_generate")
generate_long = _in_background(_generate_long_job, "generate_long")
reroll_block = _in_background(_reroll_job, "reroll_block")


def load_or_follow(poll=3):
    """Süren bir üretim varsa günlüğünü canlı gösterir; iş bitince (ya da hiç yoksa) son üretimi diskten yükler."""
    keep = tuple(gr.update() for _ in range(10))
    while job_running():
        last = _job.get("last")
        log = last[0] if last else ""
        yield ("⏳ Arka planda süren üretim izleniyor (sekme kapansa da çalışmaya devam eder)…\n\n" + log,) + keep
        _job["thread"].join(timeout=poll)
    yield load_last_outputs()


class ColabProxyHost:
    """ASGI ara katmanı: Colab'in `*.colab.dev` proxy'si (kernel.proxyPort) isteği VM'in iç adresine
    (`http://…internal:8007`) iletir ve `x-forwarded-host` eklemez; Gradio da video/görsel adreslerini bu iç
    adresle üretir, tarayıcı onlara ulaşamaz. Tarayıcının kendi gönderdiği Origin/Referer başlığındaki dış adres
    `*.colab.dev` ise onu `x-forwarded-host` (https) olarak ekler. Başka hiçbir isteğe dokunmaz."""

    SUFFIX = ".colab.dev"

    def __init__(self, app):
        self.app = app

    @classmethod
    def external_host(cls, headers):
        h = {k.lower(): v for k, v in headers}
        if b"x-forwarded-host" in h:
            return None
        from urllib.parse import urlsplit
        for key in (b"origin", b"referer"):
            raw = h.get(key)
            if not raw:
                continue
            parts = urlsplit(raw.decode("latin-1"))
            if parts.scheme == "https" and (parts.hostname or "").endswith(cls.SUFFIX):
                return parts.netloc
        return None

    async def __call__(self, scope, receive, send):
        if scope.get("type") in ("http", "websocket"):
            host = self.external_host(scope.get("headers") or [])
            if host:
                scope = dict(scope)
                scope["headers"] = list(scope["headers"]) + [(b"x-forwarded-host", host.encode("latin-1")),
                                                              (b"x-forwarded-proto", b"https")]
        await self.app(scope, receive, send)


CSS = """
.h3-title h1 {margin-bottom: 0}
.h3-note {opacity: .8; font-size: .9em}
"""


def build():
    with gr.Blocks(title="H3 Video Stüdyosu") as demo:
        gr.Markdown("# H3 Video Stüdyosu\nKısa fikrinizi yazın → Gemini detaylı brief'e ve MiniMax-H3'ün resmi prompt yapısına "
                    "çevirsin → isterseniz bir karakter görseliyle MiniMax-H3 Turbo, Pika H3 Flex ve/veya MiniMax-H3 ile video üretin.", elem_classes="h3-title")
        status = gr.Markdown(_status_line())

        with gr.Group(elem_id="hizli-uretim"):
            gr.Markdown("## ⚡ Hızlı üretim (tek düğme)\nFikri yazın, süreyi, stili ve yöntemi seçin ve "
                        "**Fikirden videoya: hepsini yap** düğmesine basın: brief, H3 prompt'u ve video sırayla yapılır, "
                        "durum aşağıda tek satırda yazar. Karakter / arka plan görseli kullanmak için 1. bölüme yükleyip "
                        "rollerini seçin; hızlı üretim onları da kullanır.", elem_classes="h3-note")
            q_idea = gr.Textbox(label="Fikir (replikleri ve ekran yazılarını tırnak içinde yazın)", lines=3,
                                elem_id="hizli-fikir",
                                placeholder='Örn: Kapadokya\'da bir çay ocağında yaşlı çaycı gence çay uzatıp '
                                            '"Buyur evlat, taze demlendi." diyor.')
            with gr.Row():
                q_seconds = gr.Slider(config.MIN_SECONDS, config.MAX_SECONDS, value=10, step=1, label="Süre (saniye)",
                                      elem_id="hizli-sure")
                q_style = gr.Radio([(v["label"], k) for k, v in config.STYLES.items()], value=config.DEFAULT_STYLE,
                                   label="Stil", elem_id="hizli-stil")
                q_method = gr.Radio(QUICK_METHODS, value="pika", label="Yöntem", elem_id="hizli-yontem")
            q_btn = gr.Button("Fikirden videoya: hepsini yap", variant="primary", elem_id="hizli-uret")
            q_status = gr.Textbox(label="Durum", interactive=False, lines=1, elem_id="hizli-durum")
            q_video = gr.Video(label="Video", interactive=False, elem_id="hizli-video")
            q_files = gr.File(label="İndir (video, prompt, bilgi, kontrol)", file_count="multiple", interactive=False,
                              elem_id="hizli-dosyalar")
            with gr.Accordion("Ayrıntılı günlük", open=False):
                q_log = gr.Textbox(lines=8, interactive=False, show_label=False, elem_id="hizli-gunluk")

        gr.Markdown("## 1 · Fikir")
        idea = gr.Textbox(label="Kısa fikir (herhangi bir dilde; ekranda görünecek yazıları ve replikleri tırnak içinde yazın)",
                          lines=4, placeholder='Örn: Gece yağmurlu bir İstanbul sokağı, neon tabelada "DUR, BİR BAK!" yazıyor…')
        with gr.Row():
            refs = gr.File(label=f"Karakter / referans görselleri (isteğe bağlı, en fazla {config.MAX_REF_IMAGES})",
                           file_count="multiple", file_types=["image"], scale=2)
            ref_preview = gr.Gallery(label="Önizleme (numara ve rol)", columns=3, height=180, scale=1,
                                     interactive=False)
        with gr.Accordion("Görsel rolleri: hangi görsel kim ya da ne?", open=True):
            gr.Markdown("Her görsel için rol seçin. **Karakter**ler yükleme sırasıyla Karakter 1, 2… olur ve "
                        "konuşurlarsa S1, S2… konuşmacısıdır. **Arka plan / mekân** görselinden kişi çıkarılmaz, sahne "
                        "o mekânda geçer. **Nesne** bir eşyadır. İsterseniz ad verin (ör. Çaycı Ahmet) ve fikirde bu "
                        "adla anın.", elem_classes="h3-note")
            role_rows, role_dds, role_names = [], [], []
            for i in range(config.MAX_REF_IMAGES):
                with gr.Row(visible=False) as row:
                    role_dds.append(gr.Dropdown(ROLE_CHOICES, value=prompt_writer.ROLE_CHARACTER,
                                                label=f"Görsel {i + 1} rolü", scale=1))
                    role_names.append(gr.Textbox(label=f"Görsel {i + 1} adı (isteğe bağlı)",
                                                 placeholder="ör. Çaycı Ahmet", scale=2))
                role_rows.append(row)
            ref_roles = gr.State([])
        style = gr.Radio([(v["label"], k) for k, v in config.STYLES.items()], value=config.DEFAULT_STYLE,
                         label="Stil (prompt'tan önce seçin)")
        with gr.Accordion("Çizgi / 3D stilde karakter görseli", open=False):
            gr.Markdown("Anime, Disney Pixar ya da Suluboya seçtiyseniz görselleri önce o stilde çizdirin: Gemini "
                        "karakterleri baştan ayağa fikirdeki kıyafetle, arka planı insansız bir mekân olarak, nesneyi "
                        "tek başına çizer; her görsel için 2 seçenek gelir, birini işaretleyin. İşaretlenmeyen "
                        "görselin yerine yüklenen hâli kullanılır; sıra ve roller değişmez.", elem_classes="h3-note")
            stylize_btn = gr.Button("Görselleri seçilen stilde çizdir")
            styled_gallery = gr.Gallery(label="Stilde çizilen görseller", columns=4, height=220, interactive=False)
            picks = gr.CheckboxGroup([], label="Kullanılacak görseller")
            stylize_msg = gr.Markdown()
            styled = gr.State({})
        with gr.Row():
            seconds = gr.Slider(config.MIN_SECONDS, config.MAX_SECONDS, value=15, step=1, label="Süre (saniye)")
            model = gr.Dropdown(label="Gemini modeli", choices=[], allow_custom_value=True)
        with gr.Row():
            key = gr.Textbox(label="Gemini API anahtarı", type="password", scale=3,
                             placeholder="Colab Secrets'ta GEMINI / GEMINI_API_KEY varsa boş bırakın")
            list_btn = gr.Button("Modelleri listele", scale=1)
        expand_btn = gr.Button("Gemini ile detaylandır (brief)", variant="primary")
        brief = gr.Textbox(label="Detaylı brief (Gemini yazar; düzenleyebilir ya da doğrudan kendi brief'inizi yazabilirsiniz)",
                           lines=8)
        rewrite_btn = gr.Button("H3 prompt'una çevir", variant="primary")
        rewrite_msg = gr.Markdown()

        gr.Markdown("## 2 · H3 prompt'u\nResmi H3 yapısında prompt; görsel yüklendiyse altı bölümlü referans (Ref2VA) yapısı. "
                    "Üretmeden önce düzenleyebilirsiniz.", elem_classes="h3-note")
        h3_prompt = gr.Textbox(label="H3 prompt'u", lines=14)

        gr.Markdown("## 3 · Üretim")
        with gr.Row():
            methods = gr.CheckboxGroup([(v["label"], k) for k, v in config.METHODS.items()], value=["pika"],
                                       label="Yöntem", scale=3)
            with gr.Column(scale=2):
                resolution = gr.Dropdown(list(config.RESOLUTIONS), value=list(config.RESOLUTIONS)[0], label="Boyut")
                seed = gr.Number(value=22, precision=0, label="Seed")
        with gr.Row():
            review_on = gr.Checkbox(value=True, label="Otomatik izleme: Gemini videoyu sesiyle izlesin, ciddi kusurda "
                                    "yeni seed ile yeniden üretsin", scale=3, elem_id="izleme-acik")
            review_retries = gr.Slider(0, 3, value=config.REVIEW_RETRIES, step=1, scale=1, elem_id="izleme-deneme",
                                       label="En fazla yeniden deneme")
        with gr.Row():
            gen_btn = gr.Button("Video üret", variant="primary", scale=3)
            load_btn = gr.Button("📂 Son Üretimleri Yükle (Bağlantı Koptuysa)", scale=1)

        gr.Markdown("## 4 · Sonuç")
        logbox = gr.Textbox(label="Durum", lines=10, autoscroll=True, interactive=False)
        with gr.Row():
            pika_video = gr.Video(label="Pika H3 Flex / H3 Turbo (Taslak)", interactive=False)
            h3_video = gr.Video(label="MiniMax-H3 (Final)", interactive=False)
        checks = gr.Gallery(label="Kontrol: videonun anları ve %100 kırpıntılar (eller, yüz, yazı)", columns=2,
                            height=360, interactive=False)
        files = gr.File(label="İndir (videolar, prompt, bilgi, kontrol)", file_count="multiple", interactive=False)

        with gr.Accordion("🎬 Uzun Video · Çok Çekimli Kurgu (15–60 sn)", open=False):
            gr.Markdown(
                "15 saniyeden uzun videolar **15 saniyelik çok çekimli bloklar** halinde üretilir ve `ffmpeg` ile kare-tam birleştirilir. "
                "Her blok kendi içinde `[Shot N] At 00:SS.000` ile 2–4 çekim içerir; blok sınırlarında sonda kaymayı önlemek için "
                "her blok ~0,5 sn sakin bir duruşla biter.", elem_classes="h3-note"
            )
            with gr.Row():
                total_seconds = gr.Slider(15, config.MAX_TOTAL_SECONDS, value=30, step=5,
                                          label="Toplam süre (saniye, 15 sn'lik bloklara bölünür)")
                audio_mode = gr.Radio([(v, k) for k, v in config.AUDIO_MODES.items()],
                                      value=config.DEFAULT_AUDIO_MODE, label="Ses modu")
            with gr.Row():
                music_file = gr.Audio(label="Telifsiz müzik dosyası (Müzikli mod için)", type="filepath", scale=3)
                keep_ambient = gr.Checkbox(value=False, label="Müzik altında blok ortam sesini (-12 dB) koru", scale=1)
            plan_btn = gr.Button("1 · Blok ve Çekim Tablosu Planla (Gemini)", variant="primary")
            long_msg = gr.Markdown()
            world_anchor = gr.Textbox(
                label="Ortak Karakter ve Dünya Çapası (tüm blokların prompt'una birebir girer)", lines=3
            )
            shot_table = gr.Dataframe(
                headers=["Blok", "Çekim", "Başlangıç (sn)", "Süre (sn)", "Kadraj / Plan", "Eylem", "Kamera", "Ses / Replik"],
                datatype=["number", "number", "number", "number", "str", "str", "str", "str"],
                interactive=True, label="Blok ve Çekim Tablosu (düzenleyebilirsiniz)",
            )
            blocks_prompts_btn = gr.Button("2 · Blokların H3 Prompt'larını Hazırla", variant="primary")
            block_prompts_box = gr.Textbox(
                label="Blok H3 Prompt'ları (`=== BLOK N (S sn) ===` ile ayrılır, üretmeden önce düzenleyebilirsiniz)",
                lines=12,
            )
            gen_blocks_btn = gr.Button("3 · Tüm Blokları Üret ve Birleştir", variant="primary")
            gr.Markdown("### Blok Bazlı Tekil Yeniden Üretim (Seçmeli Re-roll) ve Yeniden Birleştirme\n"
                        "Beğenmediğiniz tek bir bloğu farklı bir Seed ile yeniden üretip diğer bloklara dokunmadan birleştirebilirsiniz.",
                        elem_classes="h3-note")
            with gr.Row():
                reroll_block_num = gr.Dropdown(choices=[1, 2, 3, 4], value=1, label="Blok No")
                reroll_seed = gr.Number(value=42, precision=0, label="Yeni Seed")
                reroll_method = gr.Dropdown([(v["label"], k) for k, v in config.METHODS.items()],
                                            value="pika", label="Yöntem")
                reroll_btn = gr.Button("Seçili Bloğu Yeniden Üret ve Birleştir")
                reassemble_btn = gr.Button("Mevcut Blokları Yeniden Birleştir")
            block_videos_gallery = gr.Gallery(label="Üretilen Blok Önizlemeleri (Blok 1..N)", columns=4, height=220,
                                              interactive=False)
            blocks_state = gr.State({})

        list_btn.click(list_models, [key], [model, rewrite_msg])
        key.submit(list_models, [key], [model, rewrite_msg])
        key.blur(list_models, [key], [model, rewrite_msg])
        role_inputs = [*role_dds, *role_names]

        def live(fn, inputs):
            return _live_roles(fn, inputs, ref_roles, role_inputs)

        refs.change(_on_refs, [refs], [ref_preview, *role_rows, *role_dds, *role_names, ref_roles])
        for comp in [*role_dds, *role_names]:
            comp.change(_collect_roles, [refs, *role_dds, *role_names], [ref_roles, ref_preview])
        stylize_btn.click(*live(do_stylize, [refs, style, key, idea, ref_roles]), [styled_gallery, picks, styled, stylize_msg])
        style.change(lambda: (None, gr.update(choices=[], value=[]), {}), None, [styled_gallery, picks, styled])
        expand_btn.click(*live(do_expand, [idea, seconds, model, key, refs, style, picks, styled, ref_roles]),
                         [brief, rewrite_msg])
        rewrite_btn.click(*live(do_rewrite, [idea, brief, seconds, model, key, refs, style, picks, styled, ref_roles]),
                          [h3_prompt, rewrite_msg])
        gen_btn.click(*live(generate, [h3_prompt, methods, resolution, seconds, seed, idea, brief, refs, style, picks,
                                       styled, ref_roles, review_on, review_retries, key]),
                      [logbox, pika_video, h3_video, files, checks], concurrency_limit=1)
        load_btn.click(
            load_or_follow,
            None,
            [logbox, pika_video, h3_video, files, checks, h3_prompt,
             block_prompts_box, world_anchor, shot_table, block_videos_gallery, blocks_state],
        )
        plan_btn.click(*live(do_plan_shots, [idea, total_seconds, audio_mode, model, key, refs, style, picks, styled,
                                             ref_roles]),
                       [world_anchor, shot_table, long_msg])
        blocks_prompts_btn.click(
            *live(do_rewrite_blocks,
                  [idea, world_anchor, shot_table, audio_mode, model, key, refs, style, picks, styled, ref_roles]),
            [block_prompts_box, reroll_block_num, long_msg],
        )
        gen_blocks_btn.click(
            *live(generate_long,
                  [block_prompts_box, world_anchor, shot_table, audio_mode, music_file, keep_ambient,
                   methods, resolution, seed, idea, refs, style, picks, styled, ref_roles]),
            [logbox, pika_video, h3_video, block_videos_gallery, files, checks, blocks_state],
            concurrency_limit=1,
        )
        reroll_btn.click(
            reroll_block,
            [reroll_block_num, reroll_seed, reroll_method, block_prompts_box, audio_mode, music_file, keep_ambient, blocks_state],
            [logbox, pika_video, h3_video, block_videos_gallery, files, checks, blocks_state],
            concurrency_limit=1,
        )
        reassemble_btn.click(
            reassemble_blocks,
            [audio_mode, music_file, keep_ambient, blocks_state],
            [logbox, pika_video, h3_video, files],
        )
        q_btn.click(*live(quick_generate,
                          [q_idea, q_seconds, q_style, q_method, resolution, seed, model, key, refs, ref_roles, picks,
                           styled, review_on, review_retries]),
                    [q_status, brief, h3_prompt, q_video, q_files, q_log], concurrency_limit=1)
        # Stil seçimi iki yerde: kullanıcı hangisini değiştirirse öbürü de aynı olur (yalnızca elle değişimde)
        q_style.input(lambda v: v, q_style, style)
        style.input(lambda v: v, style, q_style)
        demo.load(lambda: _status_line(), None, status)
        if _state["secret_key"] or FAKE:
            demo.load(list_models, [key], [model, rewrite_msg])
    return demo


def launch(drive_dir=None, model_dirs=None, model_save_dir=None, **kwargs):
    """Colab'de: hücrenin altında açılır ve arka planda çalışmaya devam eder.
    model_dirs: model dosyalarının Hugging Face'ten önce aranacağı klasörler (ör. Drive'daki kopyalar).
    model_save_dir: verilirse Drive'da olmayan modeller indirildikten sonra arka planda buraya da kaydedilir."""
    _state["secret_hint"] = None
    _state["secret_key"] = _colab_secret("GEMINI") or _colab_secret("GEMINI_API_KEY")
    if _state["secret_key"]:
        _state["secret_hint"] = None
    elif _state["secret_hint"]:
        print("⚠️ Gemini anahtarı:", _state["secret_hint"])
    _state["drive_dir"] = drive_dir
    config.MODEL_SOURCE_DIRS = [d for d in (model_dirs or []) if d]
    config.MODEL_SAVE_DIR = model_save_dir or None
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    demo = build()
    demo.queue(default_concurrency_limit=1)
    # share=False: Gradio Colab'de aksi halde herkese açık bir gradio.live bağlantısı açar; uygulama
    # yalnızca Colab'in kendi (oturum açmış kullanıcıya özel) proxy'si üzerinden görünsün.
    allowed = list({os.path.abspath(config.OUTPUT_DIR), os.path.abspath(config.WORK_DIR), tempfile.gettempdir()})
    from starlette.middleware import Middleware
    opts = dict(share=False, inline=True, prevent_thread_lock=True, height=1400, allowed_paths=allowed,
                css=CSS, theme=gr.themes.Soft(), show_error=True,
                app_kwargs={"middleware": [Middleware(ColabProxyHost)]})
    opts.update(kwargs)
    return demo.launch(**opts)

## 3 · Uygulamayı başlat

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio==6.31.0", "google-genai==2.34.0"], check=True)

# Hangi GPU tahsis edildi? Colab G4 seçilse bile sessizce L4/A100 verebiliyor (17.11.2026 denemeleri).
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.strip()
gpu_name, _, gpu_mib = gpu_info.partition(",")
gpu_name, gpu_gb = gpu_name.strip() or "yok", int(gpu_mib.strip() or 0) / 1024
print(f"GPU: {gpu_name} ({gpu_gb:.0f} GB)")
if gpu_gb < 40:
    print("⛔ Bu GPU H3 için yetersiz (en az 40 GB bellek gerekir; video modeli ~21 GB, metin kodlayıcı 15-27 GB). "
          "Colab G4/A100 boş değilse sessizce L4/T4 veriyor. Çalışma zamanı → Bağlantıyı kes ve sil → "
          "Çalışma zamanı türünü değiştir'de G4 ya da A100 seçip yeniden bağlanın; olmazsa biraz sonra tekrar deneyin.")
    # Drive izni ve kurulum boşuna yapılmasın: burada dur (yalnızca bu hücre durur, runtime'ı siz kapatın)
    raise SystemExit("GPU yetersiz: uygulama başlatılmadı.")
elif "RTX PRO 6000" not in gpu_name:
    print("⚠️ G4 (RTX PRO 6000) değil. Uygulama çalışır, ama NVFP4 yerine 27 GB'lık int8 metin kodlayıcı iner ve "
          "üretim daha yavaş olabilir. G4 istiyorsanız: Çalışma zamanı → Bağlantıyı kes ve sil → türü G4 seçip "
          "yeniden bağlanın (tahsis birkaç denemede değişebiliyor).")

drive_dir, model_dirs, model_save_dir = None, [], None
if SAVE_TO_DRIVE or DRIVE_MODEL_DIRS or SAVE_MODELS_TO_DRIVE:
    from google.colab import drive
    # "credential propagation was unsuccessful" oturumdan oturuma değişiyor: birkaç kez dene, olmazsa Drive'sız sür
    mounted = False
    for attempt in range(1, 4):
        try:
            drive.mount("/content/drive", force_remount=attempt > 1)
            mounted = True
            break
        except Exception as e:
            print(f"⚠️ Drive bağlanamadı ({attempt}/3): {e}")
    if not mounted:
        print("⚠️ Drive'sız devam ediliyor: modeller Hugging Face'ten iner (~6 dk), videolar yalnızca bu runtime'da "
              "kalır (arayüzden indirin). Drive'ı sonra denemek için bu hücreyi yeniden çalıştırın; olmazsa "
              "Çalışma zamanı → Bağlantıyı kes ve sil ile yeni oturum açın.")
        SAVE_TO_DRIVE, DRIVE_MODEL_DIRS, SAVE_MODELS_TO_DRIVE = False, [], False
if SAVE_TO_DRIVE or DRIVE_MODEL_DIRS or SAVE_MODELS_TO_DRIVE:
    drive_dir = DRIVE_FOLDER if SAVE_TO_DRIVE else None
    model_dirs = DRIVE_MODEL_DIRS
    if SAVE_MODELS_TO_DRIVE:
        model_save_dir = (DRIVE_MODEL_DIRS or ["/content/drive/MyDrive/MiniMax-H3/models"])[0]

# Hücre aynı oturumda yeniden çalıştırılırsa eski arayüz 7860'ı tutar (17.11.2026). Süren bir üretim varsa ona
# dokunulmaz, eski arayüze devam edilir; yoksa eski sunucu kapatılıp yenisi açılır.
import threading
running = [t.name for t in threading.enumerate() if t.name.startswith("h3app-") and t.name != "h3app-model-save"]
if running:
    print("⚠️ Bu oturumda süren bir üretim var (" + ", ".join(running) + "); uygulama yeniden başlatılmadı. "
          "Arayüze aşağıdaki adresten devam edin ve 📂 Son Üretimleri Yükle'ye basın. Üretim bitince bu hücreyi "
          "yeniden çalıştırabilirsiniz.")
else:
    import gradio as gr
    gr.close_all()
    sys.path.insert(0, "/content/h3_video_app")
    for m in [m for m in sys.modules if m == "h3app" or m.startswith("h3app.")]:
        del sys.modules[m]
    import h3app
    h3app.launch(drive_dir=drive_dir, model_dirs=model_dirs, model_save_dir=model_save_dir, server_port=7860)
try:
    from google.colab import output
    print("PROXY_URL:", output.eval_js("google.colab.kernel.proxyPort(7860)"))
except Exception:
    pass

## Bitince
Menü: **Çalışma zamanı → Çalışma zamanı bağlantısını kes ve sil** (G4 saatte ~10 CU harcar). İndirmediğiniz videolar runtime ile birlikte silinir.

**Sürümler:** ComfyUI `0dbfd49` · Comfy-Org/MiniMax-H3 `912b6e38` · H3 skill `199b7d0e` · gradio 6.31.0 · google-genai 2.34.0